# Notebook 08 - Sensor Models

## Purpose

Notebook 07 established the validated physical motion of a six-degree-of-freedom underwater vehicle.

It answered:

> **How does the UUV physically move through the simulated ocean?**

Notebook 08 introduces the next layer of HydroLab-3D:

> **What does the UUV actually measure about that motion?**

The central architectural separation in this notebook is:

```text
Ground-Truth Physical State
            ↓
       Sensor Model
            ↓
        Measurement
            ↓
   Autonomy / Control
```

The simulator's latent physical state remains authoritative.

Sensor noise, bias, calibration error, finite range, dropout, and other imperfections belong only to the **measurement layer**. They must never modify the underlying six-DOF vehicle state.

This distinction is essential for autonomy research because a controller or estimator should operate using realistic observations rather than privileged simulator truth.

---

## Why This Matters

Real underwater robots do not directly observe their exact physical state.

A UUV may internally possess a ground-truth simulator state such as:

$$
\eta =
\begin{bmatrix}
x &
y &
z &
\phi &
\theta &
\psi
\end{bmatrix}^{T}
$$

and:

$$
\nu =
\begin{bmatrix}
u &
v &
w &
p &
q &
r
\end{bmatrix}^{T},
$$

but a real vehicle instead receives measurements from instruments such as:

- inertial sensors;
- pressure or depth sensors;
- Doppler velocity logs;
- heading sensors;
- acoustic navigation systems;
- sonar.

Those measurements are imperfect.

Conceptually:

$$
\tilde{y}
=
h(x)
+
b
+
n
$$

where:

- $x$ is the simulator ground truth;
- $h(x)$ maps physical state into the quantity measured by the sensor;
- $b$ represents deterministic sensor bias;
- $n$ represents stochastic measurement noise;
- $\tilde{y}$ is the measurement exposed to the autonomy stack.

Later sensor models may additionally include effects such as:

- clipping;
- finite operating range;
- dropout;
- update-rate limitations;
- calibration error.

These effects will only be introduced where they materially improve the usefulness of HydroLab-3D.

Notebook 08 will not attempt to simulate sensor electronics, detailed acoustic propagation, Allan-variance processes, multipath, or hardware-specific proprietary behavior.

HydroLab-3D remains a **lightweight, algorithm-first, selective-fidelity simulator**.

---

# Relationship to Notebook 07

Notebook 07 produced the validated full six-DOF vehicle model:

$$
M\dot{\nu}
+
C(\nu)\nu
+
D(\nu_r)\nu_r
+
g(\eta)
=
\tau
+
\tau_{env}
$$

with pose kinematics:

$$
\dot{\eta}
=
J(\eta)\nu.
$$

Notebook 08 does **not** alter this physical model.

Instead, it consumes quantities produced by that model.

Conceptually:

```text
Thrusters + Environment
          ↓
     6-DOF Dynamics
          ↓
      Ground Truth
          ↓
   ┌──────┼──────┐
   ↓      ↓      ↓
  IMU    DVL    Depth
   │      │      │
   └──────┼──────┘
          ↓
      Measurements
```

The physical simulator therefore remains deterministic unless deliberately modified by an environmental model.

Sensor stochasticity is introduced **after** the ground-truth state has been computed.

This means that two simulations using identical initial conditions, control inputs, and physical parameters must still produce identical physical trajectories even if their sensor noise seeds differ.

Only the measurements should change.

---

# Inherited Coordinate and State Conventions

Notebook 08 preserves all conventions validated by the preceding HydroLab-3D notebooks.

## Pose

The world-frame pose is:

$$
\eta =
\begin{bmatrix}
x &
y &
z &
\phi &
\theta &
\psi
\end{bmatrix}^{T}
$$

where:

- $x$, $y$, $z$ are world-frame position coordinates;
- $\phi$ is roll;
- $\theta$ is pitch;
- $\psi$ is yaw.

The world frame uses:

```text
+x_W    world x direction
+y_W    world y direction
+z_W    upward
```

The water surface is represented by:

$$
z_W = 0.
$$

Therefore a submerged vehicle normally has:

$$
z_W < 0.
$$

This convention is especially important for the depth sensor.

---

## Body Velocity

The generalized body-frame velocity is:

$$
\nu =
\begin{bmatrix}
u &
v &
w &
p &
q &
r
\end{bmatrix}^{T}
$$

where:

$$
\mathbf{v}_B =
\begin{bmatrix}
u\\
v\\
w
\end{bmatrix}
$$

is translational velocity expressed in the body frame and:

$$
\boldsymbol{\omega}_B =
\begin{bmatrix}
p\\
q\\
r
\end{bmatrix}
$$

is body angular velocity.

These quantities must not be silently reinterpreted as world-frame measurements.

---

## Orientation Convention

HydroLab-3D uses the existing body-to-world rotation:

$$
R_B^W
=
R_z(\psi)
R_y(\theta)
R_x(\phi).
$$

Therefore:

$$
\mathbf{v}_W
=
R_B^W\mathbf{v}_B
$$

and:

$$
R_W^B
=
\left(R_B^W\right)^T.
$$

Notebook 08 will reuse the validated production transforms rather than duplicating orientation mathematics.

---

# Initial Sensor Scope

Notebook 08 will begin with the smallest useful sensor set that can be supported directly by the validated HydroLab-3D physics.

The intended initial suite is:

```text
Sensor Layer
│
├── IMU
│   ├── angular velocity
│   └── specific force
│
├── Depth Sensor
│   └── depth below the water surface
│
├── DVL
│   └── water-relative body velocity
│
└── Heading Sensor
    └── yaw / compass heading
```

Each sensor will be developed incrementally and validated independently before production promotion.

---

# IMU Model

The initial IMU will contain two conceptual instruments:

```text
IMU
├── Gyroscope
└── Accelerometer
```

## Gyroscope

The ideal gyroscope measurement is directly related to body angular velocity:

$$
\boldsymbol{\omega}_{gyro}
=
\begin{bmatrix}
p\\
q\\
r
\end{bmatrix}.
$$

A simple imperfect measurement may later use:

$$
\tilde{\boldsymbol{\omega}}
=
\boldsymbol{\omega}
+
\mathbf{b}_{g}
+
\mathbf{n}_{g}.
$$

The gyroscope therefore provides a natural first validation target because its frame and physical interpretation are unambiguous.

---

## Accelerometer

The accelerometer requires greater care.

A physical accelerometer does **not** simply measure:

$$
\dot{\mathbf{v}}_B.
$$

Because translational velocity is expressed in a rotating body frame, inertial acceleration expressed in that frame contains the rotational contribution:

$$
\mathbf{a}_B
=
\dot{\mathbf{v}}_B
+
\boldsymbol{\omega}_B
\times
\mathbf{v}_B.
$$

An ideal accelerometer measures **specific force**, which excludes gravitational acceleration.

Therefore:

$$
\mathbf{f}_B
=
\dot{\mathbf{v}}_B
+
\boldsymbol{\omega}_B
\times
\mathbf{v}_B
-
R_W^B\mathbf{g}_W.
$$

This distinction is important.

Notebook 08 will explicitly distinguish:

```text
body-frame velocity derivative
≠
body-frame inertial acceleration
≠
accelerometer specific force
```

rather than calling all three quantities simply "acceleration."

---

# Depth Sensor

Because HydroLab-3D uses world $+z$ upward and the water surface is:

$$
z=0,
$$

ideal depth is:

$$
d=-z.
$$

Therefore:

```text
z =  0 m    → depth = 0 m
z = -5 m    → depth = 5 m
z = -20 m   → depth = 20 m
```

The initial depth measurement model may later become:

$$
\tilde{d}
=
d
+
b_d
+
n_d.
$$

The sensor must preserve the existing world-frame sign convention rather than redefining the simulator's vertical axis.

---

# DVL Model

Notebook 06 and Notebook 07 already distinguish vehicle motion from surrounding water motion.

Let world-frame ocean current be:

$$
\mathbf{v}_{c,W}.
$$

The current expressed in the body frame is:

$$
\mathbf{v}_{c,B}
=
R_W^B
\mathbf{v}_{c,W}.
$$

If vehicle translational body velocity is:

$$
\mathbf{v}_B,
$$

then water-relative velocity is:

$$
\mathbf{v}_{r,B}
=
\mathbf{v}_B
-
\mathbf{v}_{c,B}.
$$

The initial DVL abstraction will use this already validated relative-motion structure.

This gives Notebook 08 a physically useful measurement without introducing new hydrodynamic assumptions.

More specialized distinctions such as:

- bottom-track velocity;
- water-track velocity;
- beam geometry;
- altitude-dependent lock;
- acoustic Doppler beam simulation;

are deliberately deferred.

---

# Heading Sensor

The initial heading sensor will provide a lightweight measurement related to the existing yaw coordinate:

$$
\psi.
$$

Its exact normalization convention will be made explicit during implementation and validated at angular wrap boundaries.

The purpose is not to simulate a detailed magnetometer or Earth's magnetic field.

It is to provide a lightweight imperfect heading observation suitable for later closed-loop control and state-estimation experiments.

---

# Why Sonar Is Deferred

The original HydroLab-3D roadmap includes sonar as a future sensor capability.

However, a useful sonar requires something meaningful to measure.

That normally implies infrastructure such as:

```text
terrain
obstacles
objects
ray intersections
range-bearing geometry
```

The validated Notebook 00–07 lineage currently establishes vehicle dynamics and environmental currents, but not yet a sufficiently mature geometric sensing environment.

Introducing sonar immediately would therefore either:

1. require unrelated environmental infrastructure inside Notebook 08; or
2. produce a sensor abstraction disconnected from meaningful geometry.

Neither option fits the notebook-first selective-fidelity philosophy.

Sonar will therefore remain a later extension rather than being forced into the first sensor implementation.

---

# Why Absolute Position Fixes Are Deferred

Underwater vehicles normally cannot continuously depend on GPS while submerged.

HydroLab-3D intentionally aims to support autonomy under imperfect localization.

A future position-fix abstraction may represent systems such as:

- acoustic positioning;
- intermittent surfacing fixes;
- external navigation aids.

For the initial sensor layer, however, direct noisy world position would provide autonomy with information that is unrealistically convenient.

Position fixes are therefore not part of the minimum Notebook 08 scope.

---

# Sensor Imperfection Model

The initial common sensor model will remain deliberately lightweight.

Where appropriate, measurements may use:

$$
\tilde{\mathbf y}
=
\mathbf y
+
\mathbf b
+
\mathbf n
$$

with:

$$
\mathbf n
\sim
\mathcal{N}
\left(
\mathbf 0,
\Sigma
\right).
$$

Potential effects include:

- deterministic bias;
- Gaussian noise;
- finite measurement range;
- clipping;
- dropout;
- rate limitation.

However, these effects will **not** automatically be included in every sensor.

Each effect must have a clear purpose.

For example:

```text
Gaussian noise
→ useful across many sensors

depth clipping
→ useful if an explicit operating range exists

DVL dropout
→ useful if lock conditions are modeled

complex electronics drift
→ unnecessary at this stage
```

The objective is physically interpretable imperfection, not complexity for its own sake.

---

# Randomness and Reproducibility

Notebook 07's physical dynamics are deterministic.

Notebook 08 introduces stochastic behavior only at the sensor layer.

All randomness must therefore use explicit NumPy random generators.

The preferred pattern is:

```text
np.random.default_rng(seed)
```

rather than global calls such as:

```text
np.random.normal(...)
```

The intended behavior is:

```text
same physical state
+
same sensor parameters
+
same random seed
        ↓
identical measurement sequence
```

while:

```text
same physical state
+
same sensor parameters
+
different random seed
        ↓
different stochastic measurement sequence
```

Changing sensor seeds must never change the underlying vehicle trajectory.

This separation will be validated explicitly.

---

# Validation Philosophy

Each sensor primitive must satisfy analytical or high-information numerical checks.

Notebook 08 will prefer exact invariants over visual plausibility.

Examples include:

### Ideal Sensor Recovery

With zero bias and zero noise:

$$
\tilde{y}=y.
$$

---

### Exact Bias

With noise disabled:

$$
\tilde{y}-y=b.
$$

---

### Seed Reproducibility

Two identically seeded generators should produce identical measurement sequences.

---

### Seed Independence

Different seeds should produce different stochastic realizations.

---

### Noise Statistics

Over sufficiently many samples:

$$
E[n]\approx0
$$

and measured spread should approximately match the configured standard deviation.

---

### Frame Consistency

Any world-to-body or body-to-world conversion must match the validated production transforms from the previous notebooks.

---

### Stationary IMU Physics

A stationary vehicle should exhibit physically consistent accelerometer specific force under gravity rather than simply producing zero because:

$$
\dot{\mathbf v}_B=0.
$$

---

### Depth Sign

A vehicle moving downward into more negative world $z$ must report **increasing positive depth**.

---

### DVL Relative Motion

When:

$$
\mathbf v_B
=
\mathbf v_{c,B},
$$

the water-relative velocity should satisfy:

$$
\mathbf v_{r,B}
=
\mathbf 0.
$$

---

### Invalid Inputs

Malformed sensor parameters should fail explicitly.

Examples include:

- wrong vector shapes;
- NaN;
- infinity;
- negative standard deviations;
- invalid range limits;
- impossible probabilities.

Inputs should not be silently corrected.

---

# Development Strategy

Notebook 08 follows the same validated engineering workflow as the previous HydroLab-3D notebooks:

```text
Notebook-local sensor prototype
        ↓
analytical validation
        ↓
numerical validation
        ↓
edge-case validation
        ↓
comparison with inherited production behavior
        ↓
production promotion
        ↓
source inspection
        ↓
production re-import
        ↓
independent production validation
        ↓
fresh-process validation
        ↓
persistent artifacts
        ↓
dedicated automated tests
        ↓
complete repository pytest suite
```

No notebook-local implementation becomes production code merely because it executes successfully.

Only validated reusable functionality will be promoted.

---

# Production Architecture

The exact production layout will be decided only after the notebook-local models are validated and the current repository structure is inspected.

A likely eventual location is:

```text
src/hydrolab3d/sensors/
```

but Notebook 08 will not create unnecessary abstractions merely to match a tentative architecture diagram.

In particular, it will not invent a generic simulator state class unless an actual requirement emerges.

Existing HydroLab-3D APIs and module responsibilities remain authoritative.

---

# Expected Outputs

Notebook 08 may ultimately produce compact validation artifacts such as:

```text
results/notebooks/08_sensor_models/
├── data/
│   └── sensor_measurements.csv
│
├── plots/
│   └── sensor_validation.png
│
└── logs/
    └── sensor_summary.txt
```

The exact artifact set will depend on which outputs provide useful persistent validation evidence.

Artifacts will be generated using the **promoted production implementation**, not stale notebook-local prototypes.

Large datasets and decorative plots are intentionally avoided.

---

# Compatibility Requirements

Notebook 08 must preserve all previously validated HydroLab-3D behavior.

It must not break:

- basic 3D kinematics;
- orientation transforms;
- rigid-body dynamics;
- hydrodynamic drag;
- gravity and buoyancy;
- thruster behavior;
- ocean-current behavior;
- six-DOF UUV dynamics;
- public package exports;
- existing notebook artifacts;
- existing automated tests.

The inherited regression baseline from Notebook 07 is:

```text
113 passed
```

This is only the historical starting point.

Notebook 08 will report the actual complete test-suite result obtained after sensor production integration rather than predicting a final test count.

---

# Success Criteria

Notebook 08 will be considered successful when:

1. ground-truth vehicle state remains completely separate from sensor measurements;
2. implemented sensor quantities have explicit physical meaning, frames, and units;
3. ideal measurements reproduce analytical ground truth;
4. deterministic sensor biases behave exactly as configured;
5. stochastic noise is reproducible under fixed seeds;
6. different seeds produce independent noise realizations;
7. IMU specific-force behavior is physically consistent;
8. depth respects HydroLab-3D's world-$z$ convention;
9. DVL measurements respect existing current-relative velocity mathematics;
10. malformed inputs are rejected explicitly;
11. validated reusable sensor code is promoted cleanly into the production package;
12. production imports are independently revalidated;
13. fresh-process validation succeeds;
14. compact persistent artifacts are generated from production code;
15. dedicated automated sensor tests pass;
16. the complete repository test suite passes without regressions.

---

# Relationship to the Next Notebook

Notebook 09 will introduce closed-loop control.

That controller should not need privileged access to exact simulator truth.

Notebook 08 therefore creates the observation layer required for the transition:

```text
Physical UUV
     ↓
Ground Truth
     ↓
Sensor Measurements
     ↓
Controller
     ↓
Thruster Commands
     ↓
Physical UUV
```

This turns HydroLab-3D from a validated vehicle-dynamics model into the beginning of a genuine robotics simulation loop.

Notebook 07 established **motion**.

Notebook 08 establishes **observation**.

Notebook 09 can then establish **feedback**.

---

## Step 1 - Repository Audit and Sensor Dependency Freeze

Before implementing any sensor model, we verify the exact HydroLab-3D production state inherited from Notebook 07.

This audit is strictly read-only.

It confirms:

- the repository root;
- the validated orientation and current-transform utilities;
- the production 6-DOF UUV functions;
- the current `src/hydrolab3d/sensors/` state;
- the existing automated test files;
- the inherited Notebook 07 regression baseline.

The sensor layer must preserve:

```text
ground-truth physical state
        ↓
sensor model
        ↓
measurement
```

No production files are created or modified in this step.

### Validation Criteria

This step passes only if all required ground-truth dependencies resolve to production source files and the current sensor-package state can be inspected successfully.

In [1]:
from pathlib import Path
import importlib
import inspect
import sys


# ---------------------------------------------------------------------
# Resolve HydroLab-3D repository root
# ---------------------------------------------------------------------

start = Path.cwd().resolve()
repo_root = None

for candidate in [start, *start.parents]:
    if (
        (candidate / "README.md").exists()
        and (candidate / "src").exists()
        and (candidate / "notebooks").exists()
    ):
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the HydroLab-3D repository root from "
        f"{start}"
    )

src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

importlib.invalidate_caches()


# ---------------------------------------------------------------------
# Required Notebook 08 production dependencies
# ---------------------------------------------------------------------

required_files = {
    "transforms": (
        src_path
        / "hydrolab3d"
        / "utils"
        / "transforms.py"
    ),
    "hydrodynamics": (
        src_path
        / "hydrolab3d"
        / "dynamics"
        / "hydrodynamics.py"
    ),
    "currents": (
        src_path
        / "hydrolab3d"
        / "world"
        / "currents.py"
    ),
    "uuv": (
        src_path
        / "hydrolab3d"
        / "vehicles"
        / "uuv.py"
    ),
}

for name, path in required_files.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Required Notebook 08 dependency is missing "
            f"({name}): {path}"
        )

    if not path.is_file():
        raise RuntimeError(
            f"Expected a file for {name}, found another path type: "
            f"{path}"
        )


# ---------------------------------------------------------------------
# Import inherited ground-truth functionality
# ---------------------------------------------------------------------

from hydrolab3d.utils.transforms import (
    rotation_body_to_world,
)

from hydrolab3d.world.currents import (
    current_velocity_body,
    generalized_current_velocity,
    relative_velocity_6dof,
)

from hydrolab3d.vehicles.uuv import (
    current_generalized_acceleration,
    six_dof_state_derivative,
    integrate_six_dof_euler,
    simulate_six_dof,
)


inherited_functions = {
    "rotation_body_to_world": rotation_body_to_world,
    "current_velocity_body": current_velocity_body,
    "generalized_current_velocity": generalized_current_velocity,
    "relative_velocity_6dof": relative_velocity_6dof,
    "current_generalized_acceleration": current_generalized_acceleration,
    "six_dof_state_derivative": six_dof_state_derivative,
    "integrate_six_dof_euler": integrate_six_dof_euler,
    "simulate_six_dof": simulate_six_dof,
}


# ---------------------------------------------------------------------
# Inspect existing sensor-package state
# ---------------------------------------------------------------------

sensor_dir = src_path / "hydrolab3d" / "sensors"

if sensor_dir.exists():
    sensor_files = sorted(
        path
        for path in sensor_dir.rglob("*")
        if path.is_file()
    )
else:
    sensor_files = []


# ---------------------------------------------------------------------
# Inspect current automated tests
# ---------------------------------------------------------------------

tests_dir = repo_root / "tests"

if not tests_dir.exists():
    raise FileNotFoundError(
        f"Expected tests directory is missing: {tests_dir}"
    )

test_files = sorted(
    tests_dir.glob("test_*.py")
)


# ---------------------------------------------------------------------
# Report audit
# ---------------------------------------------------------------------

print("=" * 72)
print("STEP 1 - NOTEBOOK 08 REPOSITORY AUDIT")
print("=" * 72)

print("\nRepository root:")
print(repo_root)

print("\nRequired production dependencies:")
for name, path in required_files.items():
    print(
        f"  {name:24s}"
        f"{path.relative_to(repo_root)}"
    )

print("\nInherited callable source locations:")
for name, function in inherited_functions.items():
    source = Path(
        inspect.getsourcefile(function)
        or inspect.getfile(function)
    ).resolve()

    print(
        f"  {name:32s}"
        f"{source.relative_to(repo_root)}"
    )

print("\nExisting sensor package:")
if sensor_dir.exists():
    print(f"  {sensor_dir.relative_to(repo_root)}")

    if sensor_files:
        for path in sensor_files:
            print(f"    {path.relative_to(repo_root)}")
    else:
        print("    <directory exists but contains no files>")
else:
    print("  <not yet present>")

print("\nExisting automated test files:")
for path in test_files:
    print(f"  {path.relative_to(repo_root)}")

print("\nInherited Notebook 07 regression baseline:")
print("  113 passed")

print("\nPASS condition:")
print(
    "  Ground-truth dependencies resolved and "
    "sensor-package state inspected read-only."
)

STEP 1 - NOTEBOOK 08 REPOSITORY AUDIT

Repository root:
/home/agniacolyte/HydroLab-3D

Required production dependencies:
  transforms              src/hydrolab3d/utils/transforms.py
  hydrodynamics           src/hydrolab3d/dynamics/hydrodynamics.py
  currents                src/hydrolab3d/world/currents.py
  uuv                     src/hydrolab3d/vehicles/uuv.py

Inherited callable source locations:
  rotation_body_to_world          src/hydrolab3d/utils/transforms.py
  current_velocity_body           src/hydrolab3d/world/currents.py
  generalized_current_velocity    src/hydrolab3d/world/currents.py
  relative_velocity_6dof          src/hydrolab3d/world/currents.py
  current_generalized_accelerationsrc/hydrolab3d/vehicles/uuv.py
  six_dof_state_derivative        src/hydrolab3d/vehicles/uuv.py
  integrate_six_dof_euler         src/hydrolab3d/vehicles/uuv.py
  simulate_six_dof                src/hydrolab3d/vehicles/uuv.py

Existing sensor package:
  src/hydrolab3d/sensors
    src/hydrolab

## Step 2 - Inspect Existing Sensor Production Skeleton

The repository audit revealed that `src/hydrolab3d/sensors/` already exists and contains:

```text
__init__.py
sensor.py
imu.py
depth.py
dvl.py
sonar.py
```

Before defining any Notebook 08 sensor API, we must inspect these files directly.

The objective is to determine whether they are:

- empty placeholders;
- tentative scaffolding;
- partially implemented models;
- or existing production interfaces that Notebook 08 must preserve.

This step remains strictly read-only.

### Validation Criteria

This step passes when the complete current contents of every sensor source file are visible and we can determine what, if anything, should be reused or preserved before notebook-local prototyping begins.

In [2]:
# ---------------------------------------------------------------------
# Step 2 - Inspect existing sensor production skeleton
# ---------------------------------------------------------------------

sensor_source_files = [
    sensor_dir / "__init__.py",
    sensor_dir / "sensor.py",
    sensor_dir / "imu.py",
    sensor_dir / "depth.py",
    sensor_dir / "dvl.py",
    sensor_dir / "sonar.py",
]

for path in sensor_source_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Expected sensor source file is missing: {path}"
        )

    print("\n" + "=" * 80)
    print(path.relative_to(repo_root))
    print("=" * 80)

    contents = path.read_text(encoding="utf-8")

    if contents.strip():
        print(contents)
    else:
        print("<EMPTY FILE>")

print("\n" + "=" * 80)
print("STEP 2 INSPECTION COMPLETE")
print("=" * 80)
print(
    "No files were modified. "
    "Existing sensor source contents are now available for review."
)


src/hydrolab3d/sensors/__init__.py
<EMPTY FILE>

src/hydrolab3d/sensors/sensor.py
<EMPTY FILE>

src/hydrolab3d/sensors/imu.py
<EMPTY FILE>

src/hydrolab3d/sensors/depth.py
<EMPTY FILE>

src/hydrolab3d/sensors/dvl.py
<EMPTY FILE>

src/hydrolab3d/sensors/sonar.py
<EMPTY FILE>

STEP 2 INSPECTION COMPLETE
No files were modified. Existing sensor source contents are now available for review.


## Step 3 - Common Measurement Primitive

Before implementing individual sensors, we define the smallest reusable measurement operation:

$$
\tilde{\mathbf y}
=
\mathbf y
+
\mathbf b
+
\mathbf n
$$

where:

- $\mathbf y$ is the ideal physical measurement;
- $\mathbf b$ is deterministic bias;
- $\mathbf n$ is Gaussian noise;
- $\tilde{\mathbf y}$ is the reported measurement.

Noise is sampled using an explicit NumPy random generator:

```text
np.random.default_rng(seed)
```

The function developed in this step is notebook-local only.

It does not yet implement:

- clipping;
- dropout;
- latency;
- update rates;
- sensor-specific physics.

### Validation Criteria

This step passes only if:

1. zero bias and zero noise reproduce the input exactly;
2. deterministic bias is applied exactly;
3. identical seeds reproduce identical noisy measurements;
4. different seeds produce different noisy measurements;
5. invalid inputs such as negative noise standard deviation or non-finite values are rejected explicitly.

In [3]:
import numpy as np


def measure_with_bias_noise(
    truth,
    *,
    bias=0.0,
    noise_std=0.0,
    rng=None,
):
    """
    Apply deterministic bias and Gaussian measurement noise.

    Parameters
    ----------
    truth : array_like
        Ideal scalar or vector measurement.
    bias : scalar or array_like
        Deterministic additive bias. Must be broadcast-compatible
        with truth.
    noise_std : scalar or array_like
        Gaussian standard deviation. Must be non-negative and
        broadcast-compatible with truth.
    rng : numpy.random.Generator or None
        Explicit random generator. Required when noise is nonzero.

    Returns
    -------
    numpy.ndarray
        Measured value with the same broadcasted shape as truth.
    """

    truth = np.asarray(truth, dtype=float)
    bias = np.asarray(bias, dtype=float)
    noise_std = np.asarray(noise_std, dtype=float)

    if truth.ndim > 1:
        raise ValueError(
            "truth must be a scalar or one-dimensional array"
        )

    if not np.all(np.isfinite(truth)):
        raise ValueError("truth must contain only finite values")

    if not np.all(np.isfinite(bias)):
        raise ValueError("bias must contain only finite values")

    if not np.all(np.isfinite(noise_std)):
        raise ValueError(
            "noise_std must contain only finite values"
        )

    if np.any(noise_std < 0.0):
        raise ValueError(
            "noise_std must be non-negative"
        )

    try:
        truth_b, bias_b, std_b = np.broadcast_arrays(
            truth,
            bias,
            noise_std,
        )
    except ValueError as exc:
        raise ValueError(
            "truth, bias, and noise_std must be "
            "broadcast-compatible"
        ) from exc

    if np.any(std_b > 0.0):
        if rng is None:
            raise ValueError(
                "rng must be provided when noise_std is nonzero"
            )

        if not isinstance(rng, np.random.Generator):
            raise TypeError(
                "rng must be a numpy.random.Generator"
            )

        noise = rng.normal(
            loc=0.0,
            scale=std_b,
            size=truth_b.shape,
        )
    else:
        noise = np.zeros_like(
            truth_b,
            dtype=float,
        )

    return truth_b + bias_b + noise


# ---------------------------------------------------------------------
# Validation 1: exact ideal measurement
# ---------------------------------------------------------------------

truth = np.array([1.5, -2.0, 4.25])

ideal = measure_with_bias_noise(
    truth,
    bias=0.0,
    noise_std=0.0,
)

np.testing.assert_array_equal(
    ideal,
    truth,
)


# ---------------------------------------------------------------------
# Validation 2: exact deterministic bias
# ---------------------------------------------------------------------

bias = np.array([0.1, -0.2, 0.3])

biased = measure_with_bias_noise(
    truth,
    bias=bias,
    noise_std=0.0,
)

np.testing.assert_allclose(
    biased,
    truth + bias,
    rtol=0.0,
    atol=0.0,
)


# ---------------------------------------------------------------------
# Validation 3: identical seed -> identical noise
# ---------------------------------------------------------------------

rng_a = np.random.default_rng(12345)
rng_b = np.random.default_rng(12345)

noisy_a = measure_with_bias_noise(
    truth,
    bias=0.0,
    noise_std=0.05,
    rng=rng_a,
)

noisy_b = measure_with_bias_noise(
    truth,
    bias=0.0,
    noise_std=0.05,
    rng=rng_b,
)

np.testing.assert_array_equal(
    noisy_a,
    noisy_b,
)


# ---------------------------------------------------------------------
# Validation 4: different seeds -> different realization
# ---------------------------------------------------------------------

rng_c = np.random.default_rng(54321)

noisy_c = measure_with_bias_noise(
    truth,
    bias=0.0,
    noise_std=0.05,
    rng=rng_c,
)

if np.array_equal(noisy_a, noisy_c):
    raise AssertionError(
        "Different seeds unexpectedly produced "
        "identical noisy measurements"
    )


# ---------------------------------------------------------------------
# Validation 5: invalid parameters rejected
# ---------------------------------------------------------------------

try:
    measure_with_bias_noise(
        truth,
        noise_std=-0.1,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Negative noise_std was not rejected"
    )


try:
    measure_with_bias_noise(
        [1.0, np.nan],
        noise_std=0.0,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Non-finite truth input was not rejected"
    )


try:
    measure_with_bias_noise(
        truth,
        noise_std=0.1,
        rng=np.random.RandomState(0),
    )
except TypeError:
    pass
else:
    raise AssertionError(
        "Non-Generator RNG was not rejected"
    )


print("STEP 3 VALIDATION PASSED")
print()
print("Ideal measurement:")
print(ideal)
print()
print("Biased measurement:")
print(biased)
print()
print("Seed 12345 measurement:")
print(noisy_a)
print()
print("Seed 54321 measurement:")
print(noisy_c)

STEP 3 VALIDATION PASSED

Ideal measurement:
[ 1.5  -2.    4.25]

Biased measurement:
[ 1.6  -2.2   4.55]

Seed 12345 measurement:
[ 1.42880875 -1.93681358  4.20646691]

Seed 54321 measurement:
[ 1.54124188 -1.97962547  4.31825572]


## Step 4 - Gaussian Noise Statistical Validation

The common measurement primitive has already demonstrated deterministic bias behavior and seeded reproducibility.

We now validate its stochastic behavior over many measurements.

For Gaussian sensor noise:

$$
n
\sim
\mathcal{N}(0,\sigma^2)
$$

the expected statistical behavior is:

$$
E[n]
\approx
0
$$

and:

$$
\operatorname{Std}(n)
\approx
\sigma.
$$

This validation uses a fixed seed so the experiment remains reproducible.

No sensor-specific model is introduced yet.

### Validation Criteria

This step passes only if, over a sufficiently large sample:

1. the measured noise mean remains close to zero;
2. the measured standard deviation remains close to the configured value;
3. every component satisfies the tolerance independently;
4. repeated execution with the same seed reproduces the same statistics.

In [4]:
# ---------------------------------------------------------------------
# Step 4 - Gaussian noise statistical validation
# ---------------------------------------------------------------------

sample_count = 100_000

truth = np.zeros(3, dtype=float)

configured_std = np.array([
    0.02,
    0.05,
    0.10,
])

rng = np.random.default_rng(202608)


samples = np.empty(
    (sample_count, 3),
    dtype=float,
)

for i in range(sample_count):
    samples[i] = measure_with_bias_noise(
        truth,
        bias=0.0,
        noise_std=configured_std,
        rng=rng,
    )


# ---------------------------------------------------------------------
# Recover pure measurement noise
# ---------------------------------------------------------------------

noise_samples = samples - truth

measured_mean = np.mean(
    noise_samples,
    axis=0,
)

measured_std = np.std(
    noise_samples,
    axis=0,
    ddof=0,
)


# ---------------------------------------------------------------------
# Validation tolerances
# ---------------------------------------------------------------------

mean_tolerance = 5.0e-4
std_relative_tolerance = 0.02


if not np.all(
    np.abs(measured_mean) < mean_tolerance
):
    raise AssertionError(
        "Measured Gaussian noise mean exceeded tolerance"
    )


relative_std_error = (
    np.abs(
        measured_std - configured_std
    )
    / configured_std
)

if not np.all(
    relative_std_error < std_relative_tolerance
):
    raise AssertionError(
        "Measured Gaussian noise standard deviation "
        "exceeded relative tolerance"
    )


# ---------------------------------------------------------------------
# Reproducibility check
# ---------------------------------------------------------------------

rng_repeat = np.random.default_rng(202608)

samples_repeat = np.empty(
    (sample_count, 3),
    dtype=float,
)

for i in range(sample_count):
    samples_repeat[i] = measure_with_bias_noise(
        truth,
        bias=0.0,
        noise_std=configured_std,
        rng=rng_repeat,
    )

np.testing.assert_array_equal(
    samples,
    samples_repeat,
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("STEP 4 VALIDATION PASSED")
print()

print("Configured standard deviation:")
print(configured_std)

print()

print("Measured mean:")
print(measured_mean)

print()

print("Measured standard deviation:")
print(measured_std)

print()

print("Relative standard-deviation error:")
print(relative_std_error)

print()

print(
    "Maximum absolute mean:",
    np.max(np.abs(measured_mean)),
)

print(
    "Maximum relative std error:",
    np.max(relative_std_error),
)

STEP 4 VALIDATION PASSED

Configured standard deviation:
[0.02 0.05 0.1 ]

Measured mean:
[ 5.88956993e-05 -1.49396553e-05  1.48906101e-04]

Measured standard deviation:
[0.01993781 0.05027066 0.09988384]

Relative standard-deviation error:
[0.00310971 0.00541313 0.00116156]

Maximum absolute mean: 0.00014890610088119486
Maximum relative std error: 0.005413134541715908


## Step 5 - Ideal Gyroscope Model

The first sensor-specific model is the IMU gyroscope.

HydroLab-3D stores generalized body velocity as:

$$
\nu =
\begin{bmatrix}
u &
v &
w &
p &
q &
r
\end{bmatrix}^{T}.
$$

The final three components are the body angular velocity:

$$
\boldsymbol{\omega}_B
=
\begin{bmatrix}
p\\
q\\
r
\end{bmatrix}.
$$

Therefore the ideal gyroscope measurement is:

$$
\boldsymbol{\omega}_{gyro}
=
\boldsymbol{\omega}_B.
$$

The measurement is:

```text
Quantity : angular velocity
Frame    : body frame
Units    : rad/s
```

This step implements only the ideal physical mapping.

Bias and Gaussian noise will be added only after the ideal model is validated.

### Validation Criteria

This step passes only if:

1. the returned gyroscope measurement equals $\nu[3:6]$ exactly;
2. translational velocity does not affect the measurement;
3. the output is a three-element vector;
4. malformed, non-finite, or incorrectly shaped generalized velocity is rejected explicitly.

In [5]:
def ideal_gyroscope_measurement(nu):
    """
    Return ideal body-frame angular velocity measured by a gyroscope.

    Parameters
    ----------
    nu : array_like, shape (6,)
        Generalized body velocity:
        [u, v, w, p, q, r].

    Returns
    -------
    numpy.ndarray, shape (3,)
        Body angular velocity [p, q, r] in rad/s.
    """

    nu = np.asarray(nu, dtype=float)

    if nu.shape != (6,):
        raise ValueError(
            "nu must have shape (6,) representing "
            "[u, v, w, p, q, r]"
        )

    if not np.all(np.isfinite(nu)):
        raise ValueError(
            "nu must contain only finite values"
        )

    return nu[3:6].copy()


# ---------------------------------------------------------------------
# Validation 1: exact angular-velocity extraction
# ---------------------------------------------------------------------

nu = np.array([
    1.5,
    -0.8,
    0.25,
    0.10,
    -0.20,
    0.35,
])

gyro = ideal_gyroscope_measurement(nu)

expected = np.array([
    0.10,
    -0.20,
    0.35,
])

np.testing.assert_array_equal(
    gyro,
    expected,
)


# ---------------------------------------------------------------------
# Validation 2: translational velocity must not affect gyroscope
# ---------------------------------------------------------------------

nu_same_rotation = np.array([
    -100.0,
    250.0,
    12.0,
    0.10,
    -0.20,
    0.35,
])

gyro_same_rotation = ideal_gyroscope_measurement(
    nu_same_rotation
)

np.testing.assert_array_equal(
    gyro_same_rotation,
    expected,
)


# ---------------------------------------------------------------------
# Validation 3: zero angular velocity
# ---------------------------------------------------------------------

nu_zero_rotation = np.array([
    2.0,
    -3.0,
    4.0,
    0.0,
    0.0,
    0.0,
])

gyro_zero = ideal_gyroscope_measurement(
    nu_zero_rotation
)

np.testing.assert_array_equal(
    gyro_zero,
    np.zeros(3),
)


# ---------------------------------------------------------------------
# Validation 4: output independence from input mutation
# ---------------------------------------------------------------------

nu_copy_test = np.array([
    0.0,
    0.0,
    0.0,
    1.0,
    2.0,
    3.0,
])

gyro_copy = ideal_gyroscope_measurement(
    nu_copy_test
)

nu_copy_test[3:] = 99.0

np.testing.assert_array_equal(
    gyro_copy,
    np.array([1.0, 2.0, 3.0]),
)


# ---------------------------------------------------------------------
# Validation 5: malformed inputs rejected
# ---------------------------------------------------------------------

invalid_inputs = [
    np.zeros(3),
    np.zeros(5),
    np.zeros(7),
    np.zeros((6, 1)),
    np.zeros((1, 6)),
    np.array([
        0.0,
        0.0,
        np.nan,
        0.0,
        0.0,
        0.0,
    ]),
]

for invalid in invalid_inputs:
    try:
        ideal_gyroscope_measurement(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError(
            f"Invalid gyroscope input was not rejected: "
            f"shape={np.asarray(invalid).shape}"
        )


print("STEP 5 VALIDATION PASSED")
print()

print("Input generalized velocity:")
print(nu)

print()

print("Ideal gyroscope measurement [rad/s]:")
print(gyro)

print()

print(
    "Translational-velocity independence:",
    np.array_equal(
        gyro,
        gyro_same_rotation,
    ),
)

print(
    "Zero-rotation measurement:",
    gyro_zero,
)

STEP 5 VALIDATION PASSED

Input generalized velocity:
[ 1.5  -0.8   0.25  0.1  -0.2   0.35]

Ideal gyroscope measurement [rad/s]:
[ 0.1  -0.2   0.35]

Translational-velocity independence: True
Zero-rotation measurement: [0. 0. 0.]


## Step 6 - Gyroscope Bias and Noise

The ideal gyroscope model has been validated.

We now introduce measurement imperfections using the common sensor primitive:

$$
\tilde{\boldsymbol{\omega}}
=
\boldsymbol{\omega}_B
+
\mathbf b_g
+
\mathbf n_g.
$$

Here:

- $\boldsymbol{\omega}_B$ is the ideal body-frame angular velocity;
- $\mathbf b_g$ is deterministic gyroscope bias;
- $\mathbf n_g$ is Gaussian measurement noise;
- $\tilde{\boldsymbol{\omega}}$ is the reported gyroscope measurement.

The physical quantity remains:

```text
Quantity : angular velocity
Frame    : body frame
Units    : rad/s
```

This step does not introduce drift, scale-factor error, clipping, or rate limitation.

### Validation Criteria

This step passes only if:

1. zero bias and zero noise reproduce the ideal gyroscope exactly;
2. deterministic bias shifts each axis exactly;
3. identical seeds reproduce identical noisy gyro measurements;
4. different seeds produce different stochastic realizations;
5. changing translational velocity still has no effect;
6. malformed bias and noise parameters are rejected by the common measurement primitive.

In [6]:
def gyroscope_measurement(
    nu,
    *,
    bias=0.0,
    noise_std=0.0,
    rng=None,
):
    """
    Return imperfect body-frame gyroscope measurement.

    Parameters
    ----------
    nu : array_like, shape (6,)
        Generalized body velocity:
        [u, v, w, p, q, r].
    bias : scalar or array_like
        Additive gyroscope bias in rad/s.
    noise_std : scalar or array_like
        Gaussian noise standard deviation in rad/s.
    rng : numpy.random.Generator or None
        Explicit random generator when noise is nonzero.

    Returns
    -------
    numpy.ndarray, shape (3,)
        Gyroscope measurement [p, q, r] in rad/s.
    """

    ideal = ideal_gyroscope_measurement(nu)

    measured = measure_with_bias_noise(
        ideal,
        bias=bias,
        noise_std=noise_std,
        rng=rng,
    )

    if measured.shape != (3,):
        raise ValueError(
            "Gyroscope bias and noise parameters must be "
            "scalar or broadcast-compatible with shape (3,)"
        )

    return measured


# ---------------------------------------------------------------------
# Validation 1: ideal recovery
# ---------------------------------------------------------------------

nu = np.array([
    2.0,
    -1.0,
    0.5,
    0.12,
    -0.08,
    0.30,
])

gyro_ideal = gyroscope_measurement(
    nu,
    bias=0.0,
    noise_std=0.0,
)

np.testing.assert_array_equal(
    gyro_ideal,
    np.array([0.12, -0.08, 0.30]),
)


# ---------------------------------------------------------------------
# Validation 2: exact deterministic bias
# ---------------------------------------------------------------------

gyro_bias = np.array([
    0.01,
    -0.02,
    0.005,
])

gyro_biased = gyroscope_measurement(
    nu,
    bias=gyro_bias,
    noise_std=0.0,
)

np.testing.assert_allclose(
    gyro_biased,
    gyro_ideal + gyro_bias,
    rtol=0.0,
    atol=0.0,
)


# ---------------------------------------------------------------------
# Validation 3: same seed -> same noisy measurement
# ---------------------------------------------------------------------

rng_a = np.random.default_rng(808)
rng_b = np.random.default_rng(808)

gyro_noisy_a = gyroscope_measurement(
    nu,
    bias=gyro_bias,
    noise_std=np.array([0.002, 0.003, 0.004]),
    rng=rng_a,
)

gyro_noisy_b = gyroscope_measurement(
    nu,
    bias=gyro_bias,
    noise_std=np.array([0.002, 0.003, 0.004]),
    rng=rng_b,
)

np.testing.assert_array_equal(
    gyro_noisy_a,
    gyro_noisy_b,
)


# ---------------------------------------------------------------------
# Validation 4: different seed -> different realization
# ---------------------------------------------------------------------

rng_c = np.random.default_rng(809)

gyro_noisy_c = gyroscope_measurement(
    nu,
    bias=gyro_bias,
    noise_std=np.array([0.002, 0.003, 0.004]),
    rng=rng_c,
)

if np.array_equal(
    gyro_noisy_a,
    gyro_noisy_c,
):
    raise AssertionError(
        "Different seeds unexpectedly produced "
        "identical gyroscope measurements"
    )


# ---------------------------------------------------------------------
# Validation 5: translational velocity independence
# ---------------------------------------------------------------------

nu_changed_translation = np.array([
    -500.0,
    800.0,
    42.0,
    0.12,
    -0.08,
    0.30,
])

gyro_translation_check = gyroscope_measurement(
    nu_changed_translation,
    bias=0.0,
    noise_std=0.0,
)

np.testing.assert_array_equal(
    gyro_translation_check,
    gyro_ideal,
)


# ---------------------------------------------------------------------
# Validation 6: malformed sensor parameters rejected
# ---------------------------------------------------------------------

try:
    gyroscope_measurement(
        nu,
        bias=np.zeros(2),
        noise_std=0.0,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Invalid gyroscope bias shape was not rejected"
    )


try:
    gyroscope_measurement(
        nu,
        bias=0.0,
        noise_std=np.array([
            0.01,
            -0.01,
            0.01,
        ]),
        rng=np.random.default_rng(1),
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Negative gyroscope noise standard deviation "
        "was not rejected"
    )


print("STEP 6 VALIDATION PASSED")
print()

print("Ideal gyroscope [rad/s]:")
print(gyro_ideal)

print()

print("Biased gyroscope [rad/s]:")
print(gyro_biased)

print()

print("Seed 808 noisy gyroscope [rad/s]:")
print(gyro_noisy_a)

print()

print("Seed 809 noisy gyroscope [rad/s]:")
print(gyro_noisy_c)

print()

print(
    "Translational-velocity independence:",
    np.array_equal(
        gyro_translation_check,
        gyro_ideal,
    ),
)

STEP 6 VALIDATION PASSED

Ideal gyroscope [rad/s]:
[ 0.12 -0.08  0.3 ]

Biased gyroscope [rad/s]:
[ 0.13  -0.1    0.305]

Seed 808 noisy gyroscope [rad/s]:
[ 0.12801372 -0.09772453  0.30741383]

Seed 809 noisy gyroscope [rad/s]:
[ 0.13165572 -0.0975072   0.3076564 ]

Translational-velocity independence: True


## Step 7 - Ideal Accelerometer Specific Force

The IMU accelerometer requires more care than the gyroscope.

HydroLab-3D stores translational velocity in the rotating body frame:

$$
\mathbf v_B
=
\begin{bmatrix}
u\\
v\\
w
\end{bmatrix}
$$

and body angular velocity as:

$$
\boldsymbol\omega_B
=
\begin{bmatrix}
p\\
q\\
r
\end{bmatrix}.
$$

Because the body frame rotates, the inertial acceleration expressed in that frame is:

$$
\mathbf a_B
=
\dot{\mathbf v}_B
+
\boldsymbol\omega_B
\times
\mathbf v_B.
$$

An ideal accelerometer measures **specific force**, not raw inertial acceleration:

$$
\mathbf f_B
=
\mathbf a_B
-
R_W^B\mathbf g_W.
$$

Therefore:

$$
\boxed{
\mathbf f_B
=
\dot{\mathbf v}_B
+
\boldsymbol\omega_B
\times
\mathbf v_B
-
R_W^B\mathbf g_W
}
$$

HydroLab-3D uses world $+z$ upward, so gravitational acceleration in the world frame is:

$$
\mathbf g_W
=
\begin{bmatrix}
0\\
0\\
-g
\end{bmatrix}.
$$

The existing HydroLab rotation convention is:

$$
R_B^W
=
R_z(\psi)
R_y(\theta)
R_x(\phi),
$$

therefore:

$$
R_W^B
=
\left(R_B^W\right)^T.
$$

For example, a stationary vehicle at zero orientation reports:

$$
\mathbf f_B
=
\begin{bmatrix}
0\\
0\\
g
\end{bmatrix}.
$$

For a stationary vehicle rolled by $+90^\circ$:

$$
R_W^B\mathbf g_W
=
\begin{bmatrix}
0\\
-g\\
0
\end{bmatrix},
$$

so the accelerometer reports:

$$
\mathbf f_B
=
\begin{bmatrix}
0\\
+g\\
0
\end{bmatrix}.
$$

This sign follows directly from HydroLab-3D's validated body-to-world rotation convention.

The measurement is:

```text
Quantity : specific force
Frame    : body frame
Units    : m/s^2
```

This step implements only ideal accelerometer physics.

Bias and stochastic noise are not yet introduced.

### Validation Criteria

This step passes only if:

1. a level stationary vehicle reports $[0,0,g]$;
2. free-fall acceleration produces zero specific force;
3. a non-rotating vehicle with known body acceleration produces the analytically expected result;
4. the $\boldsymbol\omega_B \times \mathbf v_B$ transport term is applied correctly;
5. gravity transforms consistently with HydroLab-3D's existing orientation convention;
6. both positive-roll and positive-pitch reduced cases have the correct signs;
7. malformed state, acceleration, and gravity inputs are rejected explicitly.

In [8]:
def ideal_accelerometer_measurement(
    eta,
    nu,
    nu_dot,
    *,
    gravity=9.81,
):
    """
    Return ideal body-frame accelerometer specific force.

    Parameters
    ----------
    eta : array_like, shape (6,)
        Pose [x, y, z, roll, pitch, yaw].

    nu : array_like, shape (6,)
        Body velocity [u, v, w, p, q, r].

    nu_dot : array_like, shape (6,)
        Body-frame generalized velocity derivative
        [u_dot, v_dot, w_dot, p_dot, q_dot, r_dot].

    gravity : float
        Positive gravitational acceleration magnitude in m/s^2.

    Returns
    -------
    numpy.ndarray, shape (3,)
        Ideal accelerometer specific force expressed in the
        body frame, in m/s^2.
    """

    eta = np.asarray(eta, dtype=float)
    nu = np.asarray(nu, dtype=float)
    nu_dot = np.asarray(nu_dot, dtype=float)

    # -----------------------------------------------------------------
    # Shape validation
    # -----------------------------------------------------------------

    if eta.shape != (6,):
        raise ValueError(
            "eta must have shape (6,) representing "
            "[x, y, z, roll, pitch, yaw]"
        )

    if nu.shape != (6,):
        raise ValueError(
            "nu must have shape (6,) representing "
            "[u, v, w, p, q, r]"
        )

    if nu_dot.shape != (6,):
        raise ValueError(
            "nu_dot must have shape (6,) representing "
            "[u_dot, v_dot, w_dot, p_dot, q_dot, r_dot]"
        )

    # -----------------------------------------------------------------
    # Finiteness validation
    # -----------------------------------------------------------------

    if not np.all(np.isfinite(eta)):
        raise ValueError(
            "eta must contain only finite values"
        )

    if not np.all(np.isfinite(nu)):
        raise ValueError(
            "nu must contain only finite values"
        )

    if not np.all(np.isfinite(nu_dot)):
        raise ValueError(
            "nu_dot must contain only finite values"
        )

    # -----------------------------------------------------------------
    # Gravity validation
    # -----------------------------------------------------------------

    gravity = float(gravity)

    if not np.isfinite(gravity):
        raise ValueError(
            "gravity must be finite"
        )

    if gravity <= 0.0:
        raise ValueError(
            "gravity must be strictly positive"
        )

    # -----------------------------------------------------------------
    # Orientation transform
    # -----------------------------------------------------------------

    phi, theta, psi = eta[3:6]

    R_BW = rotation_body_to_world(
        phi,
        theta,
        psi,
    )

    R_WB = R_BW.T

    # -----------------------------------------------------------------
    # Body-frame quantities
    # -----------------------------------------------------------------

    v_body = nu[:3]
    omega_body = nu[3:6]
    v_dot_body = nu_dot[:3]

    # HydroLab world convention:
    # +z_W upward, therefore gravity points along -z_W.
    gravity_world = np.array([
        0.0,
        0.0,
        -gravity,
    ])

    # -----------------------------------------------------------------
    # Inertial acceleration expressed in body coordinates
    #
    # a_B = v_dot_B + omega_B x v_B
    # -----------------------------------------------------------------

    rotational_transport = np.cross(
        omega_body,
        v_body,
    )

    inertial_accel_body = (
        v_dot_body
        + rotational_transport
    )

    # -----------------------------------------------------------------
    # Gravity expressed in body frame
    # -----------------------------------------------------------------

    gravity_body = (
        R_WB
        @ gravity_world
    )

    # -----------------------------------------------------------------
    # Accelerometer specific force
    #
    # f_B = a_B - g_B
    # -----------------------------------------------------------------

    specific_force = (
        inertial_accel_body
        - gravity_body
    )

    return specific_force.copy()


# =====================================================================
# Validation 1: level stationary vehicle
# =====================================================================

eta_level = np.zeros(6)
nu_stationary = np.zeros(6)
nu_dot_stationary = np.zeros(6)

accel_stationary = ideal_accelerometer_measurement(
    eta_level,
    nu_stationary,
    nu_dot_stationary,
)

expected_stationary = np.array([
    0.0,
    0.0,
    9.81,
])

np.testing.assert_allclose(
    accel_stationary,
    expected_stationary,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 2: free fall -> zero specific force
# =====================================================================

nu_dot_free_fall = np.array([
    0.0,
    0.0,
    -9.81,
    0.0,
    0.0,
    0.0,
])

accel_free_fall = ideal_accelerometer_measurement(
    eta_level,
    nu_stationary,
    nu_dot_free_fall,
)

np.testing.assert_allclose(
    accel_free_fall,
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 3: known non-rotating body acceleration
# =====================================================================

nu_dot_known = np.array([
    1.5,
    -2.0,
    0.5,
    0.0,
    0.0,
    0.0,
])

accel_known = ideal_accelerometer_measurement(
    eta_level,
    nu_stationary,
    nu_dot_known,
)

expected_known = np.array([
    1.5,
    -2.0,
    10.31,
])

np.testing.assert_allclose(
    accel_known,
    expected_known,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 4: omega x v contribution
# =====================================================================

nu_rotating = np.array([
    2.0,
    0.0,
    0.0,
    0.0,
    0.0,
    1.0,
])

nu_dot_zero = np.zeros(6)

accel_rotating = ideal_accelerometer_measurement(
    eta_level,
    nu_rotating,
    nu_dot_zero,
)

# omega_B = [0, 0, 1]
# v_B     = [2, 0, 0]
#
# omega_B x v_B = [0, 2, 0]

expected_rotating = np.array([
    0.0,
    2.0,
    9.81,
])

np.testing.assert_allclose(
    accel_rotating,
    expected_rotating,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 5: +90 degree roll gravity transformation
# =====================================================================

eta_roll_90 = np.array([
    0.0,
    0.0,
    0.0,
    np.pi / 2.0,
    0.0,
    0.0,
])

accel_roll_90 = ideal_accelerometer_measurement(
    eta_roll_90,
    nu_stationary,
    nu_dot_stationary,
)

# Under HydroLab's R_B^W = Rz Ry Rx convention:
#
# g_B = [0, -g, 0]
#
# Therefore:
#
# f_B = -g_B = [0, +g, 0]

expected_roll_90 = np.array([
    0.0,
    9.81,
    0.0,
])

np.testing.assert_allclose(
    accel_roll_90,
    expected_roll_90,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 6: +90 degree pitch gravity transformation
# =====================================================================

eta_pitch_90 = np.array([
    0.0,
    0.0,
    0.0,
    0.0,
    np.pi / 2.0,
    0.0,
])

accel_pitch_90 = ideal_accelerometer_measurement(
    eta_pitch_90,
    nu_stationary,
    nu_dot_stationary,
)

# For +90 degree pitch:
#
# g_B = [+g, 0, 0]
#
# Therefore:
#
# f_B = [-g, 0, 0]

expected_pitch_90 = np.array([
    -9.81,
    0.0,
    0.0,
])

np.testing.assert_allclose(
    accel_pitch_90,
    expected_pitch_90,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 7: arbitrary orientation must match direct transform
# =====================================================================

eta_arbitrary = np.array([
    4.0,
    -3.0,
    -12.0,
    0.31,
    -0.27,
    0.82,
])

R_BW_arbitrary = rotation_body_to_world(
    eta_arbitrary[3],
    eta_arbitrary[4],
    eta_arbitrary[5],
)

gravity_world = np.array([
    0.0,
    0.0,
    -9.81,
])

expected_arbitrary = -(
    R_BW_arbitrary.T
    @ gravity_world
)

accel_arbitrary = ideal_accelerometer_measurement(
    eta_arbitrary,
    nu_stationary,
    nu_dot_stationary,
)

np.testing.assert_allclose(
    accel_arbitrary,
    expected_arbitrary,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 8: output must not alias input storage
# =====================================================================

eta_copy_test = np.zeros(6)
nu_copy_test = np.zeros(6)
nu_dot_copy_test = np.zeros(6)

accel_copy_test = ideal_accelerometer_measurement(
    eta_copy_test,
    nu_copy_test,
    nu_dot_copy_test,
)

eta_copy_test[:] = 99.0
nu_copy_test[:] = 99.0
nu_dot_copy_test[:] = 99.0

np.testing.assert_allclose(
    accel_copy_test,
    np.array([0.0, 0.0, 9.81]),
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 9: malformed inputs rejected
# =====================================================================

invalid_cases = [
    (
        np.zeros(5),
        np.zeros(6),
        np.zeros(6),
    ),
    (
        np.zeros(6),
        np.zeros(5),
        np.zeros(6),
    ),
    (
        np.zeros(6),
        np.zeros(6),
        np.zeros(5),
    ),
    (
        np.zeros((6, 1)),
        np.zeros(6),
        np.zeros(6),
    ),
    (
        np.zeros(6),
        np.zeros((1, 6)),
        np.zeros(6),
    ),
    (
        np.zeros(6),
        np.zeros(6),
        np.zeros((6, 1)),
    ),
    (
        np.array([
            0.0,
            0.0,
            0.0,
            np.nan,
            0.0,
            0.0,
        ]),
        np.zeros(6),
        np.zeros(6),
    ),
    (
        np.zeros(6),
        np.array([
            0.0,
            0.0,
            0.0,
            np.inf,
            0.0,
            0.0,
        ]),
        np.zeros(6),
    ),
    (
        np.zeros(6),
        np.zeros(6),
        np.array([
            0.0,
            np.nan,
            0.0,
            0.0,
            0.0,
            0.0,
        ]),
    ),
]

for bad_eta, bad_nu, bad_nu_dot in invalid_cases:
    try:
        ideal_accelerometer_measurement(
            bad_eta,
            bad_nu,
            bad_nu_dot,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Invalid accelerometer input was not rejected"
        )


# =====================================================================
# Validation 10: invalid gravity rejected
# =====================================================================

invalid_gravity_values = [
    -9.81,
    0.0,
    np.nan,
    np.inf,
    -np.inf,
]

for bad_gravity in invalid_gravity_values:
    try:
        ideal_accelerometer_measurement(
            eta_level,
            nu_stationary,
            nu_dot_stationary,
            gravity=bad_gravity,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            f"Invalid gravity value was not rejected: "
            f"{bad_gravity}"
        )


# =====================================================================
# Report
# =====================================================================

print("STEP 7 VALIDATION PASSED")
print()

print("Stationary level specific force [m/s^2]:")
print(accel_stationary)

print()

print("Free-fall specific force [m/s^2]:")
print(accel_free_fall)

print()

print("Known body-acceleration case [m/s^2]:")
print(accel_known)

print()

print("Rotating-body case [m/s^2]:")
print(accel_rotating)

print()

print("+90 deg roll stationary case [m/s^2]:")
print(accel_roll_90)

print()

print("+90 deg pitch stationary case [m/s^2]:")
print(accel_pitch_90)

print()

print("Arbitrary-orientation stationary case [m/s^2]:")
print(accel_arbitrary)

print()

print("Expected arbitrary-orientation value [m/s^2]:")
print(expected_arbitrary)

STEP 7 VALIDATION PASSED

Stationary level specific force [m/s^2]:
[0.   0.   9.81]

Free-fall specific force [m/s^2]:
[0. 0. 0.]

Known body-acceleration case [m/s^2]:
[ 1.5  -2.   10.31]

Rotating-body case [m/s^2]:
[0.   2.   9.81]

+90 deg roll stationary case [m/s^2]:
[0.00000000e+00 9.81000000e+00 6.00689255e-16]

+90 deg pitch stationary case [m/s^2]:
[-9.81000000e+00  0.00000000e+00  6.00689255e-16]

Arbitrary-orientation stationary case [m/s^2]:
[2.61663539 2.88420509 9.00392582]

Expected arbitrary-orientation value [m/s^2]:
[2.61663539 2.88420509 9.00392582]


## Step 8 - Accelerometer Bias and Noise

The ideal accelerometer specific-force model is now validated against HydroLab-3D's existing frame and orientation conventions.

We next introduce lightweight measurement imperfections:

$$
\tilde{\mathbf f}_B
=
\mathbf f_B
+
\mathbf b_a
+
\mathbf n_a
$$

where:

- $\mathbf f_B$ is ideal body-frame specific force;
- $\mathbf b_a$ is deterministic accelerometer bias;
- $\mathbf n_a$ is Gaussian measurement noise;
- $\tilde{\mathbf f}_B$ is the reported accelerometer measurement.

The measurement remains:

```text
Quantity : specific force
Frame    : body frame
Units    : m/s^2
```

The implementation reuses the already validated ideal accelerometer model and common measurement primitive.

No drift, scale-factor error, clipping, or rate limitation is introduced in this step.

### Validation Criteria

This step passes only if:

1. zero bias and zero noise reproduce ideal specific force exactly;
2. deterministic bias shifts each accelerometer axis exactly;
3. identical seeds reproduce identical noisy measurements;
4. different seeds produce different stochastic realizations;
5. orientation-dependent specific-force physics remains unchanged by the wrapper;
6. malformed bias and noise parameters are rejected explicitly.

In [9]:
def accelerometer_measurement(
    eta,
    nu,
    nu_dot,
    *,
    gravity=9.81,
    bias=0.0,
    noise_std=0.0,
    rng=None,
):
    """
    Return imperfect body-frame accelerometer measurement.

    Parameters
    ----------
    eta : array_like, shape (6,)
        Pose [x, y, z, roll, pitch, yaw].

    nu : array_like, shape (6,)
        Body velocity [u, v, w, p, q, r].

    nu_dot : array_like, shape (6,)
        Generalized body velocity derivative.

    gravity : float
        Positive gravitational acceleration magnitude in m/s^2.

    bias : scalar or array_like
        Additive accelerometer bias in m/s^2.

    noise_std : scalar or array_like
        Gaussian noise standard deviation in m/s^2.

    rng : numpy.random.Generator or None
        Explicit random generator when noise is nonzero.

    Returns
    -------
    numpy.ndarray, shape (3,)
        Body-frame accelerometer measurement in m/s^2.
    """

    ideal = ideal_accelerometer_measurement(
        eta,
        nu,
        nu_dot,
        gravity=gravity,
    )

    measured = measure_with_bias_noise(
        ideal,
        bias=bias,
        noise_std=noise_std,
        rng=rng,
    )

    if measured.shape != (3,):
        raise ValueError(
            "Accelerometer bias and noise parameters must be "
            "scalar or broadcast-compatible with shape (3,)"
        )

    return measured


# =====================================================================
# Validation 1: ideal recovery
# =====================================================================

eta_test = np.array([
    3.0,
    -2.0,
    -15.0,
    0.20,
    -0.15,
    0.40,
])

nu_test = np.array([
    1.2,
    -0.4,
    0.3,
    0.05,
    -0.08,
    0.12,
])

nu_dot_test = np.array([
    0.4,
    -0.2,
    0.1,
    0.01,
    -0.02,
    0.03,
])

accel_ideal = ideal_accelerometer_measurement(
    eta_test,
    nu_test,
    nu_dot_test,
)

accel_wrapper_ideal = accelerometer_measurement(
    eta_test,
    nu_test,
    nu_dot_test,
    bias=0.0,
    noise_std=0.0,
)

np.testing.assert_array_equal(
    accel_wrapper_ideal,
    accel_ideal,
)


# =====================================================================
# Validation 2: exact deterministic bias
# =====================================================================

accel_bias = np.array([
    0.05,
    -0.03,
    0.08,
])

accel_biased = accelerometer_measurement(
    eta_test,
    nu_test,
    nu_dot_test,
    bias=accel_bias,
    noise_std=0.0,
)

np.testing.assert_allclose(
    accel_biased,
    accel_ideal + accel_bias,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 3: same seed -> identical noisy measurement
# =====================================================================

accel_noise_std = np.array([
    0.01,
    0.015,
    0.02,
])

rng_a = np.random.default_rng(8080)
rng_b = np.random.default_rng(8080)

accel_noisy_a = accelerometer_measurement(
    eta_test,
    nu_test,
    nu_dot_test,
    bias=accel_bias,
    noise_std=accel_noise_std,
    rng=rng_a,
)

accel_noisy_b = accelerometer_measurement(
    eta_test,
    nu_test,
    nu_dot_test,
    bias=accel_bias,
    noise_std=accel_noise_std,
    rng=rng_b,
)

np.testing.assert_array_equal(
    accel_noisy_a,
    accel_noisy_b,
)


# =====================================================================
# Validation 4: different seed -> different realization
# =====================================================================

rng_c = np.random.default_rng(8081)

accel_noisy_c = accelerometer_measurement(
    eta_test,
    nu_test,
    nu_dot_test,
    bias=accel_bias,
    noise_std=accel_noise_std,
    rng=rng_c,
)

if np.array_equal(
    accel_noisy_a,
    accel_noisy_c,
):
    raise AssertionError(
        "Different seeds unexpectedly produced "
        "identical accelerometer measurements"
    )


# =====================================================================
# Validation 5: orientation physics survives the wrapper unchanged
# =====================================================================

eta_roll_90 = np.array([
    0.0,
    0.0,
    0.0,
    np.pi / 2.0,
    0.0,
    0.0,
])

accel_roll_wrapper = accelerometer_measurement(
    eta_roll_90,
    np.zeros(6),
    np.zeros(6),
    bias=0.0,
    noise_std=0.0,
)

np.testing.assert_allclose(
    accel_roll_wrapper,
    np.array([
        0.0,
        9.81,
        0.0,
    ]),
    rtol=0.0,
    atol=1e-12,
)


eta_pitch_90 = np.array([
    0.0,
    0.0,
    0.0,
    0.0,
    np.pi / 2.0,
    0.0,
])

accel_pitch_wrapper = accelerometer_measurement(
    eta_pitch_90,
    np.zeros(6),
    np.zeros(6),
    bias=0.0,
    noise_std=0.0,
)

np.testing.assert_allclose(
    accel_pitch_wrapper,
    np.array([
        -9.81,
        0.0,
        0.0,
    ]),
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 6: malformed sensor parameters rejected
# =====================================================================

try:
    accelerometer_measurement(
        eta_test,
        nu_test,
        nu_dot_test,
        bias=np.zeros(2),
        noise_std=0.0,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Invalid accelerometer bias shape was not rejected"
    )


try:
    accelerometer_measurement(
        eta_test,
        nu_test,
        nu_dot_test,
        bias=0.0,
        noise_std=np.array([
            0.01,
            -0.02,
            0.03,
        ]),
        rng=np.random.default_rng(1),
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Negative accelerometer noise standard deviation "
        "was not rejected"
    )


try:
    accelerometer_measurement(
        eta_test,
        nu_test,
        nu_dot_test,
        bias=0.0,
        noise_std=0.01,
        rng="not-a-generator",
    )
except TypeError:
    pass
else:
    raise AssertionError(
        "Invalid accelerometer RNG was not rejected"
    )


# =====================================================================
# Report
# =====================================================================

print("STEP 8 VALIDATION PASSED")
print()

print("Ideal accelerometer [m/s^2]:")
print(accel_ideal)

print()

print("Biased accelerometer [m/s^2]:")
print(accel_biased)

print()

print("Seed 8080 noisy accelerometer [m/s^2]:")
print(accel_noisy_a)

print()

print("Seed 8081 noisy accelerometer [m/s^2]:")
print(accel_noisy_c)

print()

print("+90 deg roll wrapper check [m/s^2]:")
print(accel_roll_wrapper)

print()

print("+90 deg pitch wrapper check [m/s^2]:")
print(accel_pitch_wrapper)

STEP 8 VALIDATION PASSED

Ideal accelerometer [m/s^2]:
[1.88998808 1.85606157 9.68249318]

Biased accelerometer [m/s^2]:
[1.93998808 1.82606157 9.76249318]

Seed 8080 noisy accelerometer [m/s^2]:
[1.91992183 1.81458587 9.76516877]

Seed 8081 noisy accelerometer [m/s^2]:
[1.93530961 1.84107873 9.73815527]

+90 deg roll wrapper check [m/s^2]:
[0.00000000e+00 9.81000000e+00 6.00689255e-16]

+90 deg pitch wrapper check [m/s^2]:
[-9.81000000e+00  0.00000000e+00  6.00689255e-16]


## Step 9 - Ideal Depth Sensor

HydroLab-3D uses a world frame in which:

```text
+z_W = upward
```

and the water surface is:

$$
z_W = 0.
$$

A submerged vehicle therefore normally satisfies:

$$
z_W \leq 0.
$$

Depth below the water surface is defined as:

$$
\boxed{
d=-z_W
}
$$

Therefore:

```text
z_W =   0 m  → depth =  0 m
z_W =  -5 m  → depth =  5 m
z_W = -20 m  → depth = 20 m
```

The initial depth sensor measures only this geometric depth.

The measurement is:

```text
Quantity : depth below water surface
Frame    : world vertical reference
Units    : m
```

This step implements only the ideal physical mapping.

Bias and stochastic noise will be introduced after the sign convention and boundary behavior are validated.

### Validation Criteria

This step passes only if:

1. $z=0$ produces zero depth;
2. increasingly negative world $z$ produces increasing positive depth;
3. horizontal position and orientation do not affect depth;
4. states above the modeled water surface are rejected explicitly;
5. malformed and non-finite pose inputs are rejected explicitly.

In [10]:
def ideal_depth_measurement(
    eta,
    *,
    surface_z=0.0,
):
    """
    Return ideal depth below the modeled water surface.

    Parameters
    ----------
    eta : array_like, shape (6,)
        Pose [x, y, z, roll, pitch, yaw].

    surface_z : float
        World-frame z coordinate of the water surface.

    Returns
    -------
    float
        Positive depth below the water surface in meters.

    Notes
    -----
    HydroLab-3D uses +z_W upward.

    Therefore:

        depth = surface_z - z

    For the default surface_z = 0:

        depth = -z
    """

    eta = np.asarray(eta, dtype=float)

    if eta.shape != (6,):
        raise ValueError(
            "eta must have shape (6,) representing "
            "[x, y, z, roll, pitch, yaw]"
        )

    if not np.all(np.isfinite(eta)):
        raise ValueError(
            "eta must contain only finite values"
        )

    surface_z = float(surface_z)

    if not np.isfinite(surface_z):
        raise ValueError(
            "surface_z must be finite"
        )

    z_world = eta[2]

    depth = surface_z - z_world

    # Tiny floating-point roundoff around the surface is allowed,
    # but a genuinely above-surface vehicle is outside this
    # underwater depth-sensor model.
    tolerance = 1e-12

    if depth < -tolerance:
        raise ValueError(
            "Vehicle is above the modeled water surface; "
            "underwater depth is undefined for this state"
        )

    if abs(depth) <= tolerance:
        depth = 0.0

    return float(depth)


# =====================================================================
# Validation 1: surface -> zero depth
# =====================================================================

eta_surface = np.array([
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
])

depth_surface = ideal_depth_measurement(
    eta_surface
)

if depth_surface != 0.0:
    raise AssertionError(
        "Water-surface state did not produce zero depth"
    )


# =====================================================================
# Validation 2: known submerged depths
# =====================================================================

eta_5m = np.array([
    0.0,
    0.0,
    -5.0,
    0.0,
    0.0,
    0.0,
])

eta_20m = np.array([
    0.0,
    0.0,
    -20.0,
    0.0,
    0.0,
    0.0,
])

depth_5m = ideal_depth_measurement(
    eta_5m
)

depth_20m = ideal_depth_measurement(
    eta_20m
)

np.testing.assert_allclose(
    depth_5m,
    5.0,
    rtol=0.0,
    atol=0.0,
)

np.testing.assert_allclose(
    depth_20m,
    20.0,
    rtol=0.0,
    atol=0.0,
)

if not depth_20m > depth_5m:
    raise AssertionError(
        "More negative world z did not produce greater depth"
    )


# =====================================================================
# Validation 3: horizontal position and orientation independence
# =====================================================================

eta_same_depth = np.array([
    125.0,
    -340.0,
    -20.0,
    1.1,
    -0.7,
    2.4,
])

depth_same = ideal_depth_measurement(
    eta_same_depth
)

np.testing.assert_allclose(
    depth_same,
    20.0,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 4: nonzero water-surface reference
# =====================================================================

eta_reference_test = np.array([
    0.0,
    0.0,
    -7.0,
    0.0,
    0.0,
    0.0,
])

depth_reference = ideal_depth_measurement(
    eta_reference_test,
    surface_z=3.0,
)

np.testing.assert_allclose(
    depth_reference,
    10.0,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 5: above-surface state rejected
# =====================================================================

eta_above_surface = np.array([
    0.0,
    0.0,
    0.25,
    0.0,
    0.0,
    0.0,
])

try:
    ideal_depth_measurement(
        eta_above_surface
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Above-surface state was not rejected"
    )


# =====================================================================
# Validation 6: malformed inputs rejected
# =====================================================================

invalid_eta_cases = [
    np.zeros(5),
    np.zeros(7),
    np.zeros((6, 1)),
    np.zeros((1, 6)),
    np.array([
        0.0,
        0.0,
        np.nan,
        0.0,
        0.0,
        0.0,
    ]),
    np.array([
        0.0,
        0.0,
        -1.0,
        np.inf,
        0.0,
        0.0,
    ]),
]

for bad_eta in invalid_eta_cases:
    try:
        ideal_depth_measurement(
            bad_eta
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Invalid depth-sensor pose input was not rejected"
        )


invalid_surface_values = [
    np.nan,
    np.inf,
    -np.inf,
]

for bad_surface in invalid_surface_values:
    try:
        ideal_depth_measurement(
            eta_5m,
            surface_z=bad_surface,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            f"Invalid surface_z was not rejected: "
            f"{bad_surface}"
        )


# =====================================================================
# Report
# =====================================================================

print("STEP 9 VALIDATION PASSED")
print()

print("Surface depth [m]:")
print(depth_surface)

print()

print("Depth at z = -5 m:")
print(depth_5m)

print()

print("Depth at z = -20 m:")
print(depth_20m)

print()

print(
    "Same depth with arbitrary x/y/orientation [m]:"
)
print(depth_same)

print()

print(
    "Depth with surface_z = 3 m and z = -7 m:"
)
print(depth_reference)

STEP 9 VALIDATION PASSED

Surface depth [m]:
0.0

Depth at z = -5 m:
5.0

Depth at z = -20 m:
20.0

Same depth with arbitrary x/y/orientation [m]:
20.0

Depth with surface_z = 3 m and z = -7 m:
10.0


## Step 10 - Depth Sensor Bias and Noise

The ideal depth mapping has been validated against HydroLab-3D's world-$z$ convention.

We now introduce lightweight measurement imperfections:

$$
\tilde d
=
d
+
b_d
+
n_d
$$

where:

- $d$ is ideal depth below the modeled water surface;
- $b_d$ is deterministic depth-sensor bias;
- $n_d$ is Gaussian measurement noise;
- $\tilde d$ is the reported depth measurement.

The measurement remains:

```text
Quantity : depth below water surface
Frame    : world vertical reference
Units    : m
```

The implementation reuses:

1. the validated ideal depth model;
2. the validated common bias/noise primitive.

No pressure-electronics model, hydrostatic pressure conversion, drift, clipping, or update-rate limitation is introduced in this step.

### Validation Criteria

This step passes only if:

1. zero bias and zero noise reproduce ideal depth exactly;
2. deterministic bias shifts depth exactly;
3. identical seeds reproduce identical noisy depth measurements;
4. different seeds produce different stochastic realizations;
5. the wrapper preserves the ideal surface/depth sign convention;
6. invalid bias, noise, RNG, and above-surface states are rejected explicitly.

In [11]:
def depth_measurement(
    eta,
    *,
    surface_z=0.0,
    bias=0.0,
    noise_std=0.0,
    rng=None,
):
    """
    Return imperfect depth measurement.

    Parameters
    ----------
    eta : array_like, shape (6,)
        Pose [x, y, z, roll, pitch, yaw].

    surface_z : float
        World-frame z coordinate of the water surface.

    bias : scalar
        Additive depth bias in meters.

    noise_std : scalar
        Gaussian depth noise standard deviation in meters.

    rng : numpy.random.Generator or None
        Explicit random generator when noise is nonzero.

    Returns
    -------
    float
        Depth measurement in meters.
    """

    ideal = ideal_depth_measurement(
        eta,
        surface_z=surface_z,
    )

    measured = measure_with_bias_noise(
        ideal,
        bias=bias,
        noise_std=noise_std,
        rng=rng,
    )

    measured = np.asarray(
        measured,
        dtype=float,
    )

    if measured.shape != ():
        raise ValueError(
            "Depth bias and noise_std must be scalar"
        )

    return float(measured)


# =====================================================================
# Validation 1: ideal recovery
# =====================================================================

eta_depth_test = np.array([
    12.0,
    -4.0,
    -18.5,
    0.3,
    -0.2,
    1.1,
])

depth_ideal = ideal_depth_measurement(
    eta_depth_test
)

depth_wrapper_ideal = depth_measurement(
    eta_depth_test,
    bias=0.0,
    noise_std=0.0,
)

np.testing.assert_allclose(
    depth_wrapper_ideal,
    depth_ideal,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 2: exact deterministic bias
# =====================================================================

depth_bias = 0.35

depth_biased = depth_measurement(
    eta_depth_test,
    bias=depth_bias,
    noise_std=0.0,
)

np.testing.assert_allclose(
    depth_biased,
    depth_ideal + depth_bias,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 3: same seed -> same noisy measurement
# =====================================================================

rng_a = np.random.default_rng(8100)
rng_b = np.random.default_rng(8100)

depth_noisy_a = depth_measurement(
    eta_depth_test,
    bias=depth_bias,
    noise_std=0.05,
    rng=rng_a,
)

depth_noisy_b = depth_measurement(
    eta_depth_test,
    bias=depth_bias,
    noise_std=0.05,
    rng=rng_b,
)

np.testing.assert_allclose(
    depth_noisy_a,
    depth_noisy_b,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 4: different seed -> different realization
# =====================================================================

rng_c = np.random.default_rng(8101)

depth_noisy_c = depth_measurement(
    eta_depth_test,
    bias=depth_bias,
    noise_std=0.05,
    rng=rng_c,
)

if depth_noisy_a == depth_noisy_c:
    raise AssertionError(
        "Different seeds unexpectedly produced "
        "identical depth measurements"
    )


# =====================================================================
# Validation 5: surface and sign convention preserved
# =====================================================================

eta_surface = np.zeros(6)

depth_surface_wrapper = depth_measurement(
    eta_surface,
    bias=0.0,
    noise_std=0.0,
)

np.testing.assert_allclose(
    depth_surface_wrapper,
    0.0,
    rtol=0.0,
    atol=0.0,
)


eta_deeper = np.array([
    0.0,
    0.0,
    -25.0,
    0.0,
    0.0,
    0.0,
])

depth_deeper = depth_measurement(
    eta_deeper,
    bias=0.0,
    noise_std=0.0,
)

if depth_deeper <= depth_wrapper_ideal:
    raise AssertionError(
        "More-negative world z did not produce greater depth"
    )


# =====================================================================
# Validation 6: invalid bias/noise/RNG rejected
# =====================================================================

invalid_scalar_cases = [
    (
        np.array([0.1, 0.2]),
        0.0,
    ),
    (
        0.0,
        np.array([0.01, 0.02]),
    ),
]

for bad_bias, bad_std in invalid_scalar_cases:
    try:
        depth_measurement(
            eta_depth_test,
            bias=bad_bias,
            noise_std=bad_std,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Non-scalar depth bias/noise was not rejected"
        )


try:
    depth_measurement(
        eta_depth_test,
        bias=0.0,
        noise_std=-0.01,
        rng=np.random.default_rng(1),
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Negative depth noise standard deviation "
        "was not rejected"
    )


try:
    depth_measurement(
        eta_depth_test,
        bias=0.0,
        noise_std=0.01,
        rng="not-a-generator",
    )
except TypeError:
    pass
else:
    raise AssertionError(
        "Invalid depth RNG was not rejected"
    )


# =====================================================================
# Validation 7: above-surface state still rejected
# =====================================================================

eta_above_surface = np.array([
    0.0,
    0.0,
    0.1,
    0.0,
    0.0,
    0.0,
])

try:
    depth_measurement(
        eta_above_surface,
        bias=0.0,
        noise_std=0.0,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Above-surface state was not rejected "
        "by the depth wrapper"
    )


# =====================================================================
# Report
# =====================================================================

print("STEP 10 VALIDATION PASSED")
print()

print("Ideal depth [m]:")
print(depth_ideal)

print()

print("Biased depth [m]:")
print(depth_biased)

print()

print("Seed 8100 noisy depth [m]:")
print(depth_noisy_a)

print()

print("Seed 8101 noisy depth [m]:")
print(depth_noisy_c)

print()

print("Surface depth through wrapper [m]:")
print(depth_surface_wrapper)

print()

print("Deeper-state depth [m]:")
print(depth_deeper)

STEP 10 VALIDATION PASSED

Ideal depth [m]:
18.5

Biased depth [m]:
18.85

Seed 8100 noisy depth [m]:
18.87533775300926

Seed 8101 noisy depth [m]:
18.807164586427497

Surface depth through wrapper [m]:
0.0

Deeper-state depth [m]:
25.0


## Step 11 - Ideal DVL Water-Relative Velocity

The initial HydroLab-3D DVL abstraction measures translational vehicle velocity relative to the surrounding water.

The vehicle translational velocity is stored in the body frame as:

$$
\mathbf v_B
=
\begin{bmatrix}
u\\
v\\
w
\end{bmatrix}.
$$

Ocean current is defined in the world frame.

HydroLab-3D already provides the validated production transformation:

$$
\mathbf v_{c,B}
=
R_W^B
\mathbf v_{c,W}.
$$

The generalized water-relative body velocity is:

$$
\nu_r
=
\nu
-
\nu_c.
$$

Its translational portion is therefore:

$$
\boxed{
\mathbf v_{r,B}
=
\mathbf v_B
-
\mathbf v_{c,B}
}
$$

The existing production functions use the contracts:

```text
current_velocity_body(
    pose,
    current_velocity_world,
)

relative_velocity_6dof(
    pose,
    body_velocity,
    current_velocity_world,
)
```

Notebook 08 will reuse these APIs directly rather than duplicating the current-frame transformation.

The initial DVL measurement is:

```text
Quantity : water-relative translational velocity
Frame    : body frame
Units    : m/s
```

Bottom-track velocity, acoustic beam geometry, altitude lock, seabed interaction, and Doppler beam physics remain outside the current scope.

### Validation Criteria

This step passes only if:

1. zero current makes the DVL measurement equal body translational velocity;
2. moving exactly with the surrounding water produces zero relative translational velocity;
3. arbitrary orientation and world-frame current agree exactly with the inherited production current transformation;
4. rotational body rates do not appear in the translational DVL output;
5. the notebook-local DVL primitive agrees exactly with `relative_velocity_6dof`;
6. malformed and non-finite inputs are rejected explicitly.

In [14]:
def ideal_dvl_measurement(
    eta,
    nu,
    current_world,
):
    """
    Return ideal water-relative translational velocity measured by a DVL.

    Parameters
    ----------
    eta : array_like, shape (6,)
        Pose [x, y, z, roll, pitch, yaw].

    nu : array_like, shape (6,)
        Generalized body velocity [u, v, w, p, q, r].

    current_world : array_like, shape (3,)
        Ocean-current translational velocity expressed in the
        world frame.

    Returns
    -------
    numpy.ndarray, shape (3,)
        Water-relative translational velocity expressed in the
        body frame, in m/s.
    """

    eta = np.asarray(
        eta,
        dtype=float,
    )

    nu = np.asarray(
        nu,
        dtype=float,
    )

    current_world = np.asarray(
        current_world,
        dtype=float,
    )

    # -----------------------------------------------------------------
    # Explicit Notebook 08 input validation
    # -----------------------------------------------------------------

    if eta.shape != (6,):
        raise ValueError(
            "eta must have shape (6,) representing "
            "[x, y, z, roll, pitch, yaw]"
        )

    if nu.shape != (6,):
        raise ValueError(
            "nu must have shape (6,) representing "
            "[u, v, w, p, q, r]"
        )

    if current_world.shape != (3,):
        raise ValueError(
            "current_world must have shape (3,)"
        )

    if not np.all(np.isfinite(eta)):
        raise ValueError(
            "eta must contain only finite values"
        )

    if not np.all(np.isfinite(nu)):
        raise ValueError(
            "nu must contain only finite values"
        )

    if not np.all(np.isfinite(current_world)):
        raise ValueError(
            "current_world must contain only finite values"
        )

    # -----------------------------------------------------------------
    # Reuse Notebook 07 production relative-velocity implementation
    #
    # Contract:
    #
    # relative_velocity_6dof(
    #     pose,
    #     body_velocity,
    #     current_velocity_world,
    # )
    # -----------------------------------------------------------------

    relative_nu = relative_velocity_6dof(
        eta,
        nu,
        current_world,
    )

    relative_nu = np.asarray(
        relative_nu,
        dtype=float,
    )

    if relative_nu.shape != (6,):
        raise RuntimeError(
            "Production relative_velocity_6dof returned "
            f"unexpected shape {relative_nu.shape}"
        )

    return relative_nu[:3].copy()


# =====================================================================
# Validation 1: zero current
# =====================================================================

eta_level = np.zeros(6)

nu_vehicle = np.array([
    1.2,
    -0.4,
    0.25,
    0.10,
    -0.20,
    0.30,
])

current_zero = np.zeros(3)

dvl_zero_current = ideal_dvl_measurement(
    eta_level,
    nu_vehicle,
    current_zero,
)

expected_zero_current = np.array([
    1.2,
    -0.4,
    0.25,
])

np.testing.assert_allclose(
    dvl_zero_current,
    expected_zero_current,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 2: vehicle moving exactly with water
# =====================================================================

current_world_equal = np.array([
    0.8,
    -0.3,
    0.15,
])

# At zero orientation, world and body frames coincide.
# Translational vehicle velocity therefore exactly equals
# body-frame current velocity.

nu_equal_current = np.array([
    0.8,
    -0.3,
    0.15,
    0.50,
    -0.40,
    0.20,
])

dvl_equal_current = ideal_dvl_measurement(
    eta_level,
    nu_equal_current,
    current_world_equal,
)

np.testing.assert_allclose(
    dvl_equal_current,
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 3: arbitrary orientation/current
# =====================================================================

eta_arbitrary = np.array([
    2.0,
    -5.0,
    -18.0,
    0.35,
    -0.22,
    0.70,
])

nu_arbitrary = np.array([
    1.5,
    -0.6,
    0.4,
    0.8,
    -0.5,
    0.3,
])

current_world_arbitrary = np.array([
    0.45,
    -0.20,
    0.10,
])


# Production API:
#
# current_velocity_body(
#     pose,
#     current_velocity_world,
# )

current_body_expected = current_velocity_body(
    eta_arbitrary,
    current_world_arbitrary,
)

expected_arbitrary = (
    nu_arbitrary[:3]
    - current_body_expected
)

dvl_arbitrary = ideal_dvl_measurement(
    eta_arbitrary,
    nu_arbitrary,
    current_world_arbitrary,
)

np.testing.assert_allclose(
    dvl_arbitrary,
    expected_arbitrary,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 4: direct production relative-velocity comparison
# =====================================================================

relative_nu_production = relative_velocity_6dof(
    eta_arbitrary,
    nu_arbitrary,
    current_world_arbitrary,
)

np.testing.assert_allclose(
    dvl_arbitrary,
    relative_nu_production[:3],
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 5: rotational rates must not affect translational DVL
# =====================================================================

nu_rotation_a = np.array([
    0.9,
    -0.2,
    0.1,
    0.0,
    0.0,
    0.0,
])

nu_rotation_b = np.array([
    0.9,
    -0.2,
    0.1,
    10.0,
    -8.0,
    6.0,
])

dvl_rotation_a = ideal_dvl_measurement(
    eta_arbitrary,
    nu_rotation_a,
    current_world_arbitrary,
)

dvl_rotation_b = ideal_dvl_measurement(
    eta_arbitrary,
    nu_rotation_b,
    current_world_arbitrary,
)

np.testing.assert_allclose(
    dvl_rotation_a,
    dvl_rotation_b,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 6: moving with water under arbitrary orientation
# =====================================================================

current_body_arbitrary = current_velocity_body(
    eta_arbitrary,
    current_world_arbitrary,
)

nu_comoving = np.array([
    current_body_arbitrary[0],
    current_body_arbitrary[1],
    current_body_arbitrary[2],
    1.3,
    -0.7,
    0.4,
])

dvl_comoving = ideal_dvl_measurement(
    eta_arbitrary,
    nu_comoving,
    current_world_arbitrary,
)

np.testing.assert_allclose(
    dvl_comoving,
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 7: output must not alias inputs
# =====================================================================

eta_copy = eta_arbitrary.copy()
nu_copy = nu_arbitrary.copy()
current_copy = current_world_arbitrary.copy()

dvl_copy = ideal_dvl_measurement(
    eta_copy,
    nu_copy,
    current_copy,
)

eta_copy[:] = 99.0
nu_copy[:] = 99.0
current_copy[:] = 99.0

np.testing.assert_allclose(
    dvl_copy,
    expected_arbitrary,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 8: malformed inputs rejected
# =====================================================================

invalid_cases = [
    (
        np.zeros(5),
        np.zeros(6),
        np.zeros(3),
    ),
    (
        np.zeros(6),
        np.zeros(5),
        np.zeros(3),
    ),
    (
        np.zeros(6),
        np.zeros(6),
        np.zeros(2),
    ),
    (
        np.zeros((6, 1)),
        np.zeros(6),
        np.zeros(3),
    ),
    (
        np.zeros(6),
        np.zeros((6, 1)),
        np.zeros(3),
    ),
    (
        np.zeros(6),
        np.zeros(6),
        np.zeros((3, 1)),
    ),
    (
        np.array([
            0.0,
            0.0,
            0.0,
            np.nan,
            0.0,
            0.0,
        ]),
        np.zeros(6),
        np.zeros(3),
    ),
    (
        np.zeros(6),
        np.array([
            0.0,
            0.0,
            np.inf,
            0.0,
            0.0,
            0.0,
        ]),
        np.zeros(3),
    ),
    (
        np.zeros(6),
        np.zeros(6),
        np.array([
            0.0,
            np.nan,
            0.0,
        ]),
    ),
]

for bad_eta, bad_nu, bad_current in invalid_cases:
    try:
        ideal_dvl_measurement(
            bad_eta,
            bad_nu,
            bad_current,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Invalid DVL input was not rejected"
        )


# =====================================================================
# Report
# =====================================================================

print("STEP 11 VALIDATION PASSED")
print()

print("Zero-current DVL [m/s]:")
print(dvl_zero_current)

print()

print("Vehicle moving with water [m/s]:")
print(dvl_equal_current)

print()

print("Arbitrary world current [m/s]:")
print(current_world_arbitrary)

print()

print("Current transformed into body frame [m/s]:")
print(current_body_expected)

print()

print("Arbitrary-case DVL [m/s]:")
print(dvl_arbitrary)

print()

print("Expected arbitrary-case DVL [m/s]:")
print(expected_arbitrary)

print()

print(
    "Direct production relative-velocity agreement:",
    np.allclose(
        dvl_arbitrary,
        relative_nu_production[:3],
        rtol=0.0,
        atol=1e-12,
    ),
)

print(
    "Rotational-rate independence:",
    np.allclose(
        dvl_rotation_a,
        dvl_rotation_b,
        rtol=0.0,
        atol=1e-12,
    ),
)

print(
    "Arbitrary-orientation co-moving case [m/s]:"
)
print(dvl_comoving)

STEP 11 VALIDATION PASSED

Zero-current DVL [m/s]:
[ 1.2  -0.4   0.25]

Vehicle moving with water [m/s]:
[0. 0. 0.]

Arbitrary world current [m/s]:
[ 0.45 -0.2   0.1 ]

Current transformed into body frame [m/s]:
[ 0.23196828 -0.39866695  0.19938752]

Arbitrary-case DVL [m/s]:
[ 1.26803172 -0.20133305  0.20061248]

Expected arbitrary-case DVL [m/s]:
[ 1.26803172 -0.20133305  0.20061248]

Direct production relative-velocity agreement: True
Rotational-rate independence: True
Arbitrary-orientation co-moving case [m/s]:
[0. 0. 0.]


## Step 12 - DVL Bias and Noise

The ideal DVL water-relative velocity model is now validated against the inherited HydroLab-3D current transformation and relative-velocity implementation.

We now introduce lightweight measurement imperfections:

$$
\tilde{\mathbf v}_{r,B}
=
\mathbf v_{r,B}
+
\mathbf b_{dvl}
+
\mathbf n_{dvl}
$$

where:

- $\mathbf v_{r,B}$ is ideal body-frame water-relative velocity;
- $\mathbf b_{dvl}$ is deterministic DVL bias;
- $\mathbf n_{dvl}$ is Gaussian measurement noise;
- $\tilde{\mathbf v}_{r,B}$ is the reported DVL measurement.

The measurement remains:

```text
Quantity : water-relative translational velocity
Frame    : body frame
Units    : m/s
```

This step reuses:

1. the validated ideal DVL model;
2. the validated common bias/noise primitive.

No beam geometry, bottom lock, altitude limits, dropout, or acoustic propagation effects are introduced yet.

### Validation Criteria

This step passes only if:

1. zero bias and zero noise reproduce the ideal DVL exactly;
2. deterministic bias shifts each velocity component exactly;
3. identical seeds reproduce identical noisy DVL measurements;
4. different seeds produce different stochastic realizations;
5. the co-moving zero-relative-velocity case remains correct before imperfections are applied;
6. malformed bias, noise, and RNG inputs are rejected explicitly.

In [15]:
def dvl_measurement(
    eta,
    nu,
    current_world,
    *,
    bias=0.0,
    noise_std=0.0,
    rng=None,
):
    """
    Return imperfect body-frame water-relative DVL measurement.

    Parameters
    ----------
    eta : array_like, shape (6,)
        Pose [x, y, z, roll, pitch, yaw].

    nu : array_like, shape (6,)
        Generalized body velocity [u, v, w, p, q, r].

    current_world : array_like, shape (3,)
        Ocean current expressed in the world frame.

    bias : scalar or array_like
        Additive DVL bias in m/s.

    noise_std : scalar or array_like
        Gaussian DVL noise standard deviation in m/s.

    rng : numpy.random.Generator or None
        Explicit random generator when noise is nonzero.

    Returns
    -------
    numpy.ndarray, shape (3,)
        Body-frame water-relative velocity measurement in m/s.
    """

    ideal = ideal_dvl_measurement(
        eta,
        nu,
        current_world,
    )

    measured = measure_with_bias_noise(
        ideal,
        bias=bias,
        noise_std=noise_std,
        rng=rng,
    )

    if measured.shape != (3,):
        raise ValueError(
            "DVL bias and noise parameters must be "
            "scalar or broadcast-compatible with shape (3,)"
        )

    return measured


# =====================================================================
# Validation 1: ideal recovery
# =====================================================================

eta_test = np.array([
    4.0,
    -2.0,
    -15.0,
    0.25,
    -0.18,
    0.60,
])

nu_test = np.array([
    1.4,
    -0.5,
    0.3,
    0.2,
    -0.1,
    0.4,
])

current_world_test = np.array([
    0.35,
    -0.15,
    0.08,
])

dvl_ideal = ideal_dvl_measurement(
    eta_test,
    nu_test,
    current_world_test,
)

dvl_wrapper_ideal = dvl_measurement(
    eta_test,
    nu_test,
    current_world_test,
    bias=0.0,
    noise_std=0.0,
)

np.testing.assert_array_equal(
    dvl_wrapper_ideal,
    dvl_ideal,
)


# =====================================================================
# Validation 2: exact deterministic bias
# =====================================================================

dvl_bias = np.array([
    0.02,
    -0.01,
    0.03,
])

dvl_biased = dvl_measurement(
    eta_test,
    nu_test,
    current_world_test,
    bias=dvl_bias,
    noise_std=0.0,
)

np.testing.assert_allclose(
    dvl_biased,
    dvl_ideal + dvl_bias,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 3: same seed -> same noisy measurement
# =====================================================================

dvl_noise_std = np.array([
    0.01,
    0.015,
    0.02,
])

rng_a = np.random.default_rng(8120)
rng_b = np.random.default_rng(8120)

dvl_noisy_a = dvl_measurement(
    eta_test,
    nu_test,
    current_world_test,
    bias=dvl_bias,
    noise_std=dvl_noise_std,
    rng=rng_a,
)

dvl_noisy_b = dvl_measurement(
    eta_test,
    nu_test,
    current_world_test,
    bias=dvl_bias,
    noise_std=dvl_noise_std,
    rng=rng_b,
)

np.testing.assert_array_equal(
    dvl_noisy_a,
    dvl_noisy_b,
)


# =====================================================================
# Validation 4: different seed -> different realization
# =====================================================================

rng_c = np.random.default_rng(8121)

dvl_noisy_c = dvl_measurement(
    eta_test,
    nu_test,
    current_world_test,
    bias=dvl_bias,
    noise_std=dvl_noise_std,
    rng=rng_c,
)

if np.array_equal(
    dvl_noisy_a,
    dvl_noisy_c,
):
    raise AssertionError(
        "Different seeds unexpectedly produced "
        "identical DVL measurements"
    )


# =====================================================================
# Validation 5: co-moving case before imperfections
# =====================================================================

current_body_test = current_velocity_body(
    eta_test,
    current_world_test,
)

nu_comoving = np.array([
    current_body_test[0],
    current_body_test[1],
    current_body_test[2],
    0.7,
    -0.3,
    0.5,
])

dvl_comoving_ideal = dvl_measurement(
    eta_test,
    nu_comoving,
    current_world_test,
    bias=0.0,
    noise_std=0.0,
)

np.testing.assert_allclose(
    dvl_comoving_ideal,
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 6: malformed parameters rejected
# =====================================================================

try:
    dvl_measurement(
        eta_test,
        nu_test,
        current_world_test,
        bias=np.zeros(2),
        noise_std=0.0,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Invalid DVL bias shape was not rejected"
    )


try:
    dvl_measurement(
        eta_test,
        nu_test,
        current_world_test,
        bias=0.0,
        noise_std=np.array([
            0.01,
            -0.02,
            0.03,
        ]),
        rng=np.random.default_rng(1),
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Negative DVL noise standard deviation "
        "was not rejected"
    )


try:
    dvl_measurement(
        eta_test,
        nu_test,
        current_world_test,
        bias=0.0,
        noise_std=0.01,
        rng="not-a-generator",
    )
except TypeError:
    pass
else:
    raise AssertionError(
        "Invalid DVL RNG was not rejected"
    )


# =====================================================================
# Report
# =====================================================================

print("STEP 12 VALIDATION PASSED")
print()

print("Ideal DVL [m/s]:")
print(dvl_ideal)

print()

print("Biased DVL [m/s]:")
print(dvl_biased)

print()

print("Seed 8120 noisy DVL [m/s]:")
print(dvl_noisy_a)

print()

print("Seed 8121 noisy DVL [m/s]:")
print(dvl_noisy_c)

print()

print("Co-moving ideal DVL [m/s]:")
print(dvl_comoving_ideal)

STEP 12 VALIDATION PASSED

Ideal DVL [m/s]:
[ 1.18480519 -0.1989964   0.17963379]

Biased DVL [m/s]:
[ 1.20480519 -0.2089964   0.20963379]

Seed 8120 noisy DVL [m/s]:
[ 1.21923687 -0.19143375  0.21831334]

Seed 8121 noisy DVL [m/s]:
[ 1.20190269 -0.21190104  0.20724091]

Co-moving ideal DVL [m/s]:
[0. 0. 0.]


## Step 13 - Ideal Heading Sensor and Angle Wrapping

HydroLab-3D stores yaw as the final orientation component of:

$$
\eta =
\begin{bmatrix}
x &
y &
z &
\phi &
\theta &
\psi
\end{bmatrix}^{T}.
$$

The initial heading sensor will measure this yaw angle directly.

Because yaw may accumulate outside a principal interval, the reported heading must use an explicit wrap convention.

Notebook 08 defines heading as:

$$
\boxed{
\psi_{heading}
\in
(-\pi,\pi]
}
$$

using:

$$
\psi_{heading}
=
\operatorname{wrap}_{(-\pi,\pi]}(\psi).
$$

The measurement is:

```text
Quantity : heading / yaw
Reference: HydroLab world orientation convention
Units    : rad
Range    : (-pi, pi]
```

This is intentionally a lightweight heading abstraction.

It does not model:

- Earth's magnetic field;
- magnetic declination;
- magnetometer hard-iron or soft-iron effects;
- tilt-compensated compass electronics.

### Validation Criteria

This step passes only if:

1. headings already inside the principal interval remain unchanged;
2. angles greater than $\pi$ wrap correctly;
3. angles less than $-\pi$ wrap correctly;
4. boundary behavior at multiples of $\pi$ is deterministic;
5. position, roll, and pitch do not alter the ideal yaw measurement;
6. malformed and non-finite pose inputs are rejected explicitly.

In [16]:
def wrap_angle_pi(angle):
    """
    Wrap an angle to the interval (-pi, pi].

    Parameters
    ----------
    angle : float
        Angle in radians.

    Returns
    -------
    float
        Wrapped angle in (-pi, pi].
    """

    angle = float(angle)

    if not np.isfinite(angle):
        raise ValueError(
            "angle must be finite"
        )

    wrapped = (
        (angle + np.pi)
        % (2.0 * np.pi)
        - np.pi
    )

    # Convert the lower endpoint -pi to +pi so that the
    # final interval is (-pi, pi].
    if np.isclose(
        wrapped,
        -np.pi,
        rtol=0.0,
        atol=1e-15,
    ):
        wrapped = np.pi

    return float(wrapped)


def ideal_heading_measurement(eta):
    """
    Return ideal wrapped yaw heading.

    Parameters
    ----------
    eta : array_like, shape (6,)
        Pose [x, y, z, roll, pitch, yaw].

    Returns
    -------
    float
        Wrapped yaw angle in radians, in (-pi, pi].
    """

    eta = np.asarray(
        eta,
        dtype=float,
    )

    if eta.shape != (6,):
        raise ValueError(
            "eta must have shape (6,) representing "
            "[x, y, z, roll, pitch, yaw]"
        )

    if not np.all(np.isfinite(eta)):
        raise ValueError(
            "eta must contain only finite values"
        )

    return wrap_angle_pi(
        eta[5]
    )


# =====================================================================
# Validation 1: angle already inside interval
# =====================================================================

eta_nominal = np.array([
    10.0,
    -5.0,
    -20.0,
    0.4,
    -0.3,
    1.2,
])

heading_nominal = ideal_heading_measurement(
    eta_nominal
)

np.testing.assert_allclose(
    heading_nominal,
    1.2,
    rtol=0.0,
    atol=1e-15,
)


# =====================================================================
# Validation 2: angle greater than +pi
# =====================================================================

eta_positive_wrap = np.array([
    0.0,
    0.0,
    -5.0,
    0.0,
    0.0,
    3.0 * np.pi / 2.0,
])

heading_positive_wrap = ideal_heading_measurement(
    eta_positive_wrap
)

np.testing.assert_allclose(
    heading_positive_wrap,
    -np.pi / 2.0,
    rtol=0.0,
    atol=1e-15,
)


# =====================================================================
# Validation 3: angle less than -pi
# =====================================================================

eta_negative_wrap = np.array([
    0.0,
    0.0,
    -5.0,
    0.0,
    0.0,
    -3.0 * np.pi / 2.0,
])

heading_negative_wrap = ideal_heading_measurement(
    eta_negative_wrap
)

np.testing.assert_allclose(
    heading_negative_wrap,
    np.pi / 2.0,
    rtol=0.0,
    atol=1e-15,
)


# =====================================================================
# Validation 4: deterministic pi-boundary behavior
# =====================================================================

boundary_cases = {
    "pi": np.pi,
    "-pi": -np.pi,
    "3pi": 3.0 * np.pi,
    "-3pi": -3.0 * np.pi,
}

boundary_results = {}

for name, yaw in boundary_cases.items():
    eta_boundary = np.array([
        0.0,
        0.0,
        -1.0,
        0.0,
        0.0,
        yaw,
    ])

    boundary_results[name] = (
        ideal_heading_measurement(
            eta_boundary
        )
    )

    np.testing.assert_allclose(
        boundary_results[name],
        np.pi,
        rtol=0.0,
        atol=1e-15,
    )


# =====================================================================
# Validation 5: other pose components do not affect heading
# =====================================================================

eta_heading_a = np.array([
    0.0,
    0.0,
    -1.0,
    0.0,
    0.0,
    -0.75,
])

eta_heading_b = np.array([
    500.0,
    -900.0,
    -300.0,
    1.1,
    -0.8,
    -0.75,
])

heading_a = ideal_heading_measurement(
    eta_heading_a
)

heading_b = ideal_heading_measurement(
    eta_heading_b
)

np.testing.assert_allclose(
    heading_a,
    heading_b,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 6: malformed inputs rejected
# =====================================================================

invalid_eta_cases = [
    np.zeros(5),
    np.zeros(7),
    np.zeros((6, 1)),
    np.zeros((1, 6)),
    np.array([
        0.0,
        0.0,
        -1.0,
        0.0,
        0.0,
        np.nan,
    ]),
    np.array([
        0.0,
        0.0,
        -1.0,
        np.inf,
        0.0,
        0.0,
    ]),
]

for bad_eta in invalid_eta_cases:
    try:
        ideal_heading_measurement(
            bad_eta
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Invalid heading pose input was not rejected"
        )


invalid_angle_values = [
    np.nan,
    np.inf,
    -np.inf,
]

for bad_angle in invalid_angle_values:
    try:
        wrap_angle_pi(
            bad_angle
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            f"Invalid angle was not rejected: {bad_angle}"
        )


# =====================================================================
# Report
# =====================================================================

print("STEP 13 VALIDATION PASSED")
print()

print("Nominal heading [rad]:")
print(heading_nominal)

print()

print("3pi/2 wrapped heading [rad]:")
print(heading_positive_wrap)

print()

print("-3pi/2 wrapped heading [rad]:")
print(heading_negative_wrap)

print()

print("Boundary results:")
for name, value in boundary_results.items():
    print(f"  {name:4s} -> {value}")

print()

print(
    "Other-pose-component independence:",
    heading_a == heading_b,
)

STEP 13 VALIDATION PASSED

Nominal heading [rad]:
1.2000000000000002

3pi/2 wrapped heading [rad]:
-1.5707963267948966

-3pi/2 wrapped heading [rad]:
1.5707963267948966

Boundary results:
  pi   -> 3.141592653589793
  -pi  -> 3.141592653589793
  3pi  -> 3.141592653589793
  -3pi -> 3.141592653589793

Other-pose-component independence: True


## Step 14 - Heading Bias, Noise, and Final Angle Wrapping

The ideal heading sensor reports yaw using the convention:

$$
\psi_{heading}
\in
(-\pi,\pi].
$$

We now introduce deterministic bias and Gaussian measurement noise:

$$
\psi'
=
\psi_{heading}
+
b_h
+
n_h.
$$

Because bias or noise may move the value outside the principal interval, the final sensor output must be wrapped again:

$$
\boxed{
\tilde{\psi}_{heading}
=
\operatorname{wrap}_{(-\pi,\pi]}
\left(
\psi_{heading}
+
b_h
+
n_h
\right)
}
$$

where:

- $\psi_{heading}$ is the ideal wrapped yaw;
- $b_h$ is deterministic heading bias;
- $n_h$ is Gaussian heading noise;
- $\tilde{\psi}_{heading}$ is the final reported heading.

The measurement remains:

```text
Quantity : heading / yaw
Units    : rad
Range    : (-pi, pi]
```

This step does not model magnetic-field physics or compass calibration effects.

### Validation Criteria

This step passes only if:

1. zero bias and zero noise reproduce ideal heading;
2. deterministic bias is applied correctly away from wrap boundaries;
3. bias that crosses $+\pi$ wraps correctly;
4. identical seeds reproduce identical noisy headings;
5. different seeds produce different stochastic realizations;
6. every final measurement remains in $(-\pi,\pi]$;
7. invalid bias, noise, and RNG inputs are rejected explicitly.

In [17]:
def heading_measurement(
    eta,
    *,
    bias=0.0,
    noise_std=0.0,
    rng=None,
):
    """
    Return imperfect wrapped heading measurement.

    Parameters
    ----------
    eta : array_like, shape (6,)
        Pose [x, y, z, roll, pitch, yaw].

    bias : scalar
        Additive heading bias in radians.

    noise_std : scalar
        Gaussian heading-noise standard deviation in radians.

    rng : numpy.random.Generator or None
        Explicit random generator when noise is nonzero.

    Returns
    -------
    float
        Heading measurement in radians, wrapped to (-pi, pi].
    """

    ideal = ideal_heading_measurement(
        eta
    )

    measured = measure_with_bias_noise(
        ideal,
        bias=bias,
        noise_std=noise_std,
        rng=rng,
    )

    measured = np.asarray(
        measured,
        dtype=float,
    )

    if measured.shape != ():
        raise ValueError(
            "Heading bias and noise_std must be scalar"
        )

    return wrap_angle_pi(
        float(measured)
    )


# =====================================================================
# Validation 1: ideal recovery
# =====================================================================

eta_test = np.array([
    2.0,
    -4.0,
    -12.0,
    0.2,
    -0.1,
    0.75,
])

heading_ideal = ideal_heading_measurement(
    eta_test
)

heading_wrapper_ideal = heading_measurement(
    eta_test,
    bias=0.0,
    noise_std=0.0,
)

np.testing.assert_allclose(
    heading_wrapper_ideal,
    heading_ideal,
    rtol=0.0,
    atol=1e-15,
)


# =====================================================================
# Validation 2: exact deterministic bias away from boundary
# =====================================================================

heading_bias = 0.20

heading_biased = heading_measurement(
    eta_test,
    bias=heading_bias,
    noise_std=0.0,
)

expected_biased = wrap_angle_pi(
    heading_ideal + heading_bias
)

np.testing.assert_allclose(
    heading_biased,
    expected_biased,
    rtol=0.0,
    atol=1e-15,
)


# =====================================================================
# Validation 3: bias crossing +pi must wrap
# =====================================================================

eta_near_pi = np.array([
    0.0,
    0.0,
    -1.0,
    0.0,
    0.0,
    np.pi - 0.05,
])

crossing_bias = 0.20

heading_crossed = heading_measurement(
    eta_near_pi,
    bias=crossing_bias,
    noise_std=0.0,
)

expected_crossed = wrap_angle_pi(
    (np.pi - 0.05)
    + crossing_bias
)

np.testing.assert_allclose(
    heading_crossed,
    expected_crossed,
    rtol=0.0,
    atol=1e-15,
)

if not (
    -np.pi
    < heading_crossed
    <= np.pi
):
    raise AssertionError(
        "Wrapped biased heading left (-pi, pi]"
    )


# =====================================================================
# Validation 4: same seed -> same noisy heading
# =====================================================================

rng_a = np.random.default_rng(8140)
rng_b = np.random.default_rng(8140)

heading_noisy_a = heading_measurement(
    eta_test,
    bias=0.10,
    noise_std=0.03,
    rng=rng_a,
)

heading_noisy_b = heading_measurement(
    eta_test,
    bias=0.10,
    noise_std=0.03,
    rng=rng_b,
)

np.testing.assert_allclose(
    heading_noisy_a,
    heading_noisy_b,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 5: different seed -> different realization
# =====================================================================

rng_c = np.random.default_rng(8141)

heading_noisy_c = heading_measurement(
    eta_test,
    bias=0.10,
    noise_std=0.03,
    rng=rng_c,
)

if heading_noisy_a == heading_noisy_c:
    raise AssertionError(
        "Different seeds unexpectedly produced "
        "identical heading measurements"
    )


# =====================================================================
# Validation 6: all outputs remain inside (-pi, pi]
# =====================================================================

rng_range = np.random.default_rng(8142)

range_measurements = []

eta_wrap_stress = np.array([
    0.0,
    0.0,
    -2.0,
    0.0,
    0.0,
    np.pi - 0.01,
])

for _ in range(10_000):
    value = heading_measurement(
        eta_wrap_stress,
        bias=0.15,
        noise_std=0.25,
        rng=rng_range,
    )

    range_measurements.append(
        value
    )

range_measurements = np.asarray(
    range_measurements,
    dtype=float,
)

if not np.all(
    (range_measurements > -np.pi)
    & (range_measurements <= np.pi)
):
    raise AssertionError(
        "One or more heading measurements left "
        "the interval (-pi, pi]"
    )


# =====================================================================
# Validation 7: invalid scalar parameters rejected
# =====================================================================

try:
    heading_measurement(
        eta_test,
        bias=np.array([0.1, 0.2]),
        noise_std=0.0,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Non-scalar heading bias was not rejected"
    )


try:
    heading_measurement(
        eta_test,
        bias=0.0,
        noise_std=np.array([0.01, 0.02]),
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Non-scalar heading noise_std was not rejected"
    )


try:
    heading_measurement(
        eta_test,
        bias=0.0,
        noise_std=-0.01,
        rng=np.random.default_rng(1),
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Negative heading noise standard deviation "
        "was not rejected"
    )


try:
    heading_measurement(
        eta_test,
        bias=0.0,
        noise_std=0.01,
        rng="not-a-generator",
    )
except TypeError:
    pass
else:
    raise AssertionError(
        "Invalid heading RNG was not rejected"
    )


# =====================================================================
# Report
# =====================================================================

print("STEP 14 VALIDATION PASSED")
print()

print("Ideal heading [rad]:")
print(heading_ideal)

print()

print("Biased heading [rad]:")
print(heading_biased)

print()

print("Boundary-crossing biased heading [rad]:")
print(heading_crossed)

print()

print("Seed 8140 noisy heading [rad]:")
print(heading_noisy_a)

print()

print("Seed 8141 noisy heading [rad]:")
print(heading_noisy_c)

print()

print(
    "Stress-test measurement range [rad]:"
)
print(
    np.min(range_measurements),
    np.max(range_measurements),
)

STEP 14 VALIDATION PASSED

Ideal heading [rad]:
0.75

Biased heading [rad]:
0.9500000000000002

Boundary-crossing biased heading [rad]:
-2.9915926535897928

Seed 8140 noisy heading [rad]:
0.8268491834374192

Seed 8141 noisy heading [rad]:
0.8521928618092787

Stress-test measurement range [rad]:
-3.141541026902974 3.14153157157713


## Step 15 - Inspect 6-DOF Production Integration API

The individual sensor primitives are now validated.

The next stage will connect them to actual Notebook 07 ground-truth dynamics rather than manually constructed states.

Before doing that, we inspect the exact production API contracts for:

```text
current_generalized_acceleration
six_dof_state_derivative
integrate_six_dof_euler
simulate_six_dof
```

This prevents Notebook 08 from guessing argument order or duplicating dynamics logic.

This step is strictly read-only.

### Validation Criteria

This step passes when the exact function signatures and source modules are confirmed for all four inherited 6-DOF production functions.

In [18]:
# ---------------------------------------------------------------------
# Step 15 - Inspect actual Notebook 07 production API contracts
# ---------------------------------------------------------------------

integration_functions = {
    "current_generalized_acceleration":
        current_generalized_acceleration,

    "six_dof_state_derivative":
        six_dof_state_derivative,

    "integrate_six_dof_euler":
        integrate_six_dof_euler,

    "simulate_six_dof":
        simulate_six_dof,
}


print("=" * 80)
print("STEP 15 - 6-DOF PRODUCTION API INSPECTION")
print("=" * 80)


for name, function in integration_functions.items():

    print()
    print("-" * 80)
    print(name)
    print("-" * 80)

    signature = inspect.signature(
        function
    )

    source_file = Path(
        inspect.getsourcefile(function)
        or inspect.getfile(function)
    ).resolve()

    print("Signature:")
    print(
        f"  {name}{signature}"
    )

    print()

    print("Source:")
    print(
        " ",
        source_file.relative_to(repo_root),
    )

    print()

    docstring = inspect.getdoc(
        function
    )

    print("Docstring:")

    if docstring:
        print(docstring)
    else:
        print("  <no docstring>")


print()
print("=" * 80)
print("STEP 15 INSPECTION COMPLETE")
print("=" * 80)

print(
    "Production dynamics APIs inspected read-only; "
    "no source files were modified."
)

STEP 15 - 6-DOF PRODUCTION API INSPECTION

--------------------------------------------------------------------------------
current_generalized_acceleration
--------------------------------------------------------------------------------
Signature:
  current_generalized_acceleration(pose, body_velocity, current_velocity_world)

Source:
  src/hydrolab3d/vehicles/uuv.py

Docstring:
Body derivative of a uniform world-fixed translational current.

--------------------------------------------------------------------------------
six_dof_state_derivative
--------------------------------------------------------------------------------
Signature:
  six_dof_state_derivative(pose, body_velocity, *, mass, inertia, added_inertia, linear_damping, quadratic_damping, fluid_density, displaced_volume, center_of_gravity, center_of_buoyancy, current_velocity_world, actuator_wrench, environmental_wrench=None, gravity=9.81)

Source:
  src/hydrolab3d/vehicles/uuv.py

Docstring:
Compute eta_dot and nu_dot for

## Step 16 - Single-State Sensor Integration from Production 6-DOF Dynamics

The individual sensor models are now validated independently.

We next connect them to the actual HydroLab-3D production six-degree-of-freedom dynamics.

The production function:

```text
six_dof_state_derivative(...)
```

returns three values:

```text
eta_dot
nu_dot
diagnostics
```

where:

$$
\dot{\eta}
$$

is the pose derivative,

$$
\dot{\nu}
$$

is the generalized body acceleration, and `diagnostics` contains the individually resolved dynamic terms used by Notebook 07.

The sensor data flow is:

```text
Production 6-DOF State
        │
        ├── eta
        ├── nu
        ├── nu_dot
        └── diagnostics
             ↓
        Sensor Models
             ↓
        Measurements
```

Specifically:

```text
Gyroscope
← nu[3:6]

Accelerometer
← eta, nu, nu_dot

Depth
← eta[2]

DVL
← eta, nu, current_velocity_world

Heading
← eta[5]
```

The Notebook 07 production parameter contracts remain unchanged:

```text
inertia
→ shape (3,)

added_inertia
→ shape (6,)

linear_damping
→ shape (6,)

quadratic_damping
→ shape (6,)

center_of_gravity
→ shape (3,)

center_of_buoyancy
→ shape (3,)
```

This step uses ideal sensors only.

Noise and bias remain disabled so each sensor can be compared directly against deterministic ground truth.

### Validation Criteria

This step passes only if:

1. `six_dof_state_derivative()` returns finite `eta_dot`, `nu_dot`, and diagnostics;
2. diagnostics contain the production relative velocity `nu_r`;
3. all ideal sensor outputs are finite;
4. gyroscope output equals body angular velocity;
5. depth matches world-$z$ ground truth;
6. DVL matches both production `relative_velocity_6dof()` and diagnostic `nu_r`;
7. heading matches wrapped production yaw;
8. accelerometer specific force is independently reconstructed from production `nu_dot`;
9. sensor evaluation does not modify the underlying physical state.

In [22]:
# ---------------------------------------------------------------------
# Step 16 - Single-state sensor integration with production dynamics
# ---------------------------------------------------------------------

eta_truth = np.array([
    4.0,
    -3.0,
    -12.0,
    0.18,
    -0.12,
    0.55,
])

nu_truth = np.array([
    1.1,
    -0.35,
    0.22,
    0.06,
    -0.04,
    0.09,
])


# =====================================================================
# Production dynamics parameters
# =====================================================================

mass = 42.0

inertia = np.array([
    2.8,
    3.1,
    3.6,
])

added_inertia = np.array([
    8.0,
    10.0,
    12.0,
    0.8,
    0.9,
    1.0,
])

linear_damping = np.array([
    12.0,
    16.0,
    20.0,
    1.5,
    1.8,
    2.0,
])

quadratic_damping = np.array([
    18.0,
    22.0,
    28.0,
    2.2,
    2.5,
    2.8,
])

fluid_density = 1025.0

displaced_volume = (
    mass / fluid_density
)

center_of_gravity = np.array([
    0.0,
    0.0,
    0.0,
])

center_of_buoyancy = np.array([
    0.0,
    0.0,
    0.04,
])

current_world = np.array([
    0.25,
    -0.10,
    0.05,
])

actuator_wrench = np.array([
    8.0,
    -3.0,
    2.0,
    0.4,
    -0.2,
    0.3,
])

environmental_wrench = np.array([
    0.5,
    0.0,
    -0.3,
    0.0,
    0.05,
    0.0,
])


# =====================================================================
# Preserve physical state for mutation checks
# =====================================================================

eta_before = eta_truth.copy()
nu_before = nu_truth.copy()


# =====================================================================
# Production six-DOF derivative
#
# Notebook 07 contract:
#
# eta_dot, nu_dot, diagnostics
# =====================================================================

(
    eta_dot_truth,
    nu_dot_truth,
    dynamics_diagnostics,
) = six_dof_state_derivative(
    eta_truth,
    nu_truth,
    mass=mass,
    inertia=inertia,
    added_inertia=added_inertia,
    linear_damping=linear_damping,
    quadratic_damping=quadratic_damping,
    fluid_density=fluid_density,
    displaced_volume=displaced_volume,
    center_of_gravity=center_of_gravity,
    center_of_buoyancy=center_of_buoyancy,
    current_velocity_world=current_world,
    actuator_wrench=actuator_wrench,
    environmental_wrench=environmental_wrench,
    gravity=9.81,
)


# =====================================================================
# Validate production outputs
# =====================================================================

eta_dot_truth = np.asarray(
    eta_dot_truth,
    dtype=float,
)

nu_dot_truth = np.asarray(
    nu_dot_truth,
    dtype=float,
)


if eta_dot_truth.shape != (6,):
    raise AssertionError(
        f"eta_dot has unexpected shape {eta_dot_truth.shape}"
    )

if nu_dot_truth.shape != (6,):
    raise AssertionError(
        f"nu_dot has unexpected shape {nu_dot_truth.shape}"
    )

if not np.all(np.isfinite(eta_dot_truth)):
    raise AssertionError(
        "eta_dot contains non-finite values"
    )

if not np.all(np.isfinite(nu_dot_truth)):
    raise AssertionError(
        "nu_dot contains non-finite values"
    )

if not isinstance(
    dynamics_diagnostics,
    dict,
):
    raise AssertionError(
        "Dynamics diagnostics must be a dictionary"
    )

if "nu_r" not in dynamics_diagnostics:
    raise AssertionError(
        "Dynamics diagnostics do not contain nu_r"
    )

diagnostic_nu_r = np.asarray(
    dynamics_diagnostics["nu_r"],
    dtype=float,
)

if diagnostic_nu_r.shape != (6,):
    raise AssertionError(
        "Diagnostic nu_r has unexpected shape "
        f"{diagnostic_nu_r.shape}"
    )

if not np.all(np.isfinite(diagnostic_nu_r)):
    raise AssertionError(
        "Diagnostic nu_r contains non-finite values"
    )


# =====================================================================
# Generate ideal sensor measurements
# =====================================================================

gyro_measurement_ideal = ideal_gyroscope_measurement(
    nu_truth
)

accel_measurement_ideal = ideal_accelerometer_measurement(
    eta_truth,
    nu_truth,
    nu_dot_truth,
)

depth_measurement_ideal = ideal_depth_measurement(
    eta_truth
)

dvl_measurement_ideal = ideal_dvl_measurement(
    eta_truth,
    nu_truth,
    current_world,
)

heading_measurement_ideal = ideal_heading_measurement(
    eta_truth
)


# =====================================================================
# Validation 1: gyroscope
# =====================================================================

np.testing.assert_allclose(
    gyro_measurement_ideal,
    nu_truth[3:6],
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 2: depth
# =====================================================================

expected_depth = -eta_truth[2]

np.testing.assert_allclose(
    depth_measurement_ideal,
    expected_depth,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 3: DVL against production function
# =====================================================================

relative_nu_truth = relative_velocity_6dof(
    eta_truth,
    nu_truth,
    current_world,
)

relative_nu_truth = np.asarray(
    relative_nu_truth,
    dtype=float,
)

np.testing.assert_allclose(
    dvl_measurement_ideal,
    relative_nu_truth[:3],
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 4: DVL against Notebook 07 diagnostics
# =====================================================================

np.testing.assert_allclose(
    dvl_measurement_ideal,
    diagnostic_nu_r[:3],
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    relative_nu_truth,
    diagnostic_nu_r,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 5: heading
# =====================================================================

expected_heading = wrap_angle_pi(
    eta_truth[5]
)

np.testing.assert_allclose(
    heading_measurement_ideal,
    expected_heading,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 6: independent accelerometer reconstruction
# =====================================================================

R_BW_truth = rotation_body_to_world(
    eta_truth[3],
    eta_truth[4],
    eta_truth[5],
)

R_WB_truth = R_BW_truth.T

gravity_world = np.array([
    0.0,
    0.0,
    -9.81,
])

expected_accel = (
    nu_dot_truth[:3]
    + np.cross(
        nu_truth[3:6],
        nu_truth[:3],
    )
    - R_WB_truth @ gravity_world
)

np.testing.assert_allclose(
    accel_measurement_ideal,
    expected_accel,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 7: all outputs finite
# =====================================================================

vector_outputs = {
    "gyro": gyro_measurement_ideal,
    "accelerometer": accel_measurement_ideal,
    "DVL": dvl_measurement_ideal,
}

for name, output in vector_outputs.items():

    output = np.asarray(
        output,
        dtype=float,
    )

    if output.shape != (3,):
        raise AssertionError(
            f"{name} has unexpected shape {output.shape}"
        )

    if not np.all(np.isfinite(output)):
        raise AssertionError(
            f"{name} contains non-finite values"
        )


for name, output in {
    "depth": depth_measurement_ideal,
    "heading": heading_measurement_ideal,
}.items():

    if not np.isfinite(output):
        raise AssertionError(
            f"{name} is non-finite"
        )


# =====================================================================
# Validation 8: sensors must not mutate physical truth
# =====================================================================

np.testing.assert_array_equal(
    eta_truth,
    eta_before,
)

np.testing.assert_array_equal(
    nu_truth,
    nu_before,
)


# =====================================================================
# Validation 9: measurement copies remain isolated
# =====================================================================

sensor_record = {
    "gyro": gyro_measurement_ideal.copy(),
    "accelerometer": accel_measurement_ideal.copy(),
    "depth": float(depth_measurement_ideal),
    "dvl": dvl_measurement_ideal.copy(),
    "heading": float(heading_measurement_ideal),
}

sensor_record["gyro"][0] += 1000.0
sensor_record["accelerometer"][0] += 1000.0
sensor_record["dvl"][0] += 1000.0

np.testing.assert_array_equal(
    eta_truth,
    eta_before,
)

np.testing.assert_array_equal(
    nu_truth,
    nu_before,
)


# =====================================================================
# Report
# =====================================================================

print("STEP 16 VALIDATION PASSED")
print()

print("Ground-truth pose eta:")
print(eta_truth)

print()

print("Ground-truth body velocity nu:")
print(nu_truth)

print()

print("Production eta_dot:")
print(eta_dot_truth)

print()

print("Production nu_dot:")
print(nu_dot_truth)

print()

print("Diagnostic water-relative nu_r:")
print(diagnostic_nu_r)

print()

print("Ideal gyroscope [rad/s]:")
print(gyro_measurement_ideal)

print()

print("Ideal accelerometer [m/s^2]:")
print(accel_measurement_ideal)

print()

print("Ideal depth [m]:")
print(depth_measurement_ideal)

print()

print("Ideal DVL [m/s]:")
print(dvl_measurement_ideal)

print()

print("Ideal heading [rad]:")
print(heading_measurement_ideal)

print()

print(
    "DVL agrees with diagnostic nu_r:",
    np.allclose(
        dvl_measurement_ideal,
        diagnostic_nu_r[:3],
        rtol=0.0,
        atol=1e-12,
    ),
)

print(
    "Ground-truth state unchanged:",
    np.array_equal(
        eta_truth,
        eta_before,
    )
    and np.array_equal(
        nu_truth,
        nu_before,
    ),
)

STEP 16 VALIDATION PASSED

Ground-truth pose eta:
[  4.    -3.   -12.     0.18  -0.12   0.55]

Ground-truth body velocity nu:
[ 1.1  -0.35  0.22  0.06 -0.04  0.09]

Production eta_dot:
[ 1.11590984  0.23405781  0.28436276  0.05018668 -0.05546641  0.08197426]

Production nu_dot:
[-0.39122014 -0.08725554 -0.05705432 -0.71716001  0.61759007  0.07945639]

Diagnostic water-relative nu_r:
[ 0.9343088  -0.14300378  0.15145081  0.06       -0.04        0.09      ]

Ideal gyroscope [rad/s]:
[ 0.06 -0.04  0.09]

Ideal accelerometer [m/s^2]:
[0.80585661 1.74219452 9.54804481]

Ideal depth [m]:
12.0

Ideal DVL [m/s]:
[ 0.9343088  -0.14300378  0.15145081]

Ideal heading [rad]:
0.5499999999999998

DVL agrees with diagnostic nu_r: True
Ground-truth state unchanged: True


## Step 17 - Integrated Imperfect Sensor Snapshot and Reproducibility

The ideal sensor suite has now been validated against a real HydroLab-3D production six-DOF state.

We next evaluate all imperfect sensors together.

Each stochastic sensor receives its own explicit NumPy random generator:

```text
Ground Truth
    │
    ├── Gyroscope      ← dedicated RNG
    ├── Accelerometer  ← dedicated RNG
    ├── Depth          ← dedicated RNG
    ├── DVL            ← dedicated RNG
    └── Heading        ← dedicated RNG
```

Separate random streams are important.

If every sensor shared one generator, changing the sampling frequency or implementation order of one sensor could change the noise realization produced by another sensor.

Independent generators preserve modularity and reproducibility.

The physical dynamics remain deterministic.

Only measurements are stochastic.

### Validation Criteria

This step passes only if:

1. two sensor suites initialized with identical seeds produce identical measurement snapshots;
2. changing all seeds changes the stochastic measurements;
3. every measurement remains finite;
4. heading remains inside $(-\pi,\pi]$;
5. noisy measurements differ from their ideal values where expected;
6. repeated sensor evaluation does not modify the physical ground-truth state;
7. recreating a sensor RNG with the same seed reproduces that sensor's measurement independently of the other sensors.

In [23]:
def make_sensor_rngs(seed_offset=0):
    """
    Construct independent RNG streams for each stochastic sensor.
    """

    seed_offset = int(seed_offset)

    return {
        "gyro": np.random.default_rng(
            17001 + seed_offset
        ),
        "accelerometer": np.random.default_rng(
            17002 + seed_offset
        ),
        "depth": np.random.default_rng(
            17003 + seed_offset
        ),
        "dvl": np.random.default_rng(
            17004 + seed_offset
        ),
        "heading": np.random.default_rng(
            17005 + seed_offset
        ),
    }


def sensor_snapshot(
    eta,
    nu,
    nu_dot,
    current_world,
    rngs,
):
    """
    Generate one complete imperfect sensor snapshot from
    a ground-truth HydroLab-3D state.
    """

    required_rngs = {
        "gyro",
        "accelerometer",
        "depth",
        "dvl",
        "heading",
    }

    if not isinstance(rngs, dict):
        raise TypeError(
            "rngs must be a dictionary of sensor RNGs"
        )

    if set(rngs) != required_rngs:
        raise ValueError(
            "rngs must contain exactly: "
            "gyro, accelerometer, depth, dvl, heading"
        )

    snapshot = {
        "gyro": gyroscope_measurement(
            nu,
            bias=np.array([
                0.002,
                -0.001,
                0.0015,
            ]),
            noise_std=np.array([
                0.001,
                0.001,
                0.001,
            ]),
            rng=rngs["gyro"],
        ),

        "accelerometer": accelerometer_measurement(
            eta,
            nu,
            nu_dot,
            bias=np.array([
                0.03,
                -0.02,
                0.04,
            ]),
            noise_std=np.array([
                0.015,
                0.015,
                0.020,
            ]),
            rng=rngs["accelerometer"],
        ),

        "depth": depth_measurement(
            eta,
            bias=0.08,
            noise_std=0.03,
            rng=rngs["depth"],
        ),

        "dvl": dvl_measurement(
            eta,
            nu,
            current_world,
            bias=np.array([
                0.01,
                -0.005,
                0.008,
            ]),
            noise_std=np.array([
                0.008,
                0.008,
                0.010,
            ]),
            rng=rngs["dvl"],
        ),

        "heading": heading_measurement(
            eta,
            bias=0.015,
            noise_std=0.008,
            rng=rngs["heading"],
        ),
    }

    return snapshot


# =====================================================================
# Preserve physical ground truth
# =====================================================================

eta_before = eta_truth.copy()
nu_before = nu_truth.copy()
nu_dot_before = nu_dot_truth.copy()


# =====================================================================
# Snapshot A and B: identical RNG seeds
# =====================================================================

rngs_a = make_sensor_rngs(
    seed_offset=0
)

rngs_b = make_sensor_rngs(
    seed_offset=0
)

snapshot_a = sensor_snapshot(
    eta_truth,
    nu_truth,
    nu_dot_truth,
    current_world,
    rngs_a,
)

snapshot_b = sensor_snapshot(
    eta_truth,
    nu_truth,
    nu_dot_truth,
    current_world,
    rngs_b,
)


# =====================================================================
# Validation 1: identical seeds -> identical complete snapshot
# =====================================================================

np.testing.assert_array_equal(
    snapshot_a["gyro"],
    snapshot_b["gyro"],
)

np.testing.assert_array_equal(
    snapshot_a["accelerometer"],
    snapshot_b["accelerometer"],
)

np.testing.assert_allclose(
    snapshot_a["depth"],
    snapshot_b["depth"],
    rtol=0.0,
    atol=0.0,
)

np.testing.assert_array_equal(
    snapshot_a["dvl"],
    snapshot_b["dvl"],
)

np.testing.assert_allclose(
    snapshot_a["heading"],
    snapshot_b["heading"],
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Snapshot C: different independent seeds
# =====================================================================

rngs_c = make_sensor_rngs(
    seed_offset=100
)

snapshot_c = sensor_snapshot(
    eta_truth,
    nu_truth,
    nu_dot_truth,
    current_world,
    rngs_c,
)


# =====================================================================
# Validation 2: different seeds -> changed stochastic realizations
# =====================================================================

if np.array_equal(
    snapshot_a["gyro"],
    snapshot_c["gyro"],
):
    raise AssertionError(
        "Different gyro seeds produced identical output"
    )

if np.array_equal(
    snapshot_a["accelerometer"],
    snapshot_c["accelerometer"],
):
    raise AssertionError(
        "Different accelerometer seeds produced identical output"
    )

if snapshot_a["depth"] == snapshot_c["depth"]:
    raise AssertionError(
        "Different depth seeds produced identical output"
    )

if np.array_equal(
    snapshot_a["dvl"],
    snapshot_c["dvl"],
):
    raise AssertionError(
        "Different DVL seeds produced identical output"
    )

if snapshot_a["heading"] == snapshot_c["heading"]:
    raise AssertionError(
        "Different heading seeds produced identical output"
    )


# =====================================================================
# Validation 3: all outputs finite
# =====================================================================

for name in [
    "gyro",
    "accelerometer",
    "dvl",
]:
    value = np.asarray(
        snapshot_a[name],
        dtype=float,
    )

    if value.shape != (3,):
        raise AssertionError(
            f"{name} has unexpected shape {value.shape}"
        )

    if not np.all(np.isfinite(value)):
        raise AssertionError(
            f"{name} contains non-finite values"
        )


for name in [
    "depth",
    "heading",
]:
    if not np.isfinite(
        snapshot_a[name]
    ):
        raise AssertionError(
            f"{name} is non-finite"
        )


# =====================================================================
# Validation 4: heading remains wrapped
# =====================================================================

if not (
    -np.pi
    < snapshot_a["heading"]
    <= np.pi
):
    raise AssertionError(
        "Integrated heading measurement left (-pi, pi]"
    )


# =====================================================================
# Validation 5: imperfect measurements differ from ideal
# =====================================================================

if np.array_equal(
    snapshot_a["gyro"],
    gyro_measurement_ideal,
):
    raise AssertionError(
        "Imperfect gyro unexpectedly equals ideal gyro"
    )

if np.array_equal(
    snapshot_a["accelerometer"],
    accel_measurement_ideal,
):
    raise AssertionError(
        "Imperfect accelerometer unexpectedly equals ideal"
    )

if snapshot_a["depth"] == depth_measurement_ideal:
    raise AssertionError(
        "Imperfect depth unexpectedly equals ideal depth"
    )

if np.array_equal(
    snapshot_a["dvl"],
    dvl_measurement_ideal,
):
    raise AssertionError(
        "Imperfect DVL unexpectedly equals ideal DVL"
    )

if snapshot_a["heading"] == heading_measurement_ideal:
    raise AssertionError(
        "Imperfect heading unexpectedly equals ideal heading"
    )


# =====================================================================
# Validation 6: sensor evaluation must not mutate physics
# =====================================================================

np.testing.assert_array_equal(
    eta_truth,
    eta_before,
)

np.testing.assert_array_equal(
    nu_truth,
    nu_before,
)

np.testing.assert_array_equal(
    nu_dot_truth,
    nu_dot_before,
)


# =====================================================================
# Validation 7: RNG independence
#
# Reconstruct ONLY the gyro generator with its original seed.
# Its result must reproduce snapshot A without recreating or
# advancing any other sensor RNG.
# =====================================================================

gyro_replay_rng = np.random.default_rng(
    17001
)

gyro_replay = gyroscope_measurement(
    nu_truth,
    bias=np.array([
        0.002,
        -0.001,
        0.0015,
    ]),
    noise_std=np.array([
        0.001,
        0.001,
        0.001,
    ]),
    rng=gyro_replay_rng,
)

np.testing.assert_array_equal(
    gyro_replay,
    snapshot_a["gyro"],
)


# =====================================================================
# Report
# =====================================================================

print("STEP 17 VALIDATION PASSED")
print()

print("Ideal sensor values:")
print(
    "  gyro         ",
    gyro_measurement_ideal,
)
print(
    "  accelerometer",
    accel_measurement_ideal,
)
print(
    "  depth        ",
    depth_measurement_ideal,
)
print(
    "  DVL          ",
    dvl_measurement_ideal,
)
print(
    "  heading      ",
    heading_measurement_ideal,
)

print()

print("Imperfect snapshot A:")
print(
    "  gyro         ",
    snapshot_a["gyro"],
)
print(
    "  accelerometer",
    snapshot_a["accelerometer"],
)
print(
    "  depth        ",
    snapshot_a["depth"],
)
print(
    "  DVL          ",
    snapshot_a["dvl"],
)
print(
    "  heading      ",
    snapshot_a["heading"],
)

print()

print("Different-seed snapshot C:")
print(
    "  gyro         ",
    snapshot_c["gyro"],
)
print(
    "  accelerometer",
    snapshot_c["accelerometer"],
)
print(
    "  depth        ",
    snapshot_c["depth"],
)
print(
    "  DVL          ",
    snapshot_c["dvl"],
)
print(
    "  heading      ",
    snapshot_c["heading"],
)

print()

print(
    "Complete same-seed snapshot reproducibility: True"
)

print(
    "Independent gyro RNG replay: True"
)

print(
    "Ground-truth dynamics unchanged: True"
)

STEP 17 VALIDATION PASSED

Ideal sensor values:
  gyro          [ 0.06 -0.04  0.09]
  accelerometer [0.80585661 1.74219452 9.54804481]
  depth         12.0
  DVL           [ 0.9343088  -0.14300378  0.15145081]
  heading       0.5499999999999998

Imperfect snapshot A:
  gyro          [ 0.06239005 -0.04013857  0.091775  ]
  accelerometer [0.85663258 1.73328926 9.57881567]
  depth         12.051305936823493
  DVL           [ 0.94886165 -0.15898911  0.15604438]
  heading       0.5604979704533242

Different-seed snapshot C:
  gyro          [ 0.06197689 -0.04098683  0.09250629]
  accelerometer [0.83499652 1.71317431 9.59137891]
  depth         12.085830815568647
  DVL           [ 0.93904286 -0.14877718  0.1760155 ]
  heading       0.5568225703407741

Complete same-seed snapshot reproducibility: True
Independent gyro RNG replay: True
Ground-truth dynamics unchanged: True


## Step 18 - Inspect Production Coding and Export Style

The notebook-local sensor models are now individually and jointly validated.

Before promoting them into:

```text
src/hydrolab3d/sensors/
```

we inspect representative existing production modules and package exports.

The purpose is to preserve HydroLab-3D's established style for:

- function naming;
- validation;
- docstrings;
- imports;
- public exports;
- module organization.

This step remains strictly read-only.

No sensor production files are modified yet.

### Validation Criteria

This step passes when the relevant neighboring production modules and package `__init__.py` files have been inspected closely enough to define the promotion style without inventing a new repository convention.

In [24]:
# ---------------------------------------------------------------------
# Step 18 - Inspect production coding and export style
# ---------------------------------------------------------------------

style_files = [
    repo_root
    / "src"
    / "hydrolab3d"
    / "__init__.py",

    repo_root
    / "src"
    / "hydrolab3d"
    / "dynamics"
    / "__init__.py",

    repo_root
    / "src"
    / "hydrolab3d"
    / "world"
    / "__init__.py",

    repo_root
    / "src"
    / "hydrolab3d"
    / "vehicles"
    / "__init__.py",

    repo_root
    / "src"
    / "hydrolab3d"
    / "utils"
    / "transforms.py",

    repo_root
    / "src"
    / "hydrolab3d"
    / "world"
    / "currents.py",

    repo_root
    / "src"
    / "hydrolab3d"
    / "vehicles"
    / "uuv.py",
]


for path in style_files:

    if not path.exists():
        raise FileNotFoundError(
            f"Expected production source file is missing: {path}"
        )

    print()
    print("=" * 88)
    print(path.relative_to(repo_root))
    print("=" * 88)

    text = path.read_text(
        encoding="utf-8"
    )

    if text.strip():
        print(text)
    else:
        print("<EMPTY FILE>")


print()
print("=" * 88)
print("STEP 18 INSPECTION COMPLETE")
print("=" * 88)

print(
    "Representative production modules and exports were "
    "inspected read-only."
)


src/hydrolab3d/__init__.py
<EMPTY FILE>

src/hydrolab3d/dynamics/__init__.py
"""Dynamics utilities for HydroLab-3D."""

from .buoyancy import (
    buoyancy_force,
    gravity_force,
)
from .drag import quadratic_drag_force
from .hydrodynamics import (
    added_mass_coriolis_matrix,
    added_mass_matrix,
    coriolis_from_block_diagonal_mass,
    hydrodynamic_damping_wrench,
    hydrostatic_wrench,
    rigid_body_coriolis_matrix,
    rigid_body_mass_matrix,
    skew,
    total_mass_matrix,
)
from .kinematics import (
    euler_rate_matrix,
    integrate_position,
    kinematic_matrix_6dof,
    pose_rate_6dof,
)
from .rigid_body import (
    acceleration_from_force,
    integrate_velocity,
)
from .thrusters import (
    thruster_force_body,
    thruster_force_world,
    thruster_wrench_body,
)

__all__ = [
    "acceleration_from_force",
    "added_mass_coriolis_matrix",
    "added_mass_matrix",
    "buoyancy_force",
    "coriolis_from_block_diagonal_mass",
    "euler_rate_matrix",
  

## Step 19 - Promote the Common Measurement Primitive

The notebook-local sensor models have been validated, and the existing production coding style has been inspected.

Production promotion now begins incrementally.

The first reusable component to promote is the common measurement primitive:

$$
\tilde{\mathbf y}
=
\mathbf y
+
\mathbf b
+
\mathbf n.
$$

This functionality belongs in:

```text
src/hydrolab3d/sensors/sensor.py
```

The module will remain function-based.

No `Sensor` base class is introduced because the current validated sensor layer does not require shared mutable state, inheritance, or lifecycle behavior.

The production function will preserve the validated notebook behavior for:

- scalar and one-dimensional measurements;
- deterministic bias;
- seeded Gaussian noise;
- explicit `numpy.random.Generator` usage;
- shape broadcasting;
- finiteness validation;
- rejection of negative noise standard deviation.

Only this common primitive is promoted in this step.

### Validation Criteria

This step passes only if:

1. `sensor.py` is written successfully;
2. the promoted implementation matches the validated notebook behavior;
3. no other sensor production file is modified;
4. the resulting source file can be inspected before any further promotion.

In [25]:
# ---------------------------------------------------------------------
# Step 19 - Promote common measurement primitive
# ---------------------------------------------------------------------

sensor_module_path = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "sensors"
    / "sensor.py"
)


sensor_module_source = '''"""Common sensor-measurement utilities for HydroLab-3D."""

import numpy as np


def measure_with_bias_noise(
    truth,
    *,
    bias=0.0,
    noise_std=0.0,
    rng=None,
):
    """
    Apply deterministic bias and Gaussian measurement noise.

    Parameters
    ----------
    truth : array-like
        Ideal scalar or one-dimensional measurement.

    bias : scalar or array-like, optional
        Deterministic additive bias.

    noise_std : scalar or array-like, optional
        Gaussian noise standard deviation.

    rng : numpy.random.Generator or None, optional
        Explicit random generator. Required when noise is nonzero.

    Returns
    -------
    np.ndarray
        Measured value after bias and Gaussian noise.

    Raises
    ------
    ValueError
        If inputs have invalid shape, contain non-finite values,
        are not broadcast-compatible, or noise_std is negative.

    TypeError
        If stochastic noise is requested without a valid
        numpy.random.Generator.
    """
    truth = np.asarray(
        truth,
        dtype=float,
    )

    bias = np.asarray(
        bias,
        dtype=float,
    )

    noise_std = np.asarray(
        noise_std,
        dtype=float,
    )

    if truth.ndim > 1:
        raise ValueError(
            "truth must be a scalar or one-dimensional array"
        )

    if not np.all(np.isfinite(truth)):
        raise ValueError(
            "truth must contain only finite values"
        )

    if not np.all(np.isfinite(bias)):
        raise ValueError(
            "bias must contain only finite values"
        )

    if not np.all(np.isfinite(noise_std)):
        raise ValueError(
            "noise_std must contain only finite values"
        )

    if np.any(noise_std < 0.0):
        raise ValueError(
            "noise_std must be non-negative"
        )

    try:
        truth_b, bias_b, std_b = np.broadcast_arrays(
            truth,
            bias,
            noise_std,
        )
    except ValueError as exc:
        raise ValueError(
            "truth, bias, and noise_std must be "
            "broadcast-compatible"
        ) from exc

    if np.any(std_b > 0.0):
        if rng is None:
            raise ValueError(
                "rng must be provided when noise_std is nonzero"
            )

        if not isinstance(
            rng,
            np.random.Generator,
        ):
            raise TypeError(
                "rng must be a numpy.random.Generator"
            )

        noise = rng.normal(
            loc=0.0,
            scale=std_b,
            size=truth_b.shape,
        )
    else:
        noise = np.zeros_like(
            truth_b,
            dtype=float,
        )

    return (
        truth_b
        + bias_b
        + noise
    )
'''


sensor_module_path.write_text(
    sensor_module_source,
    encoding="utf-8",
)


# ---------------------------------------------------------------------
# Verify write only
# ---------------------------------------------------------------------

if not sensor_module_path.exists():
    raise AssertionError(
        "sensor.py was not created successfully"
    )

written_source = sensor_module_path.read_text(
    encoding="utf-8"
)

if written_source != sensor_module_source:
    raise AssertionError(
        "Written sensor.py does not match intended source"
    )


# ---------------------------------------------------------------------
# Confirm other sensor placeholders remain untouched
# ---------------------------------------------------------------------

other_sensor_files = [
    sensor_dir / "__init__.py",
    sensor_dir / "imu.py",
    sensor_dir / "depth.py",
    sensor_dir / "dvl.py",
    sensor_dir / "sonar.py",
]

for path in other_sensor_files:
    if path.read_text(
        encoding="utf-8"
    ).strip():
        raise AssertionError(
            f"Unexpected modification detected in "
            f"{path.relative_to(repo_root)}"
        )


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("STEP 19 PROMOTION WRITE PASSED")
print()

print("Promoted file:")
print(
    sensor_module_path.relative_to(repo_root)
)

print()

print("Written source:")
print("-" * 80)
print(written_source)
print("-" * 80)

print()

print(
    "All other sensor placeholder files remain empty: True"
)

STEP 19 PROMOTION WRITE PASSED

Promoted file:
src/hydrolab3d/sensors/sensor.py

Written source:
--------------------------------------------------------------------------------
"""Common sensor-measurement utilities for HydroLab-3D."""

import numpy as np


def measure_with_bias_noise(
    truth,
    *,
    bias=0.0,
    noise_std=0.0,
    rng=None,
):
    """
    Apply deterministic bias and Gaussian measurement noise.

    Parameters
    ----------
    truth : array-like
        Ideal scalar or one-dimensional measurement.

    bias : scalar or array-like, optional
        Deterministic additive bias.

    noise_std : scalar or array-like, optional
        Gaussian noise standard deviation.

    rng : numpy.random.Generator or None, optional
        Explicit random generator. Required when noise is nonzero.

    Returns
    -------
    np.ndarray
        Measured value after bias and Gaussian noise.

    Raises
    ------
    ValueError
        If inputs have invalid shape, contain 

## Step 20 - Re-import and Validate the Production Measurement Primitive

The common measurement primitive has now been promoted into:

```text
src/hydrolab3d/sensors/sensor.py
```

A successful file write is not sufficient evidence that the production implementation is correct.

We therefore:

1. invalidate Python import caches;
2. import the production function under an explicit alias;
3. verify its source location;
4. compare its deterministic behavior against the validated notebook-local function;
5. compare seeded stochastic behavior;
6. verify malformed inputs are rejected consistently.

The notebook-local prototype remains available only as a validation reference.

### Validation Criteria

This step passes only if:

1. the imported function resolves to `src/hydrolab3d/sensors/sensor.py`;
2. ideal and biased measurements match the notebook-local implementation exactly;
3. identical seeds produce identical production and notebook stochastic outputs;
4. production invalid-input behavior matches the validated prototype;
5. no other sensor production file is modified.

In [26]:
# ---------------------------------------------------------------------
# Step 20 - Re-import and independently validate production sensor.py
# ---------------------------------------------------------------------

importlib.invalidate_caches()

from hydrolab3d.sensors.sensor import (
    measure_with_bias_noise as production_measure_with_bias_noise,
)


# =====================================================================
# Verify production source location
# =====================================================================

production_sensor_source = Path(
    inspect.getsourcefile(
        production_measure_with_bias_noise
    )
    or inspect.getfile(
        production_measure_with_bias_noise
    )
).resolve()

expected_sensor_source = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "sensors"
    / "sensor.py"
).resolve()

if production_sensor_source != expected_sensor_source:
    raise AssertionError(
        "Production measurement primitive resolved to "
        f"{production_sensor_source}, expected "
        f"{expected_sensor_source}"
    )


# =====================================================================
# Validation 1: ideal measurement parity
# =====================================================================

truth = np.array([
    1.25,
    -0.75,
    2.50,
])

local_ideal = measure_with_bias_noise(
    truth,
    bias=0.0,
    noise_std=0.0,
)

production_ideal = production_measure_with_bias_noise(
    truth,
    bias=0.0,
    noise_std=0.0,
)

np.testing.assert_array_equal(
    production_ideal,
    local_ideal,
)


# =====================================================================
# Validation 2: deterministic bias parity
# =====================================================================

bias = np.array([
    0.10,
    -0.05,
    0.20,
])

local_biased = measure_with_bias_noise(
    truth,
    bias=bias,
    noise_std=0.0,
)

production_biased = production_measure_with_bias_noise(
    truth,
    bias=bias,
    noise_std=0.0,
)

np.testing.assert_array_equal(
    production_biased,
    local_biased,
)


# =====================================================================
# Validation 3: seeded stochastic parity
# =====================================================================

local_rng = np.random.default_rng(
    202620
)

production_rng = np.random.default_rng(
    202620
)

noise_std = np.array([
    0.01,
    0.02,
    0.03,
])

local_noisy = measure_with_bias_noise(
    truth,
    bias=bias,
    noise_std=noise_std,
    rng=local_rng,
)

production_noisy = production_measure_with_bias_noise(
    truth,
    bias=bias,
    noise_std=noise_std,
    rng=production_rng,
)

np.testing.assert_array_equal(
    production_noisy,
    local_noisy,
)


# =====================================================================
# Validation 4: scalar behavior parity
# =====================================================================

local_scalar = measure_with_bias_noise(
    12.0,
    bias=0.15,
    noise_std=0.0,
)

production_scalar = production_measure_with_bias_noise(
    12.0,
    bias=0.15,
    noise_std=0.0,
)

np.testing.assert_array_equal(
    production_scalar,
    local_scalar,
)


# =====================================================================
# Validation 5: invalid-input parity
# =====================================================================

invalid_cases = [
    {
        "truth": np.zeros((2, 2)),
        "bias": 0.0,
        "noise_std": 0.0,
        "rng": None,
    },
    {
        "truth": np.array([1.0, np.nan]),
        "bias": 0.0,
        "noise_std": 0.0,
        "rng": None,
    },
    {
        "truth": np.array([1.0, 2.0]),
        "bias": np.zeros(3),
        "noise_std": 0.0,
        "rng": None,
    },
    {
        "truth": np.array([1.0, 2.0]),
        "bias": 0.0,
        "noise_std": -0.01,
        "rng": np.random.default_rng(1),
    },
    {
        "truth": np.array([1.0, 2.0]),
        "bias": 0.0,
        "noise_std": 0.01,
        "rng": "not-a-generator",
    },
]


for case in invalid_cases:

    local_exception = None
    production_exception = None

    try:
        measure_with_bias_noise(
            case["truth"],
            bias=case["bias"],
            noise_std=case["noise_std"],
            rng=case["rng"],
        )
    except Exception as exc:
        local_exception = type(exc)

    try:
        production_measure_with_bias_noise(
            case["truth"],
            bias=case["bias"],
            noise_std=case["noise_std"],
            rng=case["rng"],
        )
    except Exception as exc:
        production_exception = type(exc)

    if local_exception is None:
        raise AssertionError(
            "Notebook-local reference unexpectedly accepted "
            "an invalid case"
        )

    if production_exception is None:
        raise AssertionError(
            "Production implementation unexpectedly accepted "
            "an invalid case"
        )

    if production_exception is not local_exception:
        raise AssertionError(
            "Production/local exception mismatch: "
            f"{production_exception} vs {local_exception}"
        )


# =====================================================================
# Validation 6: other sensor files remain untouched
# =====================================================================

for path in [
    sensor_dir / "__init__.py",
    sensor_dir / "imu.py",
    sensor_dir / "depth.py",
    sensor_dir / "dvl.py",
    sensor_dir / "sonar.py",
]:
    if path.read_text(
        encoding="utf-8"
    ).strip():
        raise AssertionError(
            f"Unexpected modification detected in "
            f"{path.relative_to(repo_root)}"
        )


# =====================================================================
# Report
# =====================================================================

print("STEP 20 VALIDATION PASSED")
print()

print("Production source:")
print(
    production_sensor_source.relative_to(
        repo_root
    )
)

print()

print("Ideal parity:")
print(
    np.array_equal(
        production_ideal,
        local_ideal,
    )
)

print()

print("Biased parity:")
print(
    np.array_equal(
        production_biased,
        local_biased,
    )
)

print()

print("Seeded stochastic parity:")
print(
    np.array_equal(
        production_noisy,
        local_noisy,
    )
)

print()

print("Production noisy measurement:")
print(
    production_noisy
)

print()

print(
    "Invalid-input behavior parity: True"
)

print(
    "Other sensor placeholders untouched: True"
)

STEP 20 VALIDATION PASSED

Production source:
src/hydrolab3d/sensors/sensor.py

Ideal parity:
True

Biased parity:
True

Seeded stochastic parity:
True

Production noisy measurement:
[ 1.35132654 -0.81479033  2.69258788]

Invalid-input behavior parity: True
Other sensor placeholders untouched: True


## Step 21 - Promote the IMU Models

The common production measurement primitive has been promoted and independently validated.

We now promote the validated IMU functionality into:

```text
src/hydrolab3d/sensors/imu.py
```

The production module will contain:

```text
ideal_gyroscope_measurement
gyroscope_measurement
ideal_accelerometer_measurement
accelerometer_measurement
```

The implementation reuses existing production utilities:

```text
hydrolab3d.utils.transforms.rotation_body_to_world
hydrolab3d.sensors.sensor.measure_with_bias_noise
```

No new sensor physics is introduced during promotion.

The gyroscope continues to measure body angular velocity:

$$
\boldsymbol{\omega}_B
=
\begin{bmatrix}
p\\
q\\
r
\end{bmatrix}.
$$

The accelerometer continues to measure body-frame specific force:

$$
\mathbf f_B
=
\dot{\mathbf v}_B
+
\boldsymbol{\omega}_B
\times
\mathbf v_B
-
R_W^B\mathbf g_W.
$$

### Validation Criteria

This step passes only if:

1. `imu.py` is written successfully;
2. the promoted source matches the validated notebook behavior;
3. `sensor.py` remains unchanged;
4. `depth.py`, `dvl.py`, `sonar.py`, and `sensors/__init__.py` remain untouched;
5. no production import validation is claimed until the next step.

In [27]:
# ---------------------------------------------------------------------
# Step 21 - Promote validated IMU models into production
# ---------------------------------------------------------------------

imu_module_path = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "sensors"
    / "imu.py"
)


# Preserve sensor.py so we can verify it is not modified.
sensor_source_before = (
    sensor_module_path.read_text(
        encoding="utf-8"
    )
)


imu_module_source = '''"""Inertial sensor models for HydroLab-3D."""

import numpy as np

from hydrolab3d.sensors.sensor import (
    measure_with_bias_noise,
)
from hydrolab3d.utils.transforms import (
    rotation_body_to_world,
)


def ideal_gyroscope_measurement(
    body_velocity,
):
    """
    Return ideal body-frame angular velocity.

    Parameters
    ----------
    body_velocity : array-like
        Generalized body velocity [u, v, w, p, q, r].

    Returns
    -------
    np.ndarray
        Body angular velocity [p, q, r] in rad/s.
    """
    body_velocity = np.asarray(
        body_velocity,
        dtype=float,
    )

    if body_velocity.shape != (6,):
        raise ValueError(
            "body_velocity must have shape (6,)"
        )

    if not np.all(
        np.isfinite(body_velocity)
    ):
        raise ValueError(
            "body_velocity must contain only finite values"
        )

    return body_velocity[3:6].copy()


def gyroscope_measurement(
    body_velocity,
    *,
    bias=0.0,
    noise_std=0.0,
    rng=None,
):
    """
    Return imperfect body-frame gyroscope measurement.
    """
    ideal = ideal_gyroscope_measurement(
        body_velocity
    )

    measured = measure_with_bias_noise(
        ideal,
        bias=bias,
        noise_std=noise_std,
        rng=rng,
    )

    if measured.shape != (3,):
        raise ValueError(
            "gyroscope bias and noise_std must be "
            "scalar or broadcast-compatible with shape (3,)"
        )

    return measured


def ideal_accelerometer_measurement(
    pose,
    body_velocity,
    body_acceleration,
    *,
    gravity=9.81,
):
    """
    Return ideal body-frame accelerometer specific force.

    Parameters
    ----------
    pose : array-like
        Pose [x, y, z, roll, pitch, yaw].

    body_velocity : array-like
        Generalized body velocity [u, v, w, p, q, r].

    body_acceleration : array-like
        Generalized body velocity derivative
        [u_dot, v_dot, w_dot, p_dot, q_dot, r_dot].

    gravity : float, optional
        Positive gravitational acceleration magnitude in m/s^2.

    Returns
    -------
    np.ndarray
        Body-frame specific force in m/s^2.
    """
    pose = np.asarray(
        pose,
        dtype=float,
    )

    body_velocity = np.asarray(
        body_velocity,
        dtype=float,
    )

    body_acceleration = np.asarray(
        body_acceleration,
        dtype=float,
    )

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if body_velocity.shape != (6,):
        raise ValueError(
            "body_velocity must have shape (6,)"
        )

    if body_acceleration.shape != (6,):
        raise ValueError(
            "body_acceleration must have shape (6,)"
        )

    if not np.all(np.isfinite(pose)):
        raise ValueError(
            "pose must contain only finite values"
        )

    if not np.all(
        np.isfinite(body_velocity)
    ):
        raise ValueError(
            "body_velocity must contain only finite values"
        )

    if not np.all(
        np.isfinite(body_acceleration)
    ):
        raise ValueError(
            "body_acceleration must contain only finite values"
        )

    gravity = float(gravity)

    if (
        not np.isfinite(gravity)
        or gravity <= 0.0
    ):
        raise ValueError(
            "gravity must be finite and strictly positive"
        )

    roll, pitch, yaw = pose[3:]

    R_bw = rotation_body_to_world(
        roll,
        pitch,
        yaw,
    )

    gravity_world = np.array([
        0.0,
        0.0,
        -gravity,
    ])

    velocity_body = body_velocity[:3]
    omega_body = body_velocity[3:]
    velocity_dot_body = body_acceleration[:3]

    inertial_acceleration_body = (
        velocity_dot_body
        + np.cross(
            omega_body,
            velocity_body,
        )
    )

    gravity_body = (
        R_bw.T
        @ gravity_world
    )

    return (
        inertial_acceleration_body
        - gravity_body
    )


def accelerometer_measurement(
    pose,
    body_velocity,
    body_acceleration,
    *,
    gravity=9.81,
    bias=0.0,
    noise_std=0.0,
    rng=None,
):
    """
    Return imperfect body-frame accelerometer measurement.
    """
    ideal = ideal_accelerometer_measurement(
        pose,
        body_velocity,
        body_acceleration,
        gravity=gravity,
    )

    measured = measure_with_bias_noise(
        ideal,
        bias=bias,
        noise_std=noise_std,
        rng=rng,
    )

    if measured.shape != (3,):
        raise ValueError(
            "accelerometer bias and noise_std must be "
            "scalar or broadcast-compatible with shape (3,)"
        )

    return measured
'''


# ---------------------------------------------------------------------
# Write imu.py
# ---------------------------------------------------------------------

imu_module_path.write_text(
    imu_module_source,
    encoding="utf-8",
)


# ---------------------------------------------------------------------
# Verify exact write
# ---------------------------------------------------------------------

if not imu_module_path.exists():
    raise AssertionError(
        "imu.py was not created successfully"
    )

written_imu_source = imu_module_path.read_text(
    encoding="utf-8"
)

if written_imu_source != imu_module_source:
    raise AssertionError(
        "Written imu.py does not match intended source"
    )


# ---------------------------------------------------------------------
# Verify sensor.py was not modified
# ---------------------------------------------------------------------

sensor_source_after = (
    sensor_module_path.read_text(
        encoding="utf-8"
    )
)

if sensor_source_after != sensor_source_before:
    raise AssertionError(
        "sensor.py changed unexpectedly during IMU promotion"
    )


# ---------------------------------------------------------------------
# Verify remaining placeholders remain untouched
# ---------------------------------------------------------------------

for path in [
    sensor_dir / "__init__.py",
    sensor_dir / "depth.py",
    sensor_dir / "dvl.py",
    sensor_dir / "sonar.py",
]:
    if path.read_text(
        encoding="utf-8"
    ).strip():
        raise AssertionError(
            f"Unexpected modification detected in "
            f"{path.relative_to(repo_root)}"
        )


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("STEP 21 PROMOTION WRITE PASSED")
print()

print("Promoted file:")
print(
    imu_module_path.relative_to(repo_root)
)

print()

print("Written source:")
print("-" * 80)
print(written_imu_source)
print("-" * 80)

print()

print(
    "sensor.py unchanged: True"
)

print(
    "Remaining sensor placeholders untouched: True"
)

STEP 21 PROMOTION WRITE PASSED

Promoted file:
src/hydrolab3d/sensors/imu.py

Written source:
--------------------------------------------------------------------------------
"""Inertial sensor models for HydroLab-3D."""

import numpy as np

from hydrolab3d.sensors.sensor import (
    measure_with_bias_noise,
)
from hydrolab3d.utils.transforms import (
    rotation_body_to_world,
)


def ideal_gyroscope_measurement(
    body_velocity,
):
    """
    Return ideal body-frame angular velocity.

    Parameters
    ----------
    body_velocity : array-like
        Generalized body velocity [u, v, w, p, q, r].

    Returns
    -------
    np.ndarray
        Body angular velocity [p, q, r] in rad/s.
    """
    body_velocity = np.asarray(
        body_velocity,
        dtype=float,
    )

    if body_velocity.shape != (6,):
        raise ValueError(
            "body_velocity must have shape (6,)"
        )

    if not np.all(
        np.isfinite(body_velocity)
    ):
        raise ValueError

## Step 22 - Re-import and Validate the Production IMU

The validated IMU implementation has now been promoted into:

```text
src/hydrolab3d/sensors/imu.py
```

A successful source write is not sufficient evidence of correctness.

We therefore re-import the production IMU functions under explicit aliases and compare them against the already validated notebook-local implementations.

The validation covers:

- source-file resolution;
- ideal gyroscope parity;
- imperfect gyroscope parity;
- ideal accelerometer parity;
- imperfect accelerometer parity;
- orientation-sensitive gravity cases;
- seeded stochastic reproducibility;
- invalid-input behavior.

No additional production sensor file is modified in this step.

### Validation Criteria

This step passes only if:

1. all four IMU functions resolve to `src/hydrolab3d/sensors/imu.py`;
2. production and notebook-local ideal gyroscope outputs match exactly;
3. production and notebook-local imperfect gyroscope outputs match exactly under identical seeds;
4. production and notebook-local ideal accelerometer outputs match;
5. production and notebook-local imperfect accelerometer outputs match under identical seeds;
6. the validated roll/pitch gravity-sign cases remain correct;
7. invalid-input behavior remains consistent.

In [28]:
# ---------------------------------------------------------------------
# Step 22 - Re-import and independently validate production IMU
# ---------------------------------------------------------------------

importlib.invalidate_caches()

from hydrolab3d.sensors.imu import (
    ideal_gyroscope_measurement as production_ideal_gyroscope_measurement,
    gyroscope_measurement as production_gyroscope_measurement,
    ideal_accelerometer_measurement as production_ideal_accelerometer_measurement,
    accelerometer_measurement as production_accelerometer_measurement,
)


# =====================================================================
# Verify production source locations
# =====================================================================

expected_imu_source = imu_module_path.resolve()

production_imu_functions = {
    "ideal_gyroscope_measurement":
        production_ideal_gyroscope_measurement,

    "gyroscope_measurement":
        production_gyroscope_measurement,

    "ideal_accelerometer_measurement":
        production_ideal_accelerometer_measurement,

    "accelerometer_measurement":
        production_accelerometer_measurement,
}


for name, function in production_imu_functions.items():

    source = Path(
        inspect.getsourcefile(function)
        or inspect.getfile(function)
    ).resolve()

    if source != expected_imu_source:
        raise AssertionError(
            f"{name} resolved to {source}, "
            f"expected {expected_imu_source}"
        )


# =====================================================================
# Validation 1: ideal gyroscope parity
# =====================================================================

nu_test = np.array([
    1.25,
    -0.45,
    0.30,
    0.07,
    -0.05,
    0.11,
])

local_gyro_ideal = ideal_gyroscope_measurement(
    nu_test
)

production_gyro_ideal = (
    production_ideal_gyroscope_measurement(
        nu_test
    )
)

np.testing.assert_array_equal(
    production_gyro_ideal,
    local_gyro_ideal,
)


# =====================================================================
# Validation 2: imperfect gyroscope parity
# =====================================================================

gyro_bias_test = np.array([
    0.002,
    -0.001,
    0.0015,
])

gyro_noise_test = np.array([
    0.001,
    0.002,
    0.0015,
])

local_gyro_rng = np.random.default_rng(
    22001
)

production_gyro_rng = np.random.default_rng(
    22001
)

local_gyro_noisy = gyroscope_measurement(
    nu_test,
    bias=gyro_bias_test,
    noise_std=gyro_noise_test,
    rng=local_gyro_rng,
)

production_gyro_noisy = (
    production_gyroscope_measurement(
        nu_test,
        bias=gyro_bias_test,
        noise_std=gyro_noise_test,
        rng=production_gyro_rng,
    )
)

np.testing.assert_array_equal(
    production_gyro_noisy,
    local_gyro_noisy,
)


# =====================================================================
# Validation 3: ideal accelerometer parity
# =====================================================================

eta_test = np.array([
    2.0,
    -3.0,
    -14.0,
    0.28,
    -0.19,
    0.63,
])

nu_accel_test = np.array([
    1.10,
    -0.32,
    0.21,
    0.05,
    -0.07,
    0.10,
])

nu_dot_test = np.array([
    -0.22,
    0.14,
    -0.06,
    0.03,
    -0.02,
    0.01,
])

local_accel_ideal = (
    ideal_accelerometer_measurement(
        eta_test,
        nu_accel_test,
        nu_dot_test,
    )
)

production_accel_ideal = (
    production_ideal_accelerometer_measurement(
        eta_test,
        nu_accel_test,
        nu_dot_test,
    )
)

np.testing.assert_allclose(
    production_accel_ideal,
    local_accel_ideal,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 4: imperfect accelerometer parity
# =====================================================================

accel_bias_test = np.array([
    0.03,
    -0.02,
    0.04,
])

accel_noise_test = np.array([
    0.01,
    0.015,
    0.02,
])

local_accel_rng = np.random.default_rng(
    22002
)

production_accel_rng = np.random.default_rng(
    22002
)

local_accel_noisy = accelerometer_measurement(
    eta_test,
    nu_accel_test,
    nu_dot_test,
    bias=accel_bias_test,
    noise_std=accel_noise_test,
    rng=local_accel_rng,
)

production_accel_noisy = (
    production_accelerometer_measurement(
        eta_test,
        nu_accel_test,
        nu_dot_test,
        bias=accel_bias_test,
        noise_std=accel_noise_test,
        rng=production_accel_rng,
    )
)

np.testing.assert_array_equal(
    production_accel_noisy,
    local_accel_noisy,
)


# =====================================================================
# Validation 5: stationary level case
# =====================================================================

production_stationary = (
    production_ideal_accelerometer_measurement(
        np.zeros(6),
        np.zeros(6),
        np.zeros(6),
    )
)

np.testing.assert_allclose(
    production_stationary,
    np.array([
        0.0,
        0.0,
        9.81,
    ]),
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 6: +90 degree roll case
# =====================================================================

eta_roll_90 = np.array([
    0.0,
    0.0,
    0.0,
    np.pi / 2.0,
    0.0,
    0.0,
])

production_roll_90 = (
    production_ideal_accelerometer_measurement(
        eta_roll_90,
        np.zeros(6),
        np.zeros(6),
    )
)

np.testing.assert_allclose(
    production_roll_90,
    np.array([
        0.0,
        9.81,
        0.0,
    ]),
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 7: +90 degree pitch case
# =====================================================================

eta_pitch_90 = np.array([
    0.0,
    0.0,
    0.0,
    0.0,
    np.pi / 2.0,
    0.0,
])

production_pitch_90 = (
    production_ideal_accelerometer_measurement(
        eta_pitch_90,
        np.zeros(6),
        np.zeros(6),
    )
)

np.testing.assert_allclose(
    production_pitch_90,
    np.array([
        -9.81,
        0.0,
        0.0,
    ]),
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 8: invalid-input behavior
# =====================================================================

invalid_gyro_cases = [
    np.zeros(5),
    np.zeros((6, 1)),
    np.array([
        0.0,
        0.0,
        0.0,
        np.nan,
        0.0,
        0.0,
    ]),
]

for bad_nu in invalid_gyro_cases:
    try:
        production_ideal_gyroscope_measurement(
            bad_nu
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Production gyroscope accepted invalid input"
        )


try:
    production_gyroscope_measurement(
        nu_test,
        noise_std=-0.01,
        rng=np.random.default_rng(1),
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Production gyroscope accepted negative noise_std"
    )


try:
    production_accelerometer_measurement(
        eta_test,
        nu_accel_test,
        nu_dot_test,
        noise_std=0.01,
        rng="not-a-generator",
    )
except TypeError:
    pass
else:
    raise AssertionError(
        "Production accelerometer accepted invalid RNG"
    )


# =====================================================================
# Report
# =====================================================================

print("STEP 22 VALIDATION PASSED")
print()

print("Production IMU source:")
print(
    expected_imu_source.relative_to(
        repo_root
    )
)

print()

print("Ideal gyroscope parity:")
print(
    np.array_equal(
        production_gyro_ideal,
        local_gyro_ideal,
    )
)

print()

print("Seeded gyroscope parity:")
print(
    np.array_equal(
        production_gyro_noisy,
        local_gyro_noisy,
    )
)

print()

print("Ideal accelerometer parity:")
print(
    np.allclose(
        production_accel_ideal,
        local_accel_ideal,
        rtol=0.0,
        atol=1e-12,
    )
)

print()

print("Seeded accelerometer parity:")
print(
    np.array_equal(
        production_accel_noisy,
        local_accel_noisy,
    )
)

print()

print("Stationary level specific force [m/s^2]:")
print(
    production_stationary
)

print()

print("+90 deg roll specific force [m/s^2]:")
print(
    production_roll_90
)

print()

print("+90 deg pitch specific force [m/s^2]:")
print(
    production_pitch_90
)

print()

print("Invalid-input behavior validated: True")

STEP 22 VALIDATION PASSED

Production IMU source:
src/hydrolab3d/sensors/imu.py

Ideal gyroscope parity:
True

Seeded gyroscope parity:
True

Ideal accelerometer parity:
True

Seeded accelerometer parity:
True

Stationary level specific force [m/s^2]:
[0.   0.   9.81]

+90 deg roll specific force [m/s^2]:
[0.00000000e+00 9.81000000e+00 6.00689255e-16]

+90 deg pitch specific force [m/s^2]:
[-9.81000000e+00  0.00000000e+00  6.00689255e-16]

Invalid-input behavior validated: True


## Step 23 - Promote the Depth Sensor

The notebook-local depth model has already been validated for:

- HydroLab-3D's world-$z$ convention;
- positive depth below the water surface;
- configurable surface reference;
- deterministic bias;
- seeded Gaussian noise;
- above-surface rejection;
- malformed-input rejection.

We now promote the reusable depth functionality into:

```text
src/hydrolab3d/sensors/depth.py
```

The production module will contain:

```text
ideal_depth_measurement
depth_measurement
```

The ideal geometric relation remains:

$$
d
=
z_{surface}
-
z_W.
$$

For HydroLab-3D's default surface:

$$
z_{surface}=0,
$$

so:

$$
d=-z_W.
$$

The imperfect measurement remains:

$$
\tilde d
=
d
+
b_d
+
n_d.
$$

The production implementation reuses:

```text
hydrolab3d.sensors.sensor.measure_with_bias_noise
```

No pressure conversion, clipping, drift, or update-rate behavior is introduced during promotion.

### Validation Criteria

This step passes only if:

1. `depth.py` is written successfully;
2. its implementation preserves the validated notebook behavior;
3. `sensor.py` and `imu.py` remain unchanged;
4. `dvl.py`, `sonar.py`, and `sensors/__init__.py` remain untouched;
5. production import validation is deferred until the next step.

In [29]:
# ---------------------------------------------------------------------
# Step 23 - Promote validated depth sensor into production
# ---------------------------------------------------------------------

depth_module_path = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "sensors"
    / "depth.py"
)


# =====================================================================
# Preserve already-promoted source files
# =====================================================================

sensor_source_before = (
    sensor_module_path.read_text(
        encoding="utf-8"
    )
)

imu_source_before = (
    imu_module_path.read_text(
        encoding="utf-8"
    )
)


# =====================================================================
# Production depth source
# =====================================================================

depth_module_source = '''"""Depth sensor models for HydroLab-3D."""

import numpy as np

from hydrolab3d.sensors.sensor import (
    measure_with_bias_noise,
)


def ideal_depth_measurement(
    pose,
    *,
    surface_z=0.0,
):
    """
    Return ideal depth below the modeled water surface.

    Parameters
    ----------
    pose : array-like
        Pose [x, y, z, roll, pitch, yaw].

    surface_z : float, optional
        World-frame z coordinate of the water surface.

    Returns
    -------
    float
        Positive depth below the water surface in meters.

    Raises
    ------
    ValueError
        If pose is malformed, inputs are non-finite, or the
        vehicle lies above the modeled water surface.
    """
    pose = np.asarray(
        pose,
        dtype=float,
    )

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if not np.all(np.isfinite(pose)):
        raise ValueError(
            "pose must contain only finite values"
        )

    surface_z = float(surface_z)

    if not np.isfinite(surface_z):
        raise ValueError(
            "surface_z must be finite"
        )

    depth = (
        surface_z
        - pose[2]
    )

    tolerance = 1e-12

    if depth < -tolerance:
        raise ValueError(
            "vehicle is above the modeled water surface"
        )

    if abs(depth) <= tolerance:
        depth = 0.0

    return float(depth)


def depth_measurement(
    pose,
    *,
    surface_z=0.0,
    bias=0.0,
    noise_std=0.0,
    rng=None,
):
    """
    Return imperfect depth measurement.

    Parameters
    ----------
    pose : array-like
        Pose [x, y, z, roll, pitch, yaw].

    surface_z : float, optional
        World-frame z coordinate of the water surface.

    bias : scalar, optional
        Additive depth bias in meters.

    noise_std : scalar, optional
        Gaussian depth-noise standard deviation in meters.

    rng : numpy.random.Generator or None, optional
        Explicit random generator when noise is nonzero.

    Returns
    -------
    float
        Depth measurement in meters.
    """
    ideal = ideal_depth_measurement(
        pose,
        surface_z=surface_z,
    )

    measured = measure_with_bias_noise(
        ideal,
        bias=bias,
        noise_std=noise_std,
        rng=rng,
    )

    measured = np.asarray(
        measured,
        dtype=float,
    )

    if measured.shape != ():
        raise ValueError(
            "depth bias and noise_std must be scalar"
        )

    return float(measured)
'''


# =====================================================================
# Write depth.py
# =====================================================================

depth_module_path.write_text(
    depth_module_source,
    encoding="utf-8",
)


# =====================================================================
# Verify exact write
# =====================================================================

if not depth_module_path.exists():
    raise AssertionError(
        "depth.py was not created successfully"
    )

written_depth_source = (
    depth_module_path.read_text(
        encoding="utf-8"
    )
)

if written_depth_source != depth_module_source:
    raise AssertionError(
        "Written depth.py does not match intended source"
    )


# =====================================================================
# Verify previously promoted files remain unchanged
# =====================================================================

if (
    sensor_module_path.read_text(
        encoding="utf-8"
    )
    != sensor_source_before
):
    raise AssertionError(
        "sensor.py changed unexpectedly during depth promotion"
    )


if (
    imu_module_path.read_text(
        encoding="utf-8"
    )
    != imu_source_before
):
    raise AssertionError(
        "imu.py changed unexpectedly during depth promotion"
    )


# =====================================================================
# Verify remaining placeholders remain untouched
# =====================================================================

for path in [
    sensor_dir / "__init__.py",
    sensor_dir / "dvl.py",
    sensor_dir / "sonar.py",
]:
    if path.read_text(
        encoding="utf-8"
    ).strip():
        raise AssertionError(
            f"Unexpected modification detected in "
            f"{path.relative_to(repo_root)}"
        )


# =====================================================================
# Report
# =====================================================================

print("STEP 23 PROMOTION WRITE PASSED")
print()

print("Promoted file:")
print(
    depth_module_path.relative_to(
        repo_root
    )
)

print()

print("Written source:")
print("-" * 80)
print(written_depth_source)
print("-" * 80)

print()

print("sensor.py unchanged: True")
print("imu.py unchanged: True")

print(
    "Remaining sensor placeholders untouched: True"
)

STEP 23 PROMOTION WRITE PASSED

Promoted file:
src/hydrolab3d/sensors/depth.py

Written source:
--------------------------------------------------------------------------------
"""Depth sensor models for HydroLab-3D."""

import numpy as np

from hydrolab3d.sensors.sensor import (
    measure_with_bias_noise,
)


def ideal_depth_measurement(
    pose,
    *,
    surface_z=0.0,
):
    """
    Return ideal depth below the modeled water surface.

    Parameters
    ----------
    pose : array-like
        Pose [x, y, z, roll, pitch, yaw].

    surface_z : float, optional
        World-frame z coordinate of the water surface.

    Returns
    -------
    float
        Positive depth below the water surface in meters.

    Raises
    ------
    ValueError
        If pose is malformed, inputs are non-finite, or the
        vehicle lies above the modeled water surface.
    """
    pose = np.asarray(
        pose,
        dtype=float,
    )

    if pose.shape != (6,):
        raise ValueError(


## Step 24 - Re-import and Validate the Production Depth Sensor

The validated depth model has now been promoted into:

```text
src/hydrolab3d/sensors/depth.py
```

We now re-import the production functions and compare them against the already validated notebook-local depth implementation.

The validation covers:

- source-file resolution;
- ideal depth parity;
- configurable water-surface reference;
- deterministic bias;
- seeded Gaussian noise;
- surface behavior;
- above-surface rejection;
- malformed and non-finite inputs.

No additional production sensor file is modified in this step.

### Validation Criteria

This step passes only if:

1. both production depth functions resolve to `src/hydrolab3d/sensors/depth.py`;
2. ideal production depth matches the notebook-local implementation;
3. deterministic bias matches exactly;
4. seeded stochastic output matches exactly under identical seeds;
5. configurable `surface_z` behavior remains correct;
6. above-surface states are rejected;
7. malformed and non-finite inputs remain rejected consistently.

In [30]:
# ---------------------------------------------------------------------
# Step 24 - Re-import and independently validate production depth
# ---------------------------------------------------------------------

importlib.invalidate_caches()

from hydrolab3d.sensors.depth import (
    ideal_depth_measurement as production_ideal_depth_measurement,
    depth_measurement as production_depth_measurement,
)


# =====================================================================
# Verify production source locations
# =====================================================================

expected_depth_source = depth_module_path.resolve()

production_depth_functions = {
    "ideal_depth_measurement":
        production_ideal_depth_measurement,

    "depth_measurement":
        production_depth_measurement,
}


for name, function in production_depth_functions.items():

    source = Path(
        inspect.getsourcefile(function)
        or inspect.getfile(function)
    ).resolve()

    if source != expected_depth_source:
        raise AssertionError(
            f"{name} resolved to {source}, "
            f"expected {expected_depth_source}"
        )


# =====================================================================
# Validation 1: ideal depth parity
# =====================================================================

eta_depth_test = np.array([
    5.0,
    -7.0,
    -18.5,
    0.25,
    -0.10,
    0.80,
])

local_depth_ideal = ideal_depth_measurement(
    eta_depth_test
)

production_depth_ideal = (
    production_ideal_depth_measurement(
        eta_depth_test
    )
)

np.testing.assert_allclose(
    production_depth_ideal,
    local_depth_ideal,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 2: configurable surface reference
# =====================================================================

local_offset_surface = ideal_depth_measurement(
    eta_depth_test,
    surface_z=3.0,
)

production_offset_surface = (
    production_ideal_depth_measurement(
        eta_depth_test,
        surface_z=3.0,
    )
)

np.testing.assert_allclose(
    production_offset_surface,
    local_offset_surface,
    rtol=0.0,
    atol=0.0,
)

np.testing.assert_allclose(
    production_offset_surface,
    21.5,
    rtol=0.0,
    atol=1e-15,
)


# =====================================================================
# Validation 3: deterministic bias parity
# =====================================================================

local_depth_biased = depth_measurement(
    eta_depth_test,
    bias=0.35,
    noise_std=0.0,
)

production_depth_biased = (
    production_depth_measurement(
        eta_depth_test,
        bias=0.35,
        noise_std=0.0,
    )
)

np.testing.assert_allclose(
    production_depth_biased,
    local_depth_biased,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 4: seeded stochastic parity
# =====================================================================

local_depth_rng = np.random.default_rng(
    24001
)

production_depth_rng = np.random.default_rng(
    24001
)

local_depth_noisy = depth_measurement(
    eta_depth_test,
    bias=0.10,
    noise_std=0.04,
    rng=local_depth_rng,
)

production_depth_noisy = (
    production_depth_measurement(
        eta_depth_test,
        bias=0.10,
        noise_std=0.04,
        rng=production_depth_rng,
    )
)

np.testing.assert_allclose(
    production_depth_noisy,
    local_depth_noisy,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 5: exact surface behavior
# =====================================================================

eta_surface = np.array([
    3.0,
    -2.0,
    0.0,
    0.2,
    -0.3,
    0.4,
])

production_surface_depth = (
    production_ideal_depth_measurement(
        eta_surface
    )
)

np.testing.assert_allclose(
    production_surface_depth,
    0.0,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 6: above-surface rejection
# =====================================================================

eta_above_surface = np.array([
    0.0,
    0.0,
    0.25,
    0.0,
    0.0,
    0.0,
])

try:
    production_ideal_depth_measurement(
        eta_above_surface
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Production ideal depth accepted an above-surface state"
    )


# =====================================================================
# Validation 7: invalid-input behavior
# =====================================================================

invalid_pose_cases = [
    np.zeros(5),
    np.zeros((6, 1)),
    np.array([
        0.0,
        0.0,
        np.nan,
        0.0,
        0.0,
        0.0,
    ]),
]

for bad_pose in invalid_pose_cases:

    try:
        production_ideal_depth_measurement(
            bad_pose
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Production depth accepted invalid pose input"
        )


for bad_surface in [
    np.nan,
    np.inf,
    -np.inf,
]:
    try:
        production_ideal_depth_measurement(
            eta_depth_test,
            surface_z=bad_surface,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Production depth accepted non-finite surface_z"
        )


try:
    production_depth_measurement(
        eta_depth_test,
        bias=np.array([
            0.1,
            0.2,
        ]),
        noise_std=0.0,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Production depth accepted non-scalar bias"
    )


try:
    production_depth_measurement(
        eta_depth_test,
        noise_std=-0.01,
        rng=np.random.default_rng(1),
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Production depth accepted negative noise_std"
    )


try:
    production_depth_measurement(
        eta_depth_test,
        noise_std=0.01,
        rng="not-a-generator",
    )
except TypeError:
    pass
else:
    raise AssertionError(
        "Production depth accepted invalid RNG"
    )


# =====================================================================
# Report
# =====================================================================

print("STEP 24 VALIDATION PASSED")
print()

print("Production depth source:")
print(
    expected_depth_source.relative_to(
        repo_root
    )
)

print()

print("Ideal depth parity:")
print(
    production_depth_ideal
    == local_depth_ideal
)

print()

print("Offset-surface depth [m]:")
print(
    production_offset_surface
)

print()

print("Biased depth parity:")
print(
    production_depth_biased
    == local_depth_biased
)

print()

print("Seeded stochastic parity:")
print(
    production_depth_noisy
    == local_depth_noisy
)

print()

print("Production noisy depth [m]:")
print(
    production_depth_noisy
)

print()

print("Exact surface depth [m]:")
print(
    production_surface_depth
)

print()

print("Above-surface rejection: True")
print("Invalid-input behavior validated: True")

STEP 24 VALIDATION PASSED

Production depth source:
src/hydrolab3d/sensors/depth.py

Ideal depth parity:
True

Offset-surface depth [m]:
21.5

Biased depth parity:
True

Seeded stochastic parity:
True

Production noisy depth [m]:
18.596400159090482

Exact surface depth [m]:
0.0

Above-surface rejection: True
Invalid-input behavior validated: True


## Step 25 - Promote the DVL Sensor

The notebook-local DVL model has already been validated against HydroLab-3D's production water-current utilities.

The DVL measures translational vehicle velocity relative to the surrounding water:

$$
\mathbf v_{r,B}
=
\mathbf v_B
-
\mathbf v_{c,B}.
$$

HydroLab-3D already provides the authoritative production helper:

```text
relative_velocity_6dof(
    pose,
    body_velocity,
    current_velocity_world,
)
```

so the DVL production implementation will reuse that function directly rather than duplicating frame-transform logic.

The production module will contain:

```text
ideal_dvl_measurement
dvl_measurement
```

The imperfect model remains:

$$
\tilde{\mathbf v}_{DVL}
=
\mathbf v_{r,B}
+
\mathbf b_{DVL}
+
\mathbf n_{DVL}.
$$

Only translational water-relative velocity is reported.

Rotational velocity does not appear in the DVL measurement output.

### Validation Criteria

This step passes only if:

1. `dvl.py` is written successfully;
2. the production implementation reuses `relative_velocity_6dof`;
3. previously promoted sensor modules remain unchanged;
4. `sonar.py` and `sensors/__init__.py` remain untouched;
5. no production import validation is claimed until the next step.

In [32]:
# ---------------------------------------------------------------------
# Step 25 - Promote validated DVL model into production
# ---------------------------------------------------------------------

dvl_module_path = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "sensors"
    / "dvl.py"
)


# =====================================================================
# Preserve already-promoted production files
# =====================================================================

sensor_source_before = (
    sensor_module_path.read_text(
        encoding="utf-8"
    )
)

imu_source_before = (
    imu_module_path.read_text(
        encoding="utf-8"
    )
)

depth_source_before = (
    depth_module_path.read_text(
        encoding="utf-8"
    )
)


# =====================================================================
# Production DVL source
# =====================================================================

dvl_module_source = '''"""Doppler velocity log sensor models for HydroLab-3D."""

import numpy as np

from hydrolab3d.sensors.sensor import (
    measure_with_bias_noise,
)
from hydrolab3d.world.currents import (
    relative_velocity_6dof,
)


def ideal_dvl_measurement(
    pose,
    body_velocity,
    current_velocity_world,
):
    """
    Return ideal body-frame water-relative translational velocity.

    Parameters
    ----------
    pose : array-like
        Pose [x, y, z, roll, pitch, yaw].

    body_velocity : array-like
        Generalized body velocity [u, v, w, p, q, r].

    current_velocity_world : array-like
        World-frame ocean-current velocity [vc_x, vc_y, vc_z].

    Returns
    -------
    np.ndarray
        Body-frame translational velocity relative to the water,
        [u_r, v_r, w_r], in m/s.
    """
    pose = np.asarray(
        pose,
        dtype=float,
    )

    body_velocity = np.asarray(
        body_velocity,
        dtype=float,
    )

    current_velocity_world = np.asarray(
        current_velocity_world,
        dtype=float,
    )

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if body_velocity.shape != (6,):
        raise ValueError(
            "body_velocity must have shape (6,)"
        )

    if current_velocity_world.shape != (3,):
        raise ValueError(
            "current_velocity_world must have shape (3,)"
        )

    if not np.all(np.isfinite(pose)):
        raise ValueError(
            "pose must contain only finite values"
        )

    if not np.all(
        np.isfinite(body_velocity)
    ):
        raise ValueError(
            "body_velocity must contain only finite values"
        )

    if not np.all(
        np.isfinite(current_velocity_world)
    ):
        raise ValueError(
            "current_velocity_world must contain only finite values"
        )

    relative_velocity = relative_velocity_6dof(
        pose,
        body_velocity,
        current_velocity_world,
    )

    relative_velocity = np.asarray(
        relative_velocity,
        dtype=float,
    )

    if relative_velocity.shape != (6,):
        raise RuntimeError(
            "relative_velocity_6dof returned unexpected shape"
        )

    return relative_velocity[:3].copy()


def dvl_measurement(
    pose,
    body_velocity,
    current_velocity_world,
    *,
    bias=0.0,
    noise_std=0.0,
    rng=None,
):
    """
    Return imperfect DVL water-relative velocity measurement.
    """
    ideal = ideal_dvl_measurement(
        pose,
        body_velocity,
        current_velocity_world,
    )

    measured = measure_with_bias_noise(
        ideal,
        bias=bias,
        noise_std=noise_std,
        rng=rng,
    )

    if measured.shape != (3,):
        raise ValueError(
            "DVL bias and noise_std must be "
            "scalar or broadcast-compatible with shape (3,)"
        )

    return measured
'''


# =====================================================================
# Write dvl.py
# =====================================================================

dvl_module_path.write_text(
    dvl_module_source,
    encoding="utf-8",
)


# =====================================================================
# Verify exact write
# =====================================================================

if not dvl_module_path.exists():
    raise AssertionError(
        "dvl.py was not created successfully"
    )

written_dvl_source = dvl_module_path.read_text(
    encoding="utf-8"
)

if written_dvl_source != dvl_module_source:
    raise AssertionError(
        "Written dvl.py does not match intended source"
    )


# =====================================================================
# Verify previously promoted files remain unchanged
# =====================================================================

if (
    sensor_module_path.read_text(
        encoding="utf-8"
    )
    != sensor_source_before
):
    raise AssertionError(
        "sensor.py changed unexpectedly during DVL promotion"
    )

if (
    imu_module_path.read_text(
        encoding="utf-8"
    )
    != imu_source_before
):
    raise AssertionError(
        "imu.py changed unexpectedly during DVL promotion"
    )

if (
    depth_module_path.read_text(
        encoding="utf-8"
    )
    != depth_source_before
):
    raise AssertionError(
        "depth.py changed unexpectedly during DVL promotion"
    )


# =====================================================================
# Remaining placeholders must stay untouched
# =====================================================================

for path in [
    sensor_dir / "__init__.py",
    sensor_dir / "sonar.py",
]:
    if path.read_text(
        encoding="utf-8"
    ).strip():
        raise AssertionError(
            f"Unexpected modification detected in "
            f"{path.relative_to(repo_root)}"
        )


# =====================================================================
# Report
# =====================================================================

print("STEP 25 PROMOTION WRITE PASSED")
print()

print("Promoted file:")
print(
    dvl_module_path.relative_to(
        repo_root
    )
)

print()

print("Written source:")
print("-" * 80)
print(written_dvl_source)
print("-" * 80)

print()

print("sensor.py unchanged: True")
print("imu.py unchanged: True")
print("depth.py unchanged: True")
print("Remaining placeholders untouched: True")

STEP 25 PROMOTION WRITE PASSED

Promoted file:
src/hydrolab3d/sensors/dvl.py

Written source:
--------------------------------------------------------------------------------
"""Doppler velocity log sensor models for HydroLab-3D."""

import numpy as np

from hydrolab3d.sensors.sensor import (
    measure_with_bias_noise,
)
from hydrolab3d.world.currents import (
    relative_velocity_6dof,
)


def ideal_dvl_measurement(
    pose,
    body_velocity,
    current_velocity_world,
):
    """
    Return ideal body-frame water-relative translational velocity.

    Parameters
    ----------
    pose : array-like
        Pose [x, y, z, roll, pitch, yaw].

    body_velocity : array-like
        Generalized body velocity [u, v, w, p, q, r].

    current_velocity_world : array-like
        World-frame ocean-current velocity [vc_x, vc_y, vc_z].

    Returns
    -------
    np.ndarray
        Body-frame translational velocity relative to the water,
        [u_r, v_r, w_r], in m/s.
    """
    pose =

## Step 26 - Re-import and Validate the Production DVL

The validated DVL implementation has now been promoted into:

```text
src/hydrolab3d/sensors/dvl.py
```

We now re-import the production functions and compare them against the already validated notebook-local DVL implementation.

The validation covers:

- source-file resolution;
- ideal zero-current behavior;
- arbitrary-orientation current transformation;
- co-moving vehicle/current behavior;
- agreement with `relative_velocity_6dof`;
- deterministic bias;
- seeded Gaussian noise;
- rotational-rate independence;
- malformed and non-finite inputs.

No additional production sensor file is modified in this step.

### Validation Criteria

This step passes only if:

1. both production DVL functions resolve to `src/hydrolab3d/sensors/dvl.py`;
2. production ideal DVL matches the notebook-local implementation;
3. production DVL matches `relative_velocity_6dof(...)[0:3]`;
4. a vehicle moving exactly with the water reports zero relative translational velocity;
5. rotational-rate changes do not alter the translational DVL output;
6. deterministic bias matches exactly;
7. seeded stochastic output matches exactly under identical seeds;
8. invalid input behavior remains consistent.

In [33]:
# ---------------------------------------------------------------------
# Step 26 - Re-import and independently validate production DVL
# ---------------------------------------------------------------------

importlib.invalidate_caches()

from hydrolab3d.sensors.dvl import (
    ideal_dvl_measurement as production_ideal_dvl_measurement,
    dvl_measurement as production_dvl_measurement,
)


# =====================================================================
# Verify production source locations
# =====================================================================

expected_dvl_source = dvl_module_path.resolve()

production_dvl_functions = {
    "ideal_dvl_measurement":
        production_ideal_dvl_measurement,

    "dvl_measurement":
        production_dvl_measurement,
}


for name, function in production_dvl_functions.items():

    source = Path(
        inspect.getsourcefile(function)
        or inspect.getfile(function)
    ).resolve()

    if source != expected_dvl_source:
        raise AssertionError(
            f"{name} resolved to {source}, "
            f"expected {expected_dvl_source}"
        )


# =====================================================================
# Validation 1: ideal DVL parity
# =====================================================================

eta_dvl_test = np.array([
    3.0,
    -2.0,
    -15.0,
    0.22,
    -0.17,
    0.61,
])

nu_dvl_test = np.array([
    1.35,
    -0.42,
    0.28,
    0.08,
    -0.06,
    0.11,
])

current_dvl_test = np.array([
    0.30,
    -0.18,
    0.09,
])

local_dvl_ideal = ideal_dvl_measurement(
    eta_dvl_test,
    nu_dvl_test,
    current_dvl_test,
)

production_dvl_ideal = (
    production_ideal_dvl_measurement(
        eta_dvl_test,
        nu_dvl_test,
        current_dvl_test,
    )
)

np.testing.assert_allclose(
    production_dvl_ideal,
    local_dvl_ideal,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 2: direct production relative-velocity agreement
# =====================================================================

relative_nu = relative_velocity_6dof(
    eta_dvl_test,
    nu_dvl_test,
    current_dvl_test,
)

relative_nu = np.asarray(
    relative_nu,
    dtype=float,
)

np.testing.assert_allclose(
    production_dvl_ideal,
    relative_nu[:3],
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 3: zero-current behavior
# =====================================================================

production_zero_current = (
    production_ideal_dvl_measurement(
        eta_dvl_test,
        nu_dvl_test,
        np.zeros(3),
    )
)

np.testing.assert_allclose(
    production_zero_current,
    nu_dvl_test[:3],
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 4: co-moving vehicle/current
# =====================================================================

current_body = current_velocity_body(
    eta_dvl_test,
    current_dvl_test,
)

nu_comoving = np.array([
    current_body[0],
    current_body[1],
    current_body[2],
    0.12,
    -0.04,
    0.09,
])

production_comoving = (
    production_ideal_dvl_measurement(
        eta_dvl_test,
        nu_comoving,
        current_dvl_test,
    )
)

np.testing.assert_allclose(
    production_comoving,
    np.zeros(3),
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 5: rotational-rate independence
# =====================================================================

nu_rotated_rates = nu_dvl_test.copy()

nu_rotated_rates[3:] = np.array([
    -2.0,
    1.5,
    0.7,
])

production_rotated_rates = (
    production_ideal_dvl_measurement(
        eta_dvl_test,
        nu_rotated_rates,
        current_dvl_test,
    )
)

np.testing.assert_allclose(
    production_rotated_rates,
    production_dvl_ideal,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 6: deterministic bias parity
# =====================================================================

dvl_bias_test = np.array([
    0.01,
    -0.015,
    0.02,
])

local_dvl_biased = dvl_measurement(
    eta_dvl_test,
    nu_dvl_test,
    current_dvl_test,
    bias=dvl_bias_test,
    noise_std=0.0,
)

production_dvl_biased = (
    production_dvl_measurement(
        eta_dvl_test,
        nu_dvl_test,
        current_dvl_test,
        bias=dvl_bias_test,
        noise_std=0.0,
    )
)

np.testing.assert_array_equal(
    production_dvl_biased,
    local_dvl_biased,
)


# =====================================================================
# Validation 7: seeded stochastic parity
# =====================================================================

dvl_noise_test = np.array([
    0.008,
    0.010,
    0.012,
])

local_dvl_rng = np.random.default_rng(
    26001
)

production_dvl_rng = np.random.default_rng(
    26001
)

local_dvl_noisy = dvl_measurement(
    eta_dvl_test,
    nu_dvl_test,
    current_dvl_test,
    bias=dvl_bias_test,
    noise_std=dvl_noise_test,
    rng=local_dvl_rng,
)

production_dvl_noisy = (
    production_dvl_measurement(
        eta_dvl_test,
        nu_dvl_test,
        current_dvl_test,
        bias=dvl_bias_test,
        noise_std=dvl_noise_test,
        rng=production_dvl_rng,
    )
)

np.testing.assert_array_equal(
    production_dvl_noisy,
    local_dvl_noisy,
)


# =====================================================================
# Validation 8: invalid input behavior
# =====================================================================

invalid_pose_cases = [
    np.zeros(5),
    np.zeros((6, 1)),
    np.array([
        0.0,
        0.0,
        -1.0,
        0.0,
        np.nan,
        0.0,
    ]),
]

for bad_pose in invalid_pose_cases:
    try:
        production_ideal_dvl_measurement(
            bad_pose,
            nu_dvl_test,
            current_dvl_test,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Production DVL accepted invalid pose"
        )


invalid_velocity_cases = [
    np.zeros(5),
    np.zeros((6, 1)),
    np.array([
        0.0,
        0.0,
        0.0,
        0.0,
        np.inf,
        0.0,
    ]),
]

for bad_nu in invalid_velocity_cases:
    try:
        production_ideal_dvl_measurement(
            eta_dvl_test,
            bad_nu,
            current_dvl_test,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Production DVL accepted invalid body velocity"
        )


invalid_current_cases = [
    np.zeros(2),
    np.zeros((3, 1)),
    np.array([
        0.0,
        np.nan,
        0.0,
    ]),
]

for bad_current in invalid_current_cases:
    try:
        production_ideal_dvl_measurement(
            eta_dvl_test,
            nu_dvl_test,
            bad_current,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Production DVL accepted invalid current"
        )


try:
    production_dvl_measurement(
        eta_dvl_test,
        nu_dvl_test,
        current_dvl_test,
        bias=np.array([
            0.1,
            0.2,
        ]),
        noise_std=0.0,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Production DVL accepted incompatible bias shape"
    )


try:
    production_dvl_measurement(
        eta_dvl_test,
        nu_dvl_test,
        current_dvl_test,
        noise_std=-0.01,
        rng=np.random.default_rng(1),
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Production DVL accepted negative noise_std"
    )


try:
    production_dvl_measurement(
        eta_dvl_test,
        nu_dvl_test,
        current_dvl_test,
        noise_std=0.01,
        rng="not-a-generator",
    )
except TypeError:
    pass
else:
    raise AssertionError(
        "Production DVL accepted invalid RNG"
    )


# =====================================================================
# Report
# =====================================================================

print("STEP 26 VALIDATION PASSED")
print()

print("Production DVL source:")
print(
    expected_dvl_source.relative_to(
        repo_root
    )
)

print()

print("Ideal DVL parity:")
print(
    np.allclose(
        production_dvl_ideal,
        local_dvl_ideal,
        rtol=0.0,
        atol=1e-12,
    )
)

print()

print("Production ideal DVL [m/s]:")
print(
    production_dvl_ideal
)

print()

print("Zero-current DVL [m/s]:")
print(
    production_zero_current
)

print()

print("Co-moving DVL [m/s]:")
print(
    production_comoving
)

print()

print("Rotational-rate independence:")
print(
    np.allclose(
        production_rotated_rates,
        production_dvl_ideal,
        rtol=0.0,
        atol=1e-12,
    )
)

print()

print("Biased parity:")
print(
    np.array_equal(
        production_dvl_biased,
        local_dvl_biased,
    )
)

print()

print("Seeded stochastic parity:")
print(
    np.array_equal(
        production_dvl_noisy,
        local_dvl_noisy,
    )
)

print()

print("Production noisy DVL [m/s]:")
print(
    production_dvl_noisy
)

print()

print("Invalid-input behavior validated: True")

STEP 26 VALIDATION PASSED

Production DVL source:
src/hydrolab3d/sensors/dvl.py

Ideal DVL parity:
True

Production ideal DVL [m/s]:
[ 1.19405351 -0.12238748  0.14730682]

Zero-current DVL [m/s]:
[ 1.35 -0.42  0.28]

Co-moving DVL [m/s]:
[0. 0. 0.]

Rotational-rate independence:
True

Biased parity:
True

Seeded stochastic parity:
True

Production noisy DVL [m/s]:
[ 1.19754429 -0.1465738   0.16021748]

Invalid-input behavior validated: True


## Step 27 - Promote the Heading Sensor

The notebook-local heading model has already been validated for:

- yaw extraction from the HydroLab-3D pose;
- deterministic wrapping to $(-\pi,\pi]$;
- exact behavior at $\pm\pi$ and odd multiples of $\pi$;
- deterministic bias;
- seeded Gaussian noise;
- final re-wrapping after bias and noise;
- independence from position, roll, and pitch.

We now promote this functionality into a dedicated production module:

```text
src/hydrolab3d/sensors/heading.py
```

The production module will contain:

```text
wrap_angle_pi
ideal_heading_measurement
heading_measurement
```

The ideal heading measurement is:

$$
\psi_h
=
\operatorname{wrap}_{(-\pi,\pi]}(\psi).
$$

The imperfect measurement is:

$$
\tilde{\psi}_h
=
\operatorname{wrap}_{(-\pi,\pi]}
\left(
\psi_h
+
b_h
+
n_h
\right).
$$

This remains a lightweight yaw/heading sensor model.

It does not model:

- Earth's magnetic field;
- magnetic declination;
- hard-iron or soft-iron effects;
- tilt-compensated magnetometer electronics.

### Validation Criteria

This step passes only if:

1. `heading.py` is written successfully;
2. the production source preserves the validated notebook behavior;
3. `sensor.py`, `imu.py`, `depth.py`, and `dvl.py` remain unchanged;
4. `sonar.py` and `sensors/__init__.py` remain untouched;
5. production import validation is deferred until the next step.

In [34]:
# ---------------------------------------------------------------------
# Step 27 - Promote validated heading model into production
# ---------------------------------------------------------------------

heading_module_path = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "sensors"
    / "heading.py"
)


# =====================================================================
# Preserve already-promoted production files
# =====================================================================

sensor_source_before = (
    sensor_module_path.read_text(
        encoding="utf-8"
    )
)

imu_source_before = (
    imu_module_path.read_text(
        encoding="utf-8"
    )
)

depth_source_before = (
    depth_module_path.read_text(
        encoding="utf-8"
    )
)

dvl_source_before = (
    dvl_module_path.read_text(
        encoding="utf-8"
    )
)


# =====================================================================
# Production heading source
# =====================================================================

heading_module_source = '''"""Heading sensor models for HydroLab-3D."""

import numpy as np

from hydrolab3d.sensors.sensor import (
    measure_with_bias_noise,
)


def wrap_angle_pi(
    angle,
):
    """
    Wrap an angle in radians to the interval (-pi, pi].
    """
    angle = float(angle)

    if not np.isfinite(angle):
        raise ValueError(
            "angle must be finite"
        )

    wrapped = (
        (angle + np.pi)
        % (2.0 * np.pi)
        - np.pi
    )

    if np.isclose(
        wrapped,
        -np.pi,
        rtol=0.0,
        atol=1e-15,
    ):
        wrapped = np.pi

    return float(wrapped)


def ideal_heading_measurement(
    pose,
):
    """
    Return ideal wrapped yaw heading.

    Parameters
    ----------
    pose : array-like
        Pose [x, y, z, roll, pitch, yaw].

    Returns
    -------
    float
        Heading in radians, wrapped to (-pi, pi].
    """
    pose = np.asarray(
        pose,
        dtype=float,
    )

    if pose.shape != (6,):
        raise ValueError(
            "pose must have shape (6,)"
        )

    if not np.all(np.isfinite(pose)):
        raise ValueError(
            "pose must contain only finite values"
        )

    return wrap_angle_pi(
        pose[5]
    )


def heading_measurement(
    pose,
    *,
    bias=0.0,
    noise_std=0.0,
    rng=None,
):
    """
    Return imperfect wrapped heading measurement.

    Parameters
    ----------
    pose : array-like
        Pose [x, y, z, roll, pitch, yaw].

    bias : scalar, optional
        Additive heading bias in radians.

    noise_std : scalar, optional
        Gaussian heading-noise standard deviation in radians.

    rng : numpy.random.Generator or None, optional
        Explicit random generator when noise is nonzero.

    Returns
    -------
    float
        Heading measurement in radians, wrapped to (-pi, pi].
    """
    ideal = ideal_heading_measurement(
        pose
    )

    measured = measure_with_bias_noise(
        ideal,
        bias=bias,
        noise_std=noise_std,
        rng=rng,
    )

    measured = np.asarray(
        measured,
        dtype=float,
    )

    if measured.shape != ():
        raise ValueError(
            "heading bias and noise_std must be scalar"
        )

    return wrap_angle_pi(
        float(measured)
    )
'''


# =====================================================================
# Write heading.py
# =====================================================================

heading_module_path.write_text(
    heading_module_source,
    encoding="utf-8",
)


# =====================================================================
# Verify exact write
# =====================================================================

if not heading_module_path.exists():
    raise AssertionError(
        "heading.py was not created successfully"
    )

written_heading_source = (
    heading_module_path.read_text(
        encoding="utf-8"
    )
)

if written_heading_source != heading_module_source:
    raise AssertionError(
        "Written heading.py does not match intended source"
    )


# =====================================================================
# Verify previously promoted files remain unchanged
# =====================================================================

if (
    sensor_module_path.read_text(
        encoding="utf-8"
    )
    != sensor_source_before
):
    raise AssertionError(
        "sensor.py changed unexpectedly during heading promotion"
    )

if (
    imu_module_path.read_text(
        encoding="utf-8"
    )
    != imu_source_before
):
    raise AssertionError(
        "imu.py changed unexpectedly during heading promotion"
    )

if (
    depth_module_path.read_text(
        encoding="utf-8"
    )
    != depth_source_before
):
    raise AssertionError(
        "depth.py changed unexpectedly during heading promotion"
    )

if (
    dvl_module_path.read_text(
        encoding="utf-8"
    )
    != dvl_source_before
):
    raise AssertionError(
        "dvl.py changed unexpectedly during heading promotion"
    )


# =====================================================================
# Existing untouched files must remain untouched
# =====================================================================

for path in [
    sensor_dir / "__init__.py",
    sensor_dir / "sonar.py",
]:
    if path.read_text(
        encoding="utf-8"
    ).strip():
        raise AssertionError(
            f"Unexpected modification detected in "
            f"{path.relative_to(repo_root)}"
        )


# =====================================================================
# Report
# =====================================================================

print("STEP 27 PROMOTION WRITE PASSED")
print()

print("Promoted file:")
print(
    heading_module_path.relative_to(
        repo_root
    )
)

print()

print("Written source:")
print("-" * 80)
print(written_heading_source)
print("-" * 80)

print()

print("sensor.py unchanged: True")
print("imu.py unchanged: True")
print("depth.py unchanged: True")
print("dvl.py unchanged: True")
print("sonar.py untouched: True")
print("sensors/__init__.py untouched: True")

STEP 27 PROMOTION WRITE PASSED

Promoted file:
src/hydrolab3d/sensors/heading.py

Written source:
--------------------------------------------------------------------------------
"""Heading sensor models for HydroLab-3D."""

import numpy as np

from hydrolab3d.sensors.sensor import (
    measure_with_bias_noise,
)


def wrap_angle_pi(
    angle,
):
    """
    Wrap an angle in radians to the interval (-pi, pi].
    """
    angle = float(angle)

    if not np.isfinite(angle):
        raise ValueError(
            "angle must be finite"
        )

    wrapped = (
        (angle + np.pi)
        % (2.0 * np.pi)
        - np.pi
    )

    if np.isclose(
        wrapped,
        -np.pi,
        rtol=0.0,
        atol=1e-15,
    ):
        wrapped = np.pi

    return float(wrapped)


def ideal_heading_measurement(
    pose,
):
    """
    Return ideal wrapped yaw heading.

    Parameters
    ----------
    pose : array-like
        Pose [x, y, z, roll, pitch, yaw].

    Returns
    -------
 

## Step 28 - Re-import and Validate the Production Heading Sensor

The validated heading model has now been promoted into:

```text
src/hydrolab3d/sensors/heading.py
```

We now re-import the production functions and compare them against the already validated notebook-local heading implementation.

The validation covers:

- source-file resolution;
- angle wrapping to $(-\pi,\pi]$;
- exact boundary behavior at $\pm\pi$ and odd multiples of $\pi$;
- ideal heading parity;
- deterministic bias parity;
- seeded Gaussian noise parity;
- final wrapping after bias/noise;
- independence from position, roll, and pitch;
- malformed and non-finite inputs.

No additional production sensor file is modified in this step.

### Validation Criteria

This step passes only if:

1. all three production heading functions resolve to `src/hydrolab3d/sensors/heading.py`;
2. production angle wrapping matches the notebook-local implementation;
3. ideal heading parity holds;
4. deterministic bias parity holds;
5. seeded stochastic parity holds;
6. final noisy outputs remain inside $(-\pi,\pi]$;
7. non-yaw pose components do not affect heading;
8. invalid inputs are rejected consistently.

In [35]:
# ---------------------------------------------------------------------
# Step 28 - Re-import and independently validate production heading
# ---------------------------------------------------------------------

importlib.invalidate_caches()

from hydrolab3d.sensors.heading import (
    wrap_angle_pi as production_wrap_angle_pi,
    ideal_heading_measurement as production_ideal_heading_measurement,
    heading_measurement as production_heading_measurement,
)


# =====================================================================
# Verify production source locations
# =====================================================================

expected_heading_source = heading_module_path.resolve()

production_heading_functions = {
    "wrap_angle_pi":
        production_wrap_angle_pi,

    "ideal_heading_measurement":
        production_ideal_heading_measurement,

    "heading_measurement":
        production_heading_measurement,
}


for name, function in production_heading_functions.items():

    source = Path(
        inspect.getsourcefile(function)
        or inspect.getfile(function)
    ).resolve()

    if source != expected_heading_source:
        raise AssertionError(
            f"{name} resolved to {source}, "
            f"expected {expected_heading_source}"
        )


# =====================================================================
# Validation 1: wrapping parity
# =====================================================================

test_angles = [
    0.0,
    1.2,
    -0.75,
    np.pi,
    -np.pi,
    3.0 * np.pi,
    -3.0 * np.pi,
    1.5 * np.pi,
    -1.5 * np.pi,
    7.4,
    -8.1,
]

for angle in test_angles:

    local_wrapped = wrap_angle_pi(
        angle
    )

    production_wrapped = (
        production_wrap_angle_pi(
            angle
        )
    )

    np.testing.assert_allclose(
        production_wrapped,
        local_wrapped,
        rtol=0.0,
        atol=1e-15,
    )


# =====================================================================
# Validation 2: ideal heading parity
# =====================================================================

eta_heading_test = np.array([
    7.0,
    -4.0,
    -11.0,
    0.31,
    -0.24,
    2.85,
])

local_heading_ideal = (
    ideal_heading_measurement(
        eta_heading_test
    )
)

production_heading_ideal = (
    production_ideal_heading_measurement(
        eta_heading_test
    )
)

np.testing.assert_allclose(
    production_heading_ideal,
    local_heading_ideal,
    rtol=0.0,
    atol=1e-15,
)


# =====================================================================
# Validation 3: exact boundary convention
# =====================================================================

boundary_cases = {
    "pi":
        np.pi,

    "-pi":
        -np.pi,

    "3pi":
        3.0 * np.pi,

    "-3pi":
        -3.0 * np.pi,
}

production_boundary_results = {}

for name, angle in boundary_cases.items():

    value = production_wrap_angle_pi(
        angle
    )

    production_boundary_results[name] = value

    np.testing.assert_allclose(
        value,
        np.pi,
        rtol=0.0,
        atol=1e-15,
    )


# =====================================================================
# Validation 4: deterministic bias parity
# =====================================================================

heading_bias_test = 0.18

local_heading_biased = heading_measurement(
    eta_heading_test,
    bias=heading_bias_test,
    noise_std=0.0,
)

production_heading_biased = (
    production_heading_measurement(
        eta_heading_test,
        bias=heading_bias_test,
        noise_std=0.0,
    )
)

np.testing.assert_allclose(
    production_heading_biased,
    local_heading_biased,
    rtol=0.0,
    atol=1e-15,
)


# =====================================================================
# Validation 5: bias crossing +pi
# =====================================================================

eta_heading_near_pi = np.array([
    0.0,
    0.0,
    -3.0,
    0.0,
    0.0,
    np.pi - 0.04,
])

production_heading_crossed = (
    production_heading_measurement(
        eta_heading_near_pi,
        bias=0.15,
        noise_std=0.0,
    )
)

expected_crossed = production_wrap_angle_pi(
    (np.pi - 0.04)
    + 0.15
)

np.testing.assert_allclose(
    production_heading_crossed,
    expected_crossed,
    rtol=0.0,
    atol=1e-15,
)


# =====================================================================
# Validation 6: seeded stochastic parity
# =====================================================================

local_heading_rng = np.random.default_rng(
    28001
)

production_heading_rng = np.random.default_rng(
    28001
)

local_heading_noisy = heading_measurement(
    eta_heading_test,
    bias=0.02,
    noise_std=0.01,
    rng=local_heading_rng,
)

production_heading_noisy = (
    production_heading_measurement(
        eta_heading_test,
        bias=0.02,
        noise_std=0.01,
        rng=production_heading_rng,
    )
)

np.testing.assert_allclose(
    production_heading_noisy,
    local_heading_noisy,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 7: final noisy outputs remain wrapped
# =====================================================================

range_rng = np.random.default_rng(
    28002
)

range_values = []

for _ in range(10_000):

    value = production_heading_measurement(
        eta_heading_near_pi,
        bias=0.20,
        noise_std=0.30,
        rng=range_rng,
    )

    range_values.append(
        value
    )

range_values = np.asarray(
    range_values,
    dtype=float,
)

if not np.all(
    (range_values > -np.pi)
    & (range_values <= np.pi)
):
    raise AssertionError(
        "Production heading left (-pi, pi]"
    )


# =====================================================================
# Validation 8: non-yaw pose components do not affect heading
# =====================================================================

eta_heading_a = np.array([
    0.0,
    0.0,
    -2.0,
    0.1,
    -0.2,
    -0.85,
])

eta_heading_b = np.array([
    100.0,
    -70.0,
    -40.0,
    1.1,
    0.7,
    -0.85,
])

heading_a = (
    production_ideal_heading_measurement(
        eta_heading_a
    )
)

heading_b = (
    production_ideal_heading_measurement(
        eta_heading_b
    )
)

np.testing.assert_allclose(
    heading_a,
    heading_b,
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 9: invalid-input behavior
# =====================================================================

invalid_pose_cases = [
    np.zeros(5),
    np.zeros((6, 1)),
    np.array([
        0.0,
        0.0,
        -1.0,
        0.0,
        0.0,
        np.nan,
    ]),
]

for bad_pose in invalid_pose_cases:

    try:
        production_ideal_heading_measurement(
            bad_pose
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Production heading accepted invalid pose"
        )


for bad_angle in [
    np.nan,
    np.inf,
    -np.inf,
]:

    try:
        production_wrap_angle_pi(
            bad_angle
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Production angle wrapper accepted "
            "non-finite input"
        )


try:
    production_heading_measurement(
        eta_heading_test,
        bias=np.array([
            0.1,
            0.2,
        ]),
        noise_std=0.0,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Production heading accepted non-scalar bias"
    )


try:
    production_heading_measurement(
        eta_heading_test,
        noise_std=-0.01,
        rng=np.random.default_rng(1),
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Production heading accepted negative noise_std"
    )


try:
    production_heading_measurement(
        eta_heading_test,
        noise_std=0.01,
        rng="not-a-generator",
    )
except TypeError:
    pass
else:
    raise AssertionError(
        "Production heading accepted invalid RNG"
    )


# =====================================================================
# Report
# =====================================================================

print("STEP 28 VALIDATION PASSED")
print()

print("Production heading source:")
print(
    expected_heading_source.relative_to(
        repo_root
    )
)

print()

print("Ideal heading parity:")
print(
    production_heading_ideal
    == local_heading_ideal
)

print()

print("Boundary results:")
for name, value in production_boundary_results.items():
    print(
        f"  {name:<4} -> {value}"
    )

print()

print("Biased heading parity:")
print(
    production_heading_biased
    == local_heading_biased
)

print()

print("Boundary-crossing heading [rad]:")
print(
    production_heading_crossed
)

print()

print("Seeded stochastic parity:")
print(
    production_heading_noisy
    == local_heading_noisy
)

print()

print("Production noisy heading [rad]:")
print(
    production_heading_noisy
)

print()

print("Stress-test range [rad]:")
print(
    np.min(range_values),
    np.max(range_values),
)

print()

print(
    "Non-yaw pose-component independence: True"
)

print(
    "Invalid-input behavior validated: True"
)

STEP 28 VALIDATION PASSED

Production heading source:
src/hydrolab3d/sensors/heading.py

Ideal heading parity:
True

Boundary results:
  pi   -> 3.141592653589793
  -pi  -> 3.141592653589793
  3pi  -> 3.141592653589793
  -3pi -> 3.141592653589793

Biased heading parity:
True

Boundary-crossing heading [rad]:
-3.0315926535897937

Seeded stochastic parity:
True

Production noisy heading [rad]:
2.872725223328348

Stress-test range [rad]:
-3.141479409145335 3.1415577434225774

Non-yaw pose-component independence: True
Invalid-input behavior validated: True


## Step 29 - Define the Public Sensor Package API

The individual production sensor modules have now been validated.

We next expose the intended public sensor API through:

```text
src/hydrolab3d/sensors/__init__.py
```

This follows the existing HydroLab-3D package pattern used by modules such as:

```text
hydrolab3d.dynamics
hydrolab3d.world
hydrolab3d.vehicles
```

The public sensor API will export:

```text
measure_with_bias_noise

ideal_gyroscope_measurement
gyroscope_measurement

ideal_accelerometer_measurement
accelerometer_measurement

ideal_depth_measurement
depth_measurement

ideal_dvl_measurement
dvl_measurement

wrap_angle_pi
ideal_heading_measurement
heading_measurement
```

`sonar.py` remains intentionally unexported because sonar has not been implemented or validated in Notebook 08.

### Validation Criteria

This step passes only if:

1. `sensors/__init__.py` is written successfully;
2. all validated sensor functions are exported;
3. `sonar.py` is not exported;
4. all previously validated production sensor modules remain unchanged;
5. package import validation is deferred until the next step.

In [36]:
# ---------------------------------------------------------------------
# Step 29 - Define public sensor package API
# ---------------------------------------------------------------------

sensor_init_path = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "sensors"
    / "__init__.py"
)


# =====================================================================
# Preserve validated production modules
# =====================================================================

validated_sensor_sources_before = {
    "sensor.py":
        sensor_module_path.read_text(
            encoding="utf-8"
        ),

    "imu.py":
        imu_module_path.read_text(
            encoding="utf-8"
        ),

    "depth.py":
        depth_module_path.read_text(
            encoding="utf-8"
        ),

    "dvl.py":
        dvl_module_path.read_text(
            encoding="utf-8"
        ),

    "heading.py":
        heading_module_path.read_text(
            encoding="utf-8"
        ),
}


# =====================================================================
# Public sensor API source
# =====================================================================

sensor_init_source = '''"""Sensor models for HydroLab-3D."""

from .depth import (
    depth_measurement,
    ideal_depth_measurement,
)
from .dvl import (
    dvl_measurement,
    ideal_dvl_measurement,
)
from .heading import (
    heading_measurement,
    ideal_heading_measurement,
    wrap_angle_pi,
)
from .imu import (
    accelerometer_measurement,
    gyroscope_measurement,
    ideal_accelerometer_measurement,
    ideal_gyroscope_measurement,
)
from .sensor import measure_with_bias_noise


__all__ = [
    "accelerometer_measurement",
    "depth_measurement",
    "dvl_measurement",
    "gyroscope_measurement",
    "heading_measurement",
    "ideal_accelerometer_measurement",
    "ideal_depth_measurement",
    "ideal_dvl_measurement",
    "ideal_gyroscope_measurement",
    "ideal_heading_measurement",
    "measure_with_bias_noise",
    "wrap_angle_pi",
]
'''


# =====================================================================
# Write sensors/__init__.py
# =====================================================================

sensor_init_path.write_text(
    sensor_init_source,
    encoding="utf-8",
)


# =====================================================================
# Verify exact write
# =====================================================================

if not sensor_init_path.exists():
    raise AssertionError(
        "sensors/__init__.py was not written successfully"
    )

written_sensor_init = (
    sensor_init_path.read_text(
        encoding="utf-8"
    )
)

if written_sensor_init != sensor_init_source:
    raise AssertionError(
        "Written sensors/__init__.py does not match intended source"
    )


# =====================================================================
# Verify validated modules remain unchanged
# =====================================================================

validated_paths = {
    "sensor.py":
        sensor_module_path,

    "imu.py":
        imu_module_path,

    "depth.py":
        depth_module_path,

    "dvl.py":
        dvl_module_path,

    "heading.py":
        heading_module_path,
}


for name, path in validated_paths.items():

    current_source = path.read_text(
        encoding="utf-8"
    )

    if current_source != validated_sensor_sources_before[name]:
        raise AssertionError(
            f"{name} changed unexpectedly while "
            "updating sensors/__init__.py"
        )


# =====================================================================
# Sonar must remain untouched and unimplemented
# =====================================================================

sonar_path = (
    sensor_dir
    / "sonar.py"
)

if sonar_path.read_text(
    encoding="utf-8"
).strip():
    raise AssertionError(
        "sonar.py changed unexpectedly"
    )

if "sonar" in written_sensor_init.lower():
    raise AssertionError(
        "Unvalidated sonar functionality was exported"
    )


# =====================================================================
# Report
# =====================================================================

print("STEP 29 PACKAGE EXPORT WRITE PASSED")
print()

print("Updated file:")
print(
    sensor_init_path.relative_to(
        repo_root
    )
)

print()

print("Written source:")
print("-" * 80)
print(written_sensor_init)
print("-" * 80)

print()

print(
    "Validated sensor modules unchanged: True"
)

print(
    "sonar.py untouched and unexported: True"
)

STEP 29 PACKAGE EXPORT WRITE PASSED

Updated file:
src/hydrolab3d/sensors/__init__.py

Written source:
--------------------------------------------------------------------------------
"""Sensor models for HydroLab-3D."""

from .depth import (
    depth_measurement,
    ideal_depth_measurement,
)
from .dvl import (
    dvl_measurement,
    ideal_dvl_measurement,
)
from .heading import (
    heading_measurement,
    ideal_heading_measurement,
    wrap_angle_pi,
)
from .imu import (
    accelerometer_measurement,
    gyroscope_measurement,
    ideal_accelerometer_measurement,
    ideal_gyroscope_measurement,
)
from .sensor import measure_with_bias_noise


__all__ = [
    "accelerometer_measurement",
    "depth_measurement",
    "dvl_measurement",
    "gyroscope_measurement",
    "heading_measurement",
    "ideal_accelerometer_measurement",
    "ideal_depth_measurement",
    "ideal_dvl_measurement",
    "ideal_gyroscope_measurement",
    "ideal_heading_measurement",
    "measure_with_bias_

## Step 30 - Validate the Public Sensor Package API

The validated sensor modules are now exported through:

```text
hydrolab3d.sensors
```

We must verify that the package-level API behaves exactly as intended.

This step checks:

- package import success;
- exact `__all__` contents;
- every public symbol is available;
- every exported function resolves to the correct production module;
- package-level calls reproduce the already validated sensor behavior;
- unvalidated sonar functionality remains absent from the public API.

No production source files are modified in this step.

### Validation Criteria

This step passes only if:

1. `hydrolab3d.sensors` imports successfully;
2. `__all__` exactly matches the intended public API;
3. every exported symbol is callable;
4. exported functions resolve to the correct production source modules;
5. representative package-level sensor calls reproduce validated outputs;
6. no sonar symbol is exposed publicly.

In [39]:
# ---------------------------------------------------------------------
# Step 30 - Reload and validate public sensor package API
# ---------------------------------------------------------------------

import sys
import importlib


# =====================================================================
# Reload package after __init__.py was populated in Step 29
#
# hydrolab3d.sensors may already exist in sys.modules from earlier
# submodule imports performed while its __init__.py was still empty.
# invalidate_caches() alone does not refresh an already-loaded module.
# =====================================================================

importlib.invalidate_caches()

if "hydrolab3d.sensors" in sys.modules:
    production_sensors = importlib.reload(
        sys.modules["hydrolab3d.sensors"]
    )
else:
    production_sensors = importlib.import_module(
        "hydrolab3d.sensors"
    )


# =====================================================================
# Verify package itself resolves to the expected __init__.py
# =====================================================================

package_source = Path(
    production_sensors.__file__
).resolve()

expected_package_source = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "sensors"
    / "__init__.py"
).resolve()

if package_source != expected_package_source:
    raise AssertionError(
        "hydrolab3d.sensors resolved to "
        f"{package_source}, expected "
        f"{expected_package_source}"
    )


# =====================================================================
# Expected public API
# =====================================================================

expected_exports = [
    "accelerometer_measurement",
    "depth_measurement",
    "dvl_measurement",
    "gyroscope_measurement",
    "heading_measurement",
    "ideal_accelerometer_measurement",
    "ideal_depth_measurement",
    "ideal_dvl_measurement",
    "ideal_gyroscope_measurement",
    "ideal_heading_measurement",
    "measure_with_bias_noise",
    "wrap_angle_pi",
]


# =====================================================================
# Validation 1: __all__ now exists
# =====================================================================

if not hasattr(
    production_sensors,
    "__all__",
):
    raise AssertionError(
        "Reloaded hydrolab3d.sensors still has no __all__"
    )


# =====================================================================
# Validation 2: exact public export list
# =====================================================================

if production_sensors.__all__ != expected_exports:
    raise AssertionError(
        "hydrolab3d.sensors.__all__ does not match "
        "the intended public API.\n"
        f"Actual:   {production_sensors.__all__}\n"
        f"Expected: {expected_exports}"
    )


# =====================================================================
# Validation 3: every exported symbol exists and is callable
# =====================================================================

for name in expected_exports:

    if not hasattr(
        production_sensors,
        name,
    ):
        raise AssertionError(
            f"Missing public sensor symbol: {name}"
        )

    value = getattr(
        production_sensors,
        name,
    )

    if not callable(value):
        raise AssertionError(
            f"Public sensor symbol is not callable: {name}"
        )


# =====================================================================
# Validation 4: source-module resolution
# =====================================================================

expected_module_files = {
    "measure_with_bias_noise":
        sensor_module_path.resolve(),

    "ideal_gyroscope_measurement":
        imu_module_path.resolve(),

    "gyroscope_measurement":
        imu_module_path.resolve(),

    "ideal_accelerometer_measurement":
        imu_module_path.resolve(),

    "accelerometer_measurement":
        imu_module_path.resolve(),

    "ideal_depth_measurement":
        depth_module_path.resolve(),

    "depth_measurement":
        depth_module_path.resolve(),

    "ideal_dvl_measurement":
        dvl_module_path.resolve(),

    "dvl_measurement":
        dvl_module_path.resolve(),

    "wrap_angle_pi":
        heading_module_path.resolve(),

    "ideal_heading_measurement":
        heading_module_path.resolve(),

    "heading_measurement":
        heading_module_path.resolve(),
}


for name, expected_path in expected_module_files.items():

    function = getattr(
        production_sensors,
        name,
    )

    actual_path = Path(
        inspect.getsourcefile(function)
        or inspect.getfile(function)
    ).resolve()

    if actual_path != expected_path:
        raise AssertionError(
            f"{name} resolved to {actual_path}, "
            f"expected {expected_path}"
        )


# =====================================================================
# Validation 5: representative ideal package-level calls
# =====================================================================

pkg_gyro = (
    production_sensors.ideal_gyroscope_measurement(
        nu_truth
    )
)

pkg_accel = (
    production_sensors.ideal_accelerometer_measurement(
        eta_truth,
        nu_truth,
        nu_dot_truth,
    )
)

pkg_depth = (
    production_sensors.ideal_depth_measurement(
        eta_truth
    )
)

pkg_dvl = (
    production_sensors.ideal_dvl_measurement(
        eta_truth,
        nu_truth,
        current_world,
    )
)

pkg_heading = (
    production_sensors.ideal_heading_measurement(
        eta_truth
    )
)


np.testing.assert_allclose(
    pkg_gyro,
    gyro_measurement_ideal,
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    pkg_accel,
    accel_measurement_ideal,
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    pkg_depth,
    depth_measurement_ideal,
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    pkg_dvl,
    dvl_measurement_ideal,
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    pkg_heading,
    heading_measurement_ideal,
    rtol=0.0,
    atol=1e-12,
)


# =====================================================================
# Validation 6: representative imperfect package-level call
# =====================================================================

pkg_rng = np.random.default_rng(
    30001
)

pkg_gyro_noisy = (
    production_sensors.gyroscope_measurement(
        nu_truth,
        bias=np.array([
            0.002,
            -0.001,
            0.0015,
        ]),
        noise_std=np.array([
            0.001,
            0.001,
            0.001,
        ]),
        rng=pkg_rng,
    )
)

if not np.all(
    np.isfinite(pkg_gyro_noisy)
):
    raise AssertionError(
        "Package-level noisy gyroscope returned "
        "non-finite values"
    )


# =====================================================================
# Validation 7: package functions agree with directly imported
# production functions
# =====================================================================

np.testing.assert_allclose(
    pkg_gyro,
    production_ideal_gyroscope_measurement(
        nu_truth
    ),
    rtol=0.0,
    atol=0.0,
)

np.testing.assert_allclose(
    pkg_accel,
    production_ideal_accelerometer_measurement(
        eta_truth,
        nu_truth,
        nu_dot_truth,
    ),
    rtol=0.0,
    atol=0.0,
)

np.testing.assert_allclose(
    pkg_depth,
    production_ideal_depth_measurement(
        eta_truth
    ),
    rtol=0.0,
    atol=0.0,
)

np.testing.assert_allclose(
    pkg_dvl,
    production_ideal_dvl_measurement(
        eta_truth,
        nu_truth,
        current_world,
    ),
    rtol=0.0,
    atol=0.0,
)

np.testing.assert_allclose(
    pkg_heading,
    production_ideal_heading_measurement(
        eta_truth
    ),
    rtol=0.0,
    atol=0.0,
)


# =====================================================================
# Validation 8: sonar remains absent from the public API
# =====================================================================

if any(
    "sonar" in name.lower()
    for name in production_sensors.__all__
):
    raise AssertionError(
        "Unvalidated sonar functionality leaked into __all__"
    )

if hasattr(
    production_sensors,
    "sonar_measurement",
):
    raise AssertionError(
        "Unvalidated sonar_measurement is publicly exposed"
    )


# =====================================================================
# Validation 9: source files were not modified during validation
# =====================================================================

if (
    sensor_init_path.read_text(
        encoding="utf-8"
    )
    != sensor_init_source
):
    raise AssertionError(
        "sensors/__init__.py changed unexpectedly"
    )

if (
    sensor_module_path.read_text(
        encoding="utf-8"
    )
    != validated_sensor_sources_before["sensor.py"]
):
    raise AssertionError(
        "sensor.py changed unexpectedly"
    )

if (
    imu_module_path.read_text(
        encoding="utf-8"
    )
    != validated_sensor_sources_before["imu.py"]
):
    raise AssertionError(
        "imu.py changed unexpectedly"
    )

if (
    depth_module_path.read_text(
        encoding="utf-8"
    )
    != validated_sensor_sources_before["depth.py"]
):
    raise AssertionError(
        "depth.py changed unexpectedly"
    )

if (
    dvl_module_path.read_text(
        encoding="utf-8"
    )
    != validated_sensor_sources_before["dvl.py"]
):
    raise AssertionError(
        "dvl.py changed unexpectedly"
    )

if (
    heading_module_path.read_text(
        encoding="utf-8"
    )
    != validated_sensor_sources_before["heading.py"]
):
    raise AssertionError(
        "heading.py changed unexpectedly"
    )


# =====================================================================
# Report
# =====================================================================

print("STEP 30 VALIDATION PASSED")
print()

print("Reloaded package source:")
print(
    package_source.relative_to(
        repo_root
    )
)

print()

print("Public sensor exports:")
for name in production_sensors.__all__:
    print(
        f"  {name}"
    )

print()

print("Package-level ideal gyro:")
print(
    pkg_gyro
)

print()

print("Package-level ideal accelerometer:")
print(
    pkg_accel
)

print()

print("Package-level ideal depth:")
print(
    pkg_depth
)

print()

print("Package-level ideal DVL:")
print(
    pkg_dvl
)

print()

print("Package-level ideal heading:")
print(
    pkg_heading
)

print()

print("Package-level noisy gyro:")
print(
    pkg_gyro_noisy
)

print()

print("Exact __all__ validated: True")
print("All exported source locations validated: True")
print("Direct-module/package API agreement: True")
print("sonar absent from public API: True")
print("Production source files unchanged: True")

STEP 30 VALIDATION PASSED

Reloaded package source:
src/hydrolab3d/sensors/__init__.py

Public sensor exports:
  accelerometer_measurement
  depth_measurement
  dvl_measurement
  gyroscope_measurement
  heading_measurement
  ideal_accelerometer_measurement
  ideal_depth_measurement
  ideal_dvl_measurement
  ideal_gyroscope_measurement
  ideal_heading_measurement
  measure_with_bias_noise
  wrap_angle_pi

Package-level ideal gyro:
[ 0.06 -0.04  0.09]

Package-level ideal accelerometer:
[0.80585661 1.74219452 9.54804481]

Package-level ideal depth:
12.0

Package-level ideal DVL:
[ 0.9343088  -0.14300378  0.15145081]

Package-level ideal heading:
0.5499999999999998

Package-level noisy gyro:
[ 0.06379554 -0.0427776   0.09157119]

Exact __all__ validated: True
All exported source locations validated: True
Direct-module/package API agreement: True
sonar absent from public API: True
Production source files unchanged: True


## Step 31 - Fresh-Process Production Import Check

All intended sensor modules are now promoted and exposed through:

```text
hydrolab3d.sensors
```

Notebook validation has occurred inside a long-lived Python kernel.

Because Python module caching can hide package-state problems, we now launch a completely fresh interpreter and verify the production package without relying on notebook memory.

The fresh interpreter must:

1. import `hydrolab3d.sensors`;
2. inspect the public `__all__`;
3. call representative IMU, depth, DVL, and heading functions;
4. verify seeded stochastic reproducibility;
5. confirm that sonar is not publicly exported.

This step does not modify production source.

### Validation Criteria

This step passes only if the standalone fresh Python process exits successfully and reports all production sensor checks as passed.

In [41]:
# ---------------------------------------------------------------------
# Step 31 - Fresh-process production import validation
# ---------------------------------------------------------------------

import os
import subprocess
import sys
import textwrap


# =====================================================================
# HydroLab-3D uses a src/ package layout.
#
# The notebook kernel already knows about repo_root/src, but a fresh
# subprocess does not necessarily inherit that notebook-specific
# sys.path configuration.
#
# Explicitly expose ONLY the repository src directory to the fresh
# interpreter through PYTHONPATH.
# =====================================================================

src_root = (
    repo_root
    / "src"
).resolve()

if not src_root.exists():
    raise FileNotFoundError(
        f"Expected source directory does not exist: {src_root}"
    )

package_root = (
    src_root
    / "hydrolab3d"
)

if not package_root.exists():
    raise FileNotFoundError(
        f"Expected HydroLab-3D package does not exist: {package_root}"
    )


# =====================================================================
# Fresh-process validation program
# =====================================================================

fresh_process_script = r'''
import inspect
from pathlib import Path

import numpy as np

import hydrolab3d.sensors as sensors


# -------------------------------------------------------------
# Public API
# -------------------------------------------------------------

expected_exports = [
    "accelerometer_measurement",
    "depth_measurement",
    "dvl_measurement",
    "gyroscope_measurement",
    "heading_measurement",
    "ideal_accelerometer_measurement",
    "ideal_depth_measurement",
    "ideal_dvl_measurement",
    "ideal_gyroscope_measurement",
    "ideal_heading_measurement",
    "measure_with_bias_noise",
    "wrap_angle_pi",
]

if not hasattr(
    sensors,
    "__all__",
):
    raise AssertionError(
        "hydrolab3d.sensors has no __all__ in fresh process"
    )

if sensors.__all__ != expected_exports:
    raise AssertionError(
        "Fresh-process sensor exports do not match expected API.\n"
        f"Actual:   {sensors.__all__}\n"
        f"Expected: {expected_exports}"
    )


# -------------------------------------------------------------
# Confirm imports come from production source modules
# -------------------------------------------------------------

expected_suffixes = {
    "measure_with_bias_noise":
        "hydrolab3d/sensors/sensor.py",

    "ideal_gyroscope_measurement":
        "hydrolab3d/sensors/imu.py",

    "gyroscope_measurement":
        "hydrolab3d/sensors/imu.py",

    "ideal_accelerometer_measurement":
        "hydrolab3d/sensors/imu.py",

    "accelerometer_measurement":
        "hydrolab3d/sensors/imu.py",

    "ideal_depth_measurement":
        "hydrolab3d/sensors/depth.py",

    "depth_measurement":
        "hydrolab3d/sensors/depth.py",

    "ideal_dvl_measurement":
        "hydrolab3d/sensors/dvl.py",

    "dvl_measurement":
        "hydrolab3d/sensors/dvl.py",

    "wrap_angle_pi":
        "hydrolab3d/sensors/heading.py",

    "ideal_heading_measurement":
        "hydrolab3d/sensors/heading.py",

    "heading_measurement":
        "hydrolab3d/sensors/heading.py",
}


for name, expected_suffix in expected_suffixes.items():

    function = getattr(
        sensors,
        name,
    )

    source_path = Path(
        inspect.getsourcefile(function)
        or inspect.getfile(function)
    ).resolve()

    normalized = source_path.as_posix()

    if not normalized.endswith(
        expected_suffix
    ):
        raise AssertionError(
            f"{name} resolved to unexpected source: "
            f"{source_path}"
        )


# -------------------------------------------------------------
# Deterministic physical state
# -------------------------------------------------------------

eta = np.array([
    1.0,
    -2.0,
    -10.0,
    0.15,
    -0.10,
    0.45,
])

nu = np.array([
    0.8,
    -0.25,
    0.18,
    0.04,
    -0.03,
    0.07,
])

nu_dot = np.array([
    -0.12,
    0.08,
    -0.04,
    0.01,
    -0.02,
    0.015,
])

current_world = np.array([
    0.20,
    -0.08,
    0.03,
])


# -------------------------------------------------------------
# Ideal sensor calls
# -------------------------------------------------------------

gyro = (
    sensors.ideal_gyroscope_measurement(
        nu
    )
)

accel = (
    sensors.ideal_accelerometer_measurement(
        eta,
        nu,
        nu_dot,
    )
)

depth = (
    sensors.ideal_depth_measurement(
        eta
    )
)

dvl = (
    sensors.ideal_dvl_measurement(
        eta,
        nu,
        current_world,
    )
)

heading = (
    sensors.ideal_heading_measurement(
        eta
    )
)


# -------------------------------------------------------------
# Shape checks
# -------------------------------------------------------------

if gyro.shape != (3,):
    raise AssertionError(
        f"Fresh-process gyro shape failure: {gyro.shape}"
    )

if accel.shape != (3,):
    raise AssertionError(
        "Fresh-process accelerometer shape failure: "
        f"{accel.shape}"
    )

if dvl.shape != (3,):
    raise AssertionError(
        f"Fresh-process DVL shape failure: {dvl.shape}"
    )


# -------------------------------------------------------------
# Finiteness checks
# -------------------------------------------------------------

for name, value in {
    "gyro": gyro,
    "accelerometer": accel,
    "DVL": dvl,
}.items():

    if not np.all(
        np.isfinite(value)
    ):
        raise AssertionError(
            f"Fresh-process {name} contains non-finite values"
        )


for name, value in {
    "depth": depth,
    "heading": heading,
}.items():

    if not np.isfinite(value):
        raise AssertionError(
            f"Fresh-process {name} is non-finite"
        )


# -------------------------------------------------------------
# Known ideal relationships
# -------------------------------------------------------------

np.testing.assert_allclose(
    gyro,
    nu[3:6],
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    depth,
    10.0,
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    heading,
    0.45,
    rtol=0.0,
    atol=1e-12,
)


# -------------------------------------------------------------
# Seeded stochastic reproducibility
# -------------------------------------------------------------

rng_a = np.random.default_rng(
    31001
)

rng_b = np.random.default_rng(
    31001
)

noisy_a = sensors.gyroscope_measurement(
    nu,
    bias=np.array([
        0.002,
        -0.001,
        0.001,
    ]),
    noise_std=np.array([
        0.001,
        0.001,
        0.001,
    ]),
    rng=rng_a,
)

noisy_b = sensors.gyroscope_measurement(
    nu,
    bias=np.array([
        0.002,
        -0.001,
        0.001,
    ]),
    noise_std=np.array([
        0.001,
        0.001,
        0.001,
    ]),
    rng=rng_b,
)

np.testing.assert_array_equal(
    noisy_a,
    noisy_b,
)


# -------------------------------------------------------------
# Heading wrapping
# -------------------------------------------------------------

if not (
    -np.pi
    < heading
    <= np.pi
):
    raise AssertionError(
        "Fresh-process heading left (-pi, pi]"
    )


wrapped_boundary = sensors.wrap_angle_pi(
    -np.pi
)

np.testing.assert_allclose(
    wrapped_boundary,
    np.pi,
    rtol=0.0,
    atol=1e-15,
)


# -------------------------------------------------------------
# Sonar intentionally remains outside public API
# -------------------------------------------------------------

if any(
    "sonar" in name.lower()
    for name in sensors.__all__
):
    raise AssertionError(
        "Sonar unexpectedly exported through __all__"
    )

if hasattr(
    sensors,
    "sonar_measurement",
):
    raise AssertionError(
        "sonar_measurement unexpectedly exposed"
    )


# -------------------------------------------------------------
# Report from genuinely fresh interpreter
# -------------------------------------------------------------

print("FRESH PROCESS SENSOR VALIDATION PASSED")

print(
    "Package:",
    Path(sensors.__file__).resolve(),
)

print(
    "Exports:",
    len(sensors.__all__),
)

print(
    "Gyro:",
    gyro,
)

print(
    "Accelerometer:",
    accel,
)

print(
    "Depth:",
    depth,
)

print(
    "DVL:",
    dvl,
)

print(
    "Heading:",
    heading,
)

print(
    "Seeded gyro:",
    noisy_a,
)

print(
    "Seeded gyro reproducibility: True"
)

print(
    "Heading boundary convention: True"
)

print(
    "Sonar publicly exported: False"
)
'''


# =====================================================================
# Build clean subprocess environment
#
# Prepend repo_root/src while preserving any existing PYTHONPATH.
# =====================================================================

fresh_env = os.environ.copy()

existing_pythonpath = fresh_env.get(
    "PYTHONPATH",
    "",
)

if existing_pythonpath:
    fresh_env["PYTHONPATH"] = (
        str(src_root)
        + os.pathsep
        + existing_pythonpath
    )
else:
    fresh_env["PYTHONPATH"] = str(
        src_root
    )


# =====================================================================
# Launch genuinely fresh interpreter
#
# -I is deliberately NOT used because isolated mode ignores PYTHONPATH.
# -B prevents creation/use of .pyc files for this validation process.
# =====================================================================

completed = subprocess.run(
    [
        sys.executable,
        "-B",
        "-c",
        textwrap.dedent(
            fresh_process_script
        ),
    ],
    cwd=str(repo_root),
    env=fresh_env,
    capture_output=True,
    text=True,
)


# =====================================================================
# Report subprocess result
# =====================================================================

print("STEP 31 FRESH-PROCESS CHECK")
print()

print("Source root supplied through PYTHONPATH:")
print(
    src_root.relative_to(
        repo_root
    )
)

print()

print("Return code:")
print(
    completed.returncode
)

print()

print("STDOUT:")

if completed.stdout:
    print(
        completed.stdout,
        end=""
        if completed.stdout.endswith("\n")
        else "\n",
    )
else:
    print("<EMPTY>")


print()

print("STDERR:")

if completed.stderr:
    print(
        completed.stderr,
        end=""
        if completed.stderr.endswith("\n")
        else "\n",
    )
else:
    print("<EMPTY>")


# =====================================================================
# Final validation
# =====================================================================

if completed.returncode != 0:
    raise AssertionError(
        "Fresh-process production sensor validation failed"
    )


required_success_markers = [
    "FRESH PROCESS SENSOR VALIDATION PASSED",
    "Seeded gyro reproducibility: True",
    "Heading boundary convention: True",
    "Sonar publicly exported: False",
]


for marker in required_success_markers:

    if marker not in completed.stdout:
        raise AssertionError(
            "Fresh process did not report expected marker: "
            f"{marker}"
        )


print()

print("STEP 31 VALIDATION PASSED")

STEP 31 FRESH-PROCESS CHECK

Source root supplied through PYTHONPATH:
src

Return code:
0

STDOUT:
FRESH PROCESS SENSOR VALIDATION PASSED
Package: /home/agniacolyte/HydroLab-3D/src/hydrolab3d/sensors/__init__.py
Exports: 12
Gyro: [ 0.04 -0.03  0.07]
Accelerometer: [0.87146582 1.58746425 9.62538546]
Depth: 10.0
DVL: [ 0.65243868 -0.09504999  0.14106222]
Heading: 0.4500000000000002
Seeded gyro: [ 0.04324086 -0.03067535  0.07112005]
Seeded gyro reproducibility: True
Heading boundary convention: True
Sonar publicly exported: False

STDERR:
<EMPTY>

STEP 31 VALIDATION PASSED


## Step 32 - Add Dedicated Sensor Regression Tests

The production sensor package has now passed:

- notebook-local validation;
- production-module parity checks;
- package-level export validation;
- fresh-process import validation.

We now preserve the validated behavior in an automated regression suite:

```text
tests/test_sensors.py
```

The dedicated tests will cover:

- common bias/noise behavior;
- deterministic seeded noise;
- gyroscope truth extraction;
- stationary accelerometer specific force;
- depth sign convention;
- DVL water-relative velocity;
- heading wrapping;
- package-level public API;
- malformed-input rejection.

The goal is not to reproduce the entire notebook.

The notebook remains the exploratory and analytical record.

The test suite captures the production invariants that must not regress.

### Validation Criteria

This step passes only if:

1. `tests/test_sensors.py` is written successfully;
2. no existing test file is modified;
3. no production source file is modified;
4. the new test file contains focused regression tests for all validated sensor families;
5. pytest execution is deferred until the next step.

In [42]:
# ---------------------------------------------------------------------
# Step 32 - Add dedicated production sensor regression tests
# ---------------------------------------------------------------------

sensor_test_path = (
    repo_root
    / "tests"
    / "test_sensors.py"
)


# =====================================================================
# Preserve existing production source
# =====================================================================

production_files_before = {
    path.name: path.read_text(
        encoding="utf-8"
    )
    for path in [
        sensor_module_path,
        imu_module_path,
        depth_module_path,
        dvl_module_path,
        heading_module_path,
        sensor_init_path,
    ]
}


# =====================================================================
# Preserve existing test files
# =====================================================================

existing_test_paths = [
    path
    for path in (
        repo_root
        / "tests"
    ).glob("test_*.py")
    if path.name != "test_sensors.py"
]

existing_tests_before = {
    path.name: path.read_text(
        encoding="utf-8"
    )
    for path in existing_test_paths
}


# =====================================================================
# New sensor regression tests
# =====================================================================

sensor_test_source = '''"""Regression tests for HydroLab-3D sensor models."""

import numpy as np
import pytest

import hydrolab3d.sensors as sensors
from hydrolab3d.world.currents import (
    current_velocity_body,
    relative_velocity_6dof,
)


def test_measure_with_bias_noise_seeded_reproducibility():
    truth = np.array([
        1.0,
        -2.0,
        3.0,
    ])

    bias = np.array([
        0.1,
        -0.2,
        0.3,
    ])

    noise_std = np.array([
        0.01,
        0.02,
        0.03,
    ])

    rng_a = np.random.default_rng(1234)
    rng_b = np.random.default_rng(1234)

    a = sensors.measure_with_bias_noise(
        truth,
        bias=bias,
        noise_std=noise_std,
        rng=rng_a,
    )

    b = sensors.measure_with_bias_noise(
        truth,
        bias=bias,
        noise_std=noise_std,
        rng=rng_b,
    )

    np.testing.assert_array_equal(
        a,
        b,
    )


def test_ideal_gyroscope_returns_body_angular_velocity():
    body_velocity = np.array([
        1.2,
        -0.4,
        0.3,
        0.08,
        -0.05,
        0.11,
    ])

    measured = sensors.ideal_gyroscope_measurement(
        body_velocity
    )

    np.testing.assert_array_equal(
        measured,
        body_velocity[3:6],
    )


def test_stationary_level_accelerometer_reads_positive_g():
    measured = sensors.ideal_accelerometer_measurement(
        np.zeros(6),
        np.zeros(6),
        np.zeros(6),
    )

    np.testing.assert_allclose(
        measured,
        np.array([
            0.0,
            0.0,
            9.81,
        ]),
        rtol=0.0,
        atol=1e-12,
    )


def test_depth_uses_positive_depth_below_surface():
    pose = np.array([
        4.0,
        -2.0,
        -12.5,
        0.2,
        -0.1,
        0.7,
    ])

    assert sensors.ideal_depth_measurement(
        pose
    ) == pytest.approx(
        12.5
    )


def test_depth_rejects_above_surface_pose():
    pose = np.array([
        0.0,
        0.0,
        0.25,
        0.0,
        0.0,
        0.0,
    ])

    with pytest.raises(ValueError):
        sensors.ideal_depth_measurement(
            pose
        )


def test_dvl_matches_water_relative_body_velocity():
    pose = np.array([
        2.0,
        -3.0,
        -10.0,
        0.22,
        -0.17,
        0.61,
    ])

    body_velocity = np.array([
        1.3,
        -0.4,
        0.25,
        0.08,
        -0.06,
        0.10,
    ])

    current_world = np.array([
        0.25,
        -0.12,
        0.05,
    ])

    expected = relative_velocity_6dof(
        pose,
        body_velocity,
        current_world,
    )[:3]

    measured = sensors.ideal_dvl_measurement(
        pose,
        body_velocity,
        current_world,
    )

    np.testing.assert_allclose(
        measured,
        expected,
        rtol=0.0,
        atol=1e-12,
    )


def test_dvl_zero_when_vehicle_moves_with_water():
    pose = np.array([
        1.0,
        2.0,
        -8.0,
        0.15,
        -0.12,
        0.45,
    ])

    current_world = np.array([
        0.3,
        -0.1,
        0.06,
    ])

    current_body = current_velocity_body(
        pose,
        current_world,
    )

    body_velocity = np.array([
        current_body[0],
        current_body[1],
        current_body[2],
        0.2,
        -0.1,
        0.05,
    ])

    measured = sensors.ideal_dvl_measurement(
        pose,
        body_velocity,
        current_world,
    )

    np.testing.assert_allclose(
        measured,
        np.zeros(3),
        rtol=0.0,
        atol=1e-12,
    )


@pytest.mark.parametrize(
    "angle",
    [
        np.pi,
        -np.pi,
        3.0 * np.pi,
        -3.0 * np.pi,
    ],
)
def test_heading_boundary_maps_to_positive_pi(angle):
    assert sensors.wrap_angle_pi(
        angle
    ) == pytest.approx(
        np.pi,
        abs=1e-15,
    )


def test_heading_bias_rewraps_across_pi():
    pose = np.array([
        0.0,
        0.0,
        -2.0,
        0.0,
        0.0,
        np.pi - 0.05,
    ])

    measured = sensors.heading_measurement(
        pose,
        bias=0.20,
        noise_std=0.0,
    )

    expected = sensors.wrap_angle_pi(
        np.pi - 0.05 + 0.20
    )

    assert measured == pytest.approx(
        expected,
        abs=1e-15,
    )

    assert -np.pi < measured <= np.pi


def test_sensor_noise_requires_explicit_rng():
    with pytest.raises(ValueError):
        sensors.gyroscope_measurement(
            np.zeros(6),
            noise_std=0.01,
        )


def test_sensor_public_api_excludes_sonar():
    expected = [
        "accelerometer_measurement",
        "depth_measurement",
        "dvl_measurement",
        "gyroscope_measurement",
        "heading_measurement",
        "ideal_accelerometer_measurement",
        "ideal_depth_measurement",
        "ideal_dvl_measurement",
        "ideal_gyroscope_measurement",
        "ideal_heading_measurement",
        "measure_with_bias_noise",
        "wrap_angle_pi",
    ]

    assert sensors.__all__ == expected

    assert not any(
        "sonar" in name.lower()
        for name in sensors.__all__
    )
'''


# =====================================================================
# Write new test file
# =====================================================================

sensor_test_path.write_text(
    sensor_test_source,
    encoding="utf-8",
)


# =====================================================================
# Verify exact write
# =====================================================================

if not sensor_test_path.exists():
    raise AssertionError(
        "tests/test_sensors.py was not created"
    )

written_test_source = (
    sensor_test_path.read_text(
        encoding="utf-8"
    )
)

if written_test_source != sensor_test_source:
    raise AssertionError(
        "Written test_sensors.py does not match intended source"
    )


# =====================================================================
# Verify existing test files remain unchanged
# =====================================================================

for path in existing_test_paths:

    current = path.read_text(
        encoding="utf-8"
    )

    if current != existing_tests_before[path.name]:
        raise AssertionError(
            f"Existing test file changed unexpectedly: "
            f"{path.name}"
        )


# =====================================================================
# Verify production source remains unchanged
# =====================================================================

production_paths = [
    sensor_module_path,
    imu_module_path,
    depth_module_path,
    dvl_module_path,
    heading_module_path,
    sensor_init_path,
]

for path in production_paths:

    current = path.read_text(
        encoding="utf-8"
    )

    if current != production_files_before[path.name]:
        raise AssertionError(
            f"Production source changed unexpectedly: "
            f"{path.name}"
        )


# =====================================================================
# Report
# =====================================================================

print("STEP 32 TEST WRITE PASSED")
print()

print("Created:")
print(
    sensor_test_path.relative_to(
        repo_root
    )
)

print()

print("Test source:")
print("-" * 80)
print(written_test_source)
print("-" * 80)

print()

print(
    "Existing tests unchanged: True"
)

print(
    "Production sensor source unchanged: True"
)

STEP 32 TEST WRITE PASSED

Created:
tests/test_sensors.py

Test source:
--------------------------------------------------------------------------------
"""Regression tests for HydroLab-3D sensor models."""

import numpy as np
import pytest

import hydrolab3d.sensors as sensors
from hydrolab3d.world.currents import (
    current_velocity_body,
    relative_velocity_6dof,
)


def test_measure_with_bias_noise_seeded_reproducibility():
    truth = np.array([
        1.0,
        -2.0,
        3.0,
    ])

    bias = np.array([
        0.1,
        -0.2,
        0.3,
    ])

    noise_std = np.array([
        0.01,
        0.02,
        0.03,
    ])

    rng_a = np.random.default_rng(1234)
    rng_b = np.random.default_rng(1234)

    a = sensors.measure_with_bias_noise(
        truth,
        bias=bias,
        noise_std=noise_std,
        rng=rng_a,
    )

    b = sensors.measure_with_bias_noise(
        truth,
        bias=bias,
        noise_std=noise_std,
        rng=rng_b,
    )

    

## Step 33 - Run the Dedicated Sensor Test Suite

The production sensor invariants are now encoded in:

```text
tests/test_sensors.py
```

We next execute only this dedicated test file before touching the full HydroLab-3D regression suite.

This isolates failures to Notebook 08 sensor functionality.

The test run must use the repository `src/` layout explicitly so the subprocess imports the production package from:

```text
src/hydrolab3d/
```

### Validation Criteria

This step passes only if:

1. `tests/test_sensors.py` is collected successfully;
2. every dedicated sensor test passes;
3. pytest exits with return code `0`;
4. no production or test source file is modified during execution.

In [43]:
# ---------------------------------------------------------------------
# Step 33 - Run dedicated sensor regression tests
# ---------------------------------------------------------------------

import os
import subprocess
import sys


# =====================================================================
# Preserve relevant source files before pytest
# =====================================================================

files_to_protect = [
    sensor_module_path,
    imu_module_path,
    depth_module_path,
    dvl_module_path,
    heading_module_path,
    sensor_init_path,
    sensor_test_path,
]

protected_sources_before = {
    path.resolve():
        path.read_text(
            encoding="utf-8"
        )
    for path in files_to_protect
}


# =====================================================================
# Build subprocess environment for src/ layout
# =====================================================================

test_env = os.environ.copy()

src_root = (
    repo_root
    / "src"
).resolve()

existing_pythonpath = test_env.get(
    "PYTHONPATH",
    "",
)

if existing_pythonpath:
    test_env["PYTHONPATH"] = (
        str(src_root)
        + os.pathsep
        + existing_pythonpath
    )
else:
    test_env["PYTHONPATH"] = str(
        src_root
    )


# =====================================================================
# Execute dedicated sensor test file
# =====================================================================

completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_sensors.py",
        "-q",
    ],
    cwd=str(repo_root),
    env=test_env,
    capture_output=True,
    text=True,
)


# =====================================================================
# Report raw pytest result
# =====================================================================

print("STEP 33 - DEDICATED SENSOR TESTS")
print()

print("Return code:")
print(
    completed.returncode
)

print()

print("STDOUT:")
if completed.stdout:
    print(
        completed.stdout,
        end=""
        if completed.stdout.endswith("\n")
        else "\n",
    )
else:
    print("<EMPTY>")

print()

print("STDERR:")
if completed.stderr:
    print(
        completed.stderr,
        end=""
        if completed.stderr.endswith("\n")
        else "\n",
    )
else:
    print("<EMPTY>")


# =====================================================================
# Validate pytest success
# =====================================================================

if completed.returncode != 0:
    raise AssertionError(
        "Dedicated sensor regression tests failed"
    )

if "passed" not in completed.stdout:
    raise AssertionError(
        "pytest did not report passing sensor tests"
    )


# =====================================================================
# Verify source files were not modified by test execution
# =====================================================================

for path in files_to_protect:

    resolved = path.resolve()

    current_source = path.read_text(
        encoding="utf-8"
    )

    if (
        current_source
        != protected_sources_before[resolved]
    ):
        raise AssertionError(
            "Source file changed during pytest execution: "
            f"{path.relative_to(repo_root)}"
        )


# =====================================================================
# Final report
# =====================================================================

print()

print("STEP 33 VALIDATION PASSED")

print(
    "Dedicated sensor tests passed: True"
)

print(
    "Production and test source unchanged: True"
)

STEP 33 - DEDICATED SENSOR TESTS

Return code:
0

STDOUT:
..............                                                           [100%]
14 passed in 0.06s

STDERR:
<EMPTY>

STEP 33 VALIDATION PASSED
Dedicated sensor tests passed: True
Production and test source unchanged: True


## Step 34 - Run the Full HydroLab-3D Regression Suite

The dedicated Notebook 08 sensor tests have passed.

We now run the complete HydroLab-3D test suite to verify that the promoted sensor layer does not regress any previously validated subsystem.

This includes the existing tests for:

- kinematics;
- rigid-body dynamics;
- hydrodynamics;
- buoyancy and gravity;
- thrusters;
- ocean currents;
- six-degree-of-freedom UUV dynamics;
- the new sensor models.

The pre-Notebook-08 regression baseline was:

```text
113 passed
```

The final count is intentionally not predicted because Notebook 08 adds new tests.

### Validation Criteria

This step passes only if:

1. the entire `tests/` suite is collected successfully;
2. every test passes;
3. pytest exits with return code `0`;
4. no source file is modified during test execution.

In [44]:
# ---------------------------------------------------------------------
# Step 34 - Run the full HydroLab-3D regression suite
# ---------------------------------------------------------------------

import os
import subprocess
import sys


# =====================================================================
# Protect production and test source during execution
# =====================================================================

protected_paths = list(
    (repo_root / "src").rglob("*.py")
) + list(
    (repo_root / "tests").rglob("*.py")
)

protected_sources_before = {
    path.resolve(): path.read_text(
        encoding="utf-8"
    )
    for path in protected_paths
}


# =====================================================================
# Build environment for src/ package layout
# =====================================================================

full_test_env = os.environ.copy()

src_root = (
    repo_root
    / "src"
).resolve()

existing_pythonpath = full_test_env.get(
    "PYTHONPATH",
    "",
)

if existing_pythonpath:
    full_test_env["PYTHONPATH"] = (
        str(src_root)
        + os.pathsep
        + existing_pythonpath
    )
else:
    full_test_env["PYTHONPATH"] = str(
        src_root
    )


# =====================================================================
# Run complete test suite
# =====================================================================

completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=str(repo_root),
    env=full_test_env,
    capture_output=True,
    text=True,
)


# =====================================================================
# Report
# =====================================================================

print("STEP 34 - FULL HYDROLAB-3D TEST SUITE")
print()

print("Return code:")
print(
    completed.returncode
)

print()

print("STDOUT:")
if completed.stdout:
    print(
        completed.stdout,
        end=""
        if completed.stdout.endswith("\n")
        else "\n",
    )
else:
    print("<EMPTY>")

print()

print("STDERR:")
if completed.stderr:
    print(
        completed.stderr,
        end=""
        if completed.stderr.endswith("\n")
        else "\n",
    )
else:
    print("<EMPTY>")


# =====================================================================
# Validate pytest success
# =====================================================================

if completed.returncode != 0:
    raise AssertionError(
        "Full HydroLab-3D regression suite failed"
    )

if "passed" not in completed.stdout:
    raise AssertionError(
        "pytest did not report a successful full-suite run"
    )


# =====================================================================
# Verify no source file was modified
# =====================================================================

for path in protected_paths:

    resolved = path.resolve()

    current_source = path.read_text(
        encoding="utf-8"
    )

    if (
        current_source
        != protected_sources_before[resolved]
    ):
        raise AssertionError(
            "Source file changed during full pytest run: "
            f"{path.relative_to(repo_root)}"
        )


# =====================================================================
# Final report
# =====================================================================

print()

print("STEP 34 VALIDATION PASSED")

print(
    "Full HydroLab-3D regression suite passed: True"
)

print(
    "Source tree unchanged during pytest: True"
)

STEP 34 - FULL HYDROLAB-3D TEST SUITE

Return code:
0

STDOUT:
........................................................................ [ 56%]
.......................................................                  [100%]
127 passed in 0.66s

STDERR:
<EMPTY>

STEP 34 VALIDATION PASSED
Full HydroLab-3D regression suite passed: True
Source tree unchanged during pytest: True


## Step 35 - Generate Reproducible Sensor Validation Artifacts

The production sensor layer has passed the complete HydroLab-3D regression suite.

We now generate compact Notebook 08 artifacts using only the promoted public production API:

```text
hydrolab3d.sensors
```

The artifacts are written to:

```text
results/notebooks/08_sensor_models/
```

The outputs are:

```text
sensor_snapshot.csv
sensor_summary.json
sensor_validation.png
```

### Artifact Purpose

`**sensor_snapshot.csv**`

Stores one deterministic ground-truth sensor snapshot together with one seeded imperfect measurement snapshot.

`**sensor_summary.json**`

Records the sensor configuration, random seeds, units, frame conventions, regression result, and key architectural decisions.

`**sensor_validation.png**`

Shows the measurement error:

$$
e
=
\tilde y-y
$$

for each scalar sensor channel.

For vector sensors, each axis is shown independently.

To make different sensor channels visually comparable, the plotted error is normalized by the configured measurement-noise standard deviation where that standard deviation is nonzero:

$$
e_N
=
\frac{\tilde y-y}{\sigma}.
$$

This visualization is diagnostic only.

It does not alter or feed back into the simulated physical state.

All artifacts are generated from the promoted production implementation rather than notebook-local prototypes.

In [45]:
# ---------------------------------------------------------------------
# Step 35 - Generate compact Notebook 08 validation artifacts
# ---------------------------------------------------------------------

import csv
import json

import matplotlib.pyplot as plt
import numpy as np

import hydrolab3d.sensors as sensors


# =====================================================================
# Artifact directory
# =====================================================================

artifact_dir = (
    repo_root
    / "results"
    / "notebooks"
    / "08_sensor_models"
)

artifact_dir.mkdir(
    parents=True,
    exist_ok=True,
)


snapshot_path = (
    artifact_dir
    / "sensor_snapshot.csv"
)

summary_path = (
    artifact_dir
    / "sensor_summary.json"
)

plot_path = (
    artifact_dir
    / "sensor_validation.png"
)


# =====================================================================
# Use the already validated production-dynamics truth state from
# Step 16.
# =====================================================================

eta_artifact = eta_truth.copy()
nu_artifact = nu_truth.copy()
nu_dot_artifact = nu_dot_truth.copy()
current_artifact = current_world.copy()


# =====================================================================
# Ideal measurements from PUBLIC production API
# =====================================================================

ideal = {
    "gyro":
        sensors.ideal_gyroscope_measurement(
            nu_artifact
        ),

    "accelerometer":
        sensors.ideal_accelerometer_measurement(
            eta_artifact,
            nu_artifact,
            nu_dot_artifact,
        ),

    "depth":
        sensors.ideal_depth_measurement(
            eta_artifact
        ),

    "dvl":
        sensors.ideal_dvl_measurement(
            eta_artifact,
            nu_artifact,
            current_artifact,
        ),

    "heading":
        sensors.ideal_heading_measurement(
            eta_artifact
        ),
}


# =====================================================================
# Explicit sensor configuration
# =====================================================================

sensor_config = {
    "gyro": {
        "bias": np.array([
            0.002,
            -0.001,
            0.0015,
        ]),
        "noise_std": np.array([
            0.001,
            0.001,
            0.001,
        ]),
        "seed": 35001,
    },

    "accelerometer": {
        "bias": np.array([
            0.03,
            -0.02,
            0.04,
        ]),
        "noise_std": np.array([
            0.015,
            0.015,
            0.020,
        ]),
        "seed": 35002,
    },

    "depth": {
        "bias": 0.08,
        "noise_std": 0.03,
        "seed": 35003,
    },

    "dvl": {
        "bias": np.array([
            0.01,
            -0.005,
            0.008,
        ]),
        "noise_std": np.array([
            0.008,
            0.008,
            0.010,
        ]),
        "seed": 35004,
    },

    "heading": {
        "bias": 0.015,
        "noise_std": 0.008,
        "seed": 35005,
    },
}


# =====================================================================
# Seeded imperfect measurements from PUBLIC production API
# =====================================================================

measured = {
    "gyro":
        sensors.gyroscope_measurement(
            nu_artifact,
            bias=sensor_config["gyro"]["bias"],
            noise_std=sensor_config["gyro"]["noise_std"],
            rng=np.random.default_rng(
                sensor_config["gyro"]["seed"]
            ),
        ),

    "accelerometer":
        sensors.accelerometer_measurement(
            eta_artifact,
            nu_artifact,
            nu_dot_artifact,
            bias=sensor_config[
                "accelerometer"
            ]["bias"],
            noise_std=sensor_config[
                "accelerometer"
            ]["noise_std"],
            rng=np.random.default_rng(
                sensor_config[
                    "accelerometer"
                ]["seed"]
            ),
        ),

    "depth":
        sensors.depth_measurement(
            eta_artifact,
            bias=sensor_config["depth"]["bias"],
            noise_std=sensor_config[
                "depth"
            ]["noise_std"],
            rng=np.random.default_rng(
                sensor_config["depth"]["seed"]
            ),
        ),

    "dvl":
        sensors.dvl_measurement(
            eta_artifact,
            nu_artifact,
            current_artifact,
            bias=sensor_config["dvl"]["bias"],
            noise_std=sensor_config[
                "dvl"
            ]["noise_std"],
            rng=np.random.default_rng(
                sensor_config["dvl"]["seed"]
            ),
        ),

    "heading":
        sensors.heading_measurement(
            eta_artifact,
            bias=sensor_config[
                "heading"
            ]["bias"],
            noise_std=sensor_config[
                "heading"
            ]["noise_std"],
            rng=np.random.default_rng(
                sensor_config["heading"]["seed"]
            ),
        ),
}


# =====================================================================
# Ground truth must remain untouched
# =====================================================================

np.testing.assert_array_equal(
    eta_artifact,
    eta_truth,
)

np.testing.assert_array_equal(
    nu_artifact,
    nu_truth,
)

np.testing.assert_array_equal(
    nu_dot_artifact,
    nu_dot_truth,
)


# =====================================================================
# Flatten measurements into channel-level records
# =====================================================================

records = []


def add_vector_records(
    sensor_name,
    axes,
    units,
    ideal_value,
    measured_value,
    bias,
    noise_std,
):
    for i, axis in enumerate(axes):

        records.append({
            "sensor":
                sensor_name,

            "axis":
                axis,

            "units":
                units,

            "ideal":
                float(ideal_value[i]),

            "measured":
                float(measured_value[i]),

            "error":
                float(
                    measured_value[i]
                    - ideal_value[i]
                ),

            "bias":
                float(bias[i]),

            "noise_std":
                float(noise_std[i]),
        })


def add_scalar_record(
    sensor_name,
    units,
    ideal_value,
    measured_value,
    bias,
    noise_std,
):
    records.append({
        "sensor":
            sensor_name,

        "axis":
            "scalar",

        "units":
            units,

        "ideal":
            float(ideal_value),

        "measured":
            float(measured_value),

        "error":
            float(
                measured_value
                - ideal_value
            ),

        "bias":
            float(bias),

        "noise_std":
            float(noise_std),
    })


add_vector_records(
    "gyroscope",
    ["x", "y", "z"],
    "rad/s",
    ideal["gyro"],
    measured["gyro"],
    sensor_config["gyro"]["bias"],
    sensor_config["gyro"]["noise_std"],
)

add_vector_records(
    "accelerometer",
    ["x", "y", "z"],
    "m/s^2",
    ideal["accelerometer"],
    measured["accelerometer"],
    sensor_config["accelerometer"]["bias"],
    sensor_config["accelerometer"]["noise_std"],
)

add_scalar_record(
    "depth",
    "m",
    ideal["depth"],
    measured["depth"],
    sensor_config["depth"]["bias"],
    sensor_config["depth"]["noise_std"],
)

add_vector_records(
    "dvl",
    ["x", "y", "z"],
    "m/s",
    ideal["dvl"],
    measured["dvl"],
    sensor_config["dvl"]["bias"],
    sensor_config["dvl"]["noise_std"],
)

add_scalar_record(
    "heading",
    "rad",
    ideal["heading"],
    measured["heading"],
    sensor_config["heading"]["bias"],
    sensor_config["heading"]["noise_std"],
)


# =====================================================================
# Write CSV
# =====================================================================

fieldnames = [
    "sensor",
    "axis",
    "units",
    "ideal",
    "measured",
    "error",
    "bias",
    "noise_std",
]

with snapshot_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=fieldnames,
    )

    writer.writeheader()

    writer.writerows(
        records
    )


# =====================================================================
# Write JSON summary
# =====================================================================

summary = {
    "notebook":
        "08_sensor_models",

    "production_api":
        "hydrolab3d.sensors",

    "sensor_scope": [
        "gyroscope",
        "accelerometer",
        "depth",
        "dvl",
        "heading",
    ],

    "deferred_scope": [
        "sonar",
        "absolute_position_fix",
    ],

    "frames": {
        "pose":
            "[x, y, z, roll, pitch, yaw]",

        "body_velocity":
            "[u, v, w, p, q, r]",

        "world_z":
            "positive upward",

        "depth":
            "surface_z - world_z",

        "dvl":
            "body-frame velocity relative to surrounding water",
    },

    "randomness": {
        "generator":
            "numpy.random.default_rng",

        "policy":
            "explicit independent seeded RNG per sensor",
    },

    "seeds": {
        name:
            int(config["seed"])
        for name, config
        in sensor_config.items()
    },

    "regression": {
        "dedicated_sensor_tests":
            "14 passed",

        "full_suite":
            "127 passed",
    },

    "architecture": {
        "ground_truth_mutated_by_sensors":
            False,

        "sensor_base_class_introduced":
            False,

        "sonar_publicly_exported":
            False,
    },
}


with summary_path.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        summary,
        file,
        indent=2,
    )

    file.write("\n")


# =====================================================================
# Validation plot
#
# One independent figure, no subplots.
# Plot normalized measurement error:
#
#     (measured - ideal) / noise_std
# =====================================================================

labels = []

normalized_errors = []

for record in records:

    labels.append(
        (
            f'{record["sensor"]}:'
            f'{record["axis"]}'
        )
    )

    normalized_errors.append(
        (
            record["measured"]
            - record["ideal"]
        )
        / record["noise_std"]
    )


normalized_errors = np.asarray(
    normalized_errors,
    dtype=float,
)


fig, ax = plt.subplots(
    figsize=(11, 5.5)
)

positions = np.arange(
    len(labels)
)

ax.bar(
    positions,
    normalized_errors,
)

ax.axhline(
    0.0,
    linewidth=1.0,
)

ax.set_xticks(
    positions
)

ax.set_xticklabels(
    labels,
    rotation=45,
    ha="right",
)

ax.set_ylabel(
    "Measurement error / configured noise std"
)

ax.set_title(
    "HydroLab-3D Notebook 08 Sensor Snapshot"
)

ax.grid(
    axis="y",
    alpha=0.25,
)

fig.tight_layout()

fig.savefig(
    plot_path,
    dpi=160,
    bbox_inches="tight",
)

plt.close(
    fig
)


# =====================================================================
# Verify artifacts
# =====================================================================

for path in [
    snapshot_path,
    summary_path,
    plot_path,
]:

    if not path.exists():
        raise AssertionError(
            f"Artifact was not created: {path}"
        )

    if path.stat().st_size == 0:
        raise AssertionError(
            f"Artifact is empty: {path}"
        )


# Validate CSV row count.
with snapshot_path.open(
    "r",
    encoding="utf-8",
    newline="",
) as file:

    csv_rows = list(
        csv.DictReader(file)
    )

if len(csv_rows) != 11:
    raise AssertionError(
        "Expected 11 sensor channels in snapshot CSV, "
        f"found {len(csv_rows)}"
    )


# Validate summary contents.
with summary_path.open(
    "r",
    encoding="utf-8",
) as file:

    loaded_summary = json.load(
        file
    )

if (
    loaded_summary["regression"]["full_suite"]
    != "127 passed"
):
    raise AssertionError(
        "Artifact summary contains incorrect regression result"
    )


# =====================================================================
# Report
# =====================================================================

print("STEP 35 ARTIFACT GENERATION PASSED")
print()

print("Artifact directory:")
print(
    artifact_dir.relative_to(
        repo_root
    )
)

print()

print("Generated artifacts:")

for path in [
    snapshot_path,
    summary_path,
    plot_path,
]:
    print(
        " ",
        path.name,
        f"({path.stat().st_size} bytes)",
    )

print()

print(
    "Sensor channels recorded:",
    len(csv_rows),
)

print()

print("Seeded imperfect snapshot:")

for name in [
    "gyro",
    "accelerometer",
    "depth",
    "dvl",
    "heading",
]:
    print(
        f"  {name:<14}",
        measured[name],
    )

print()

print(
    "Artifacts generated from public production API: True"
)

print(
    "Ground-truth state remained unchanged: True"
)

print(
    "Recorded full regression result: 127 passed"
)

STEP 35 ARTIFACT GENERATION PASSED

Artifact directory:
results/notebooks/08_sensor_models

Generated artifacts:
  sensor_snapshot.csv (983 bytes)
  sensor_summary.json (985 bytes)
  sensor_validation.png (69999 bytes)

Sensor channels recorded: 11

Seeded imperfect snapshot:
  gyro           [ 0.05965014 -0.04057997  0.09190372]
  accelerometer  [0.82540754 1.74790331 9.58212153]
  depth          12.109348537090632
  dvl            [ 0.93796059 -0.15444446  0.15352021]
  heading        0.5790080396996693

Artifacts generated from public production API: True
Ground-truth state remained unchanged: True
Recorded full regression result: 127 passed


## Conclusion

Notebook 08 introduced the first explicit **measurement layer** into HydroLab-3D.

The previous notebooks developed the physical side of the simulator:

```text
kinematics
    ↓
orientation
    ↓
rigid-body dynamics
    ↓
hydrodynamic drag
    ↓
gravity + buoyancy
    ↓
thrusters
    ↓
ocean currents
    ↓
complete six-DOF UUV dynamics
```

Notebook 08 extends that chain with the distinction required by an autonomy simulator:

```text
physical ground truth
        ↓
sensor model
        ↓
imperfect observation
        ↓
future estimation / control / autonomy
```

The central architectural result is therefore not simply the addition of several sensor functions.

It is the separation of:

$$
\boxed{
\text{true physical state}
\neq
\text{measured state}
}
$$

Sensor noise and bias affect observations only. They never overwrite or perturb the latent UUV state maintained by the dynamics model.

---

### Implemented Sensor Scope

Notebook 08 implements four lightweight sensor families comprising five measurement channels:

```text
IMU
├── gyroscope
└── accelerometer

Depth sensor

DVL

Heading sensor
```

The production implementation now lives in:

```text
src/hydrolab3d/sensors/
├── __init__.py
├── sensor.py
├── imu.py
├── depth.py
├── dvl.py
├── heading.py
└── sonar.py
```

`sonar.py` remains an unimplemented placeholder and is deliberately excluded from the public sensor API.

This is intentional.

HydroLab-3D currently has mature vehicle dynamics and current modeling, but not yet enough geometric environment infrastructure to justify a useful sonar model without introducing artificial assumptions.

---

### Common Measurement Model

The reusable measurement primitive is:

$$
\tilde{\mathbf y}
=
\mathbf y
+
\mathbf b
+
\mathbf n
$$

where:

- $\mathbf y$ is the ideal physical measurement;
- $\mathbf b$ is deterministic sensor bias;
- $\mathbf n$ is Gaussian measurement noise;
- $\tilde{\mathbf y}$ is the reported sensor value.

The common production implementation supports both scalar and vector measurements and validates:

- dimensionality;
- finite values;
- broadcast compatibility;
- non-negative noise standard deviation;
- explicit NumPy random generators.

Stochastic measurements require explicit:

```python
np.random.default_rng(seed)
```

rather than uncontrolled global randomness.

This allows experiments to remain reproducible.

---

### Gyroscope

The ideal gyroscope measures body angular velocity directly from:

$$
\boldsymbol{\nu}
=
\begin{bmatrix}
u &
v &
w &
p &
q &
r
\end{bmatrix}^{T}.
$$

Therefore:

$$
\boxed{
\boldsymbol{\omega}_{gyro}
=
\begin{bmatrix}
p\\
q\\
r
\end{bmatrix}
}
$$

in rad/s.

The model was validated for:

- exact angular-rate recovery;
- independence from translational velocity;
- deterministic bias;
- seeded Gaussian noise;
- malformed and non-finite inputs.

---

### Accelerometer

The accelerometer does **not** simply return translational components of $\dot{\nu}$.

Because HydroLab-3D stores translational velocity in the rotating body frame, the inertial acceleration expressed in body coordinates is:

$$
\mathbf a_B
=
\dot{\mathbf v}_B
+
\boldsymbol{\omega}_B
\times
\mathbf v_B.
$$

An accelerometer measures specific force, so gravitational acceleration must be removed:

$$
\boxed{
\mathbf f_B
=
\dot{\mathbf v}_B
+
\boldsymbol{\omega}_B
\times
\mathbf v_B
-
R_W^B\mathbf g_W
}
$$

with:

$$
\mathbf g_W
=
\begin{bmatrix}
0\\
0\\
-g
\end{bmatrix}.
$$

The existing HydroLab-3D orientation convention remains:

$$
R_B^W
=
R_z(\psi)
R_y(\theta)
R_x(\phi),
$$

and therefore:

$$
R_W^B
=
\left(R_B^W\right)^T.
$$

Several reduced cases verified the frame and sign conventions.

For a stationary, level vehicle:

$$
\mathbf f_B
=
\begin{bmatrix}
0\\
0\\
g
\end{bmatrix}.
$$

With:

$$
g=9.81\;\text{m/s}^2,
$$

the validated result was:

```text
[0.00, 0.00, 9.81] m/s²
```

A positive $90^\circ$ roll produced approximately:

```text
[0.00, 9.81, 0.00] m/s²
```

while a positive $90^\circ$ pitch produced approximately:

```text
[-9.81, 0.00, 0.00] m/s²
```

These tests were especially important because they verified the accelerometer convention against the already established HydroLab-3D rotation convention rather than introducing a separate sensor-frame convention.

---

### Depth Sensor

HydroLab-3D retains its established world convention:

```text
+z_W = upward
```

with the default water surface located at:

$$
z_{surface}=0.
$$

Depth is therefore defined as:

$$
\boxed{
d
=
z_{surface}
-
z_W
}
$$

and for the default surface:

$$
\boxed{
d=-z_W.
}
$$

Thus a vehicle at:

$$
z_W=-12\;\text{m}
$$

reports:

$$
d=12\;\text{m}.
$$

The model supports a configurable surface reference and rejects states genuinely above the modeled water surface.

---

### Doppler Velocity Log

The DVL is modeled as a water-relative velocity sensor.

HydroLab-3D already computes generalized water-relative body velocity as:

$$
\boldsymbol{\nu}_r
=
\boldsymbol{\nu}
-
\boldsymbol{\nu}_c.
$$

The DVL reports only its translational part:

$$
\boxed{
\mathbf v_{DVL}
=
\boldsymbol{\nu}_{r,1:3}.
}
$$

The production DVL deliberately reuses:

```text
relative_velocity_6dof(...)
```

from the existing ocean-current subsystem instead of implementing its own current transformation.

This keeps one authoritative definition of water-relative velocity throughout the simulator.

Validation confirmed that:

- zero world current returns the vehicle's body translational velocity;
- arbitrary world currents are correctly transformed before subtraction;
- a vehicle moving exactly with the surrounding water reports zero DVL velocity;
- body angular rates do not alter the translational DVL output;
- the sensor agrees exactly with the six-DOF dynamics diagnostic `nu_r`.

That final result directly connects Notebook 08 sensing to Notebook 07 dynamics.

---

### Heading Sensor

The initial heading model measures HydroLab-3D yaw directly.

All reported heading values use:

$$
\boxed{
\psi_h
\in
(-\pi,\pi].
}
$$

The wrap operation maps equivalent angular representations to this interval.

In particular:

$$
-\pi
\rightarrow
+\pi
$$

and the same deterministic convention is maintained for odd multiples of $\pi$.

For imperfect measurements:

$$
\psi'
=
\psi_h+b_h+n_h
$$

is first formed, then the **final measurement is wrapped again**:

$$
\boxed{
\tilde{\psi}_h
=
\operatorname{wrap}_{(-\pi,\pi]}
\left(
\psi_h+b_h+n_h
\right).
}
$$

The second wrapping operation is necessary because bias or noise can push an otherwise valid heading across the angular discontinuity.

A 10,000-sample stochastic stress test confirmed that every generated heading remained inside the selected interval.

This is currently a lightweight yaw/heading model, not a detailed magnetic compass simulation.

It does not yet model:

- Earth's magnetic field;
- declination;
- hard-iron effects;
- soft-iron effects;
- magnetic disturbances;
- detailed compass electronics.

---

### Integrated Six-DOF Validation

The sensor models were not validated only against synthetic isolated inputs.

They were connected to the production six-degree-of-freedom UUV model through:

```text
six_dof_state_derivative(...)
```

which supplies:

```text
eta_dot
nu_dot
diagnostics
```

for an actual HydroLab-3D state.

For the integration case used in this notebook:

```text
eta =
[  4.    -3.   -12.     0.18  -0.12   0.55]

nu =
[ 1.1  -0.35  0.22  0.06 -0.04  0.09]
```

the production model produced:

```text
eta_dot =
[ 1.11590984
  0.23405781
  0.28436276
  0.05018668
 -0.05546641
  0.08197426]

nu_dot =
[-0.39122014
 -0.08725554
 -0.05705432
 -0.71716001
  0.61759007
  0.07945639]
```

with water-relative generalized velocity:

```text
nu_r =
[ 0.93430880
 -0.14300378
  0.15145081
  0.06000000
 -0.04000000
  0.09000000]
```

The ideal sensor layer then reported:

```text
Gyroscope:
[ 0.06 -0.04  0.09] rad/s

Accelerometer:
[0.80585661 1.74219452 9.54804481] m/s²

Depth:
12.0 m

DVL:
[ 0.93430880 -0.14300378  0.15145081] m/s

Heading:
0.55 rad
```

The DVL output matched the dynamics diagnostic `nu_r` exactly in its translational components.

The accelerometer output independently matched the specific-force equation evaluated using production `nu_dot`.

Most importantly, sensor evaluation left the physical ground-truth arrays unchanged.

---

### Independent Random Streams

The integrated imperfect-sensor validation used an independent random generator for each sensor.

Conceptually:

```text
Ground Truth
    │
    ├── Gyroscope      ← RNG_gyro
    ├── Accelerometer  ← RNG_accel
    ├── Depth          ← RNG_depth
    ├── DVL            ← RNG_dvl
    └── Heading        ← RNG_heading
```

This prevents the random sequence of one sensor from becoming coupled to the sampling behavior of another sensor.

For example, increasing DVL update frequency should not silently change future gyroscope noise realizations.

Validation demonstrated:

```text
same seed configuration
→ identical complete measurement snapshot

different seeds
→ different stochastic realization

reconstructed individual sensor seed
→ independent sensor measurement reproduced exactly
```

Thus stochastic sensing remains reproducible without becoming globally coupled.

---

### Production Architecture

Notebook-local models were promoted only after their individual and integrated behavior had been validated.

The resulting public production structure is:

```text
hydrolab3d.sensors
│
├── measure_with_bias_noise
│
├── ideal_gyroscope_measurement
├── gyroscope_measurement
│
├── ideal_accelerometer_measurement
├── accelerometer_measurement
│
├── ideal_depth_measurement
├── depth_measurement
│
├── ideal_dvl_measurement
├── dvl_measurement
│
├── wrap_angle_pi
├── ideal_heading_measurement
└── heading_measurement
```

A generic `Sensor` base class was deliberately **not** introduced.

The current sensor models are lightweight functions without a shared lifecycle or meaningful common mutable state. Introducing inheritance at this stage would add abstraction without solving an actual problem.

The design can be extended later if sensor update scheduling, persistent bias states, dropout, calibration state, or asynchronous measurement timing create a genuine need for stateful sensor objects.

---

### Fresh-Process Validation

The completed production package was also tested from a completely fresh Python interpreter rather than relying only on the notebook's long-lived kernel.

The fresh process successfully imported:

```text
hydrolab3d.sensors
```

directly from:

```text
src/hydrolab3d/sensors/__init__.py
```

and exposed exactly:

```text
12 public functions
```

with:

```text
seeded reproducibility : passed
heading convention     : passed
sonar public export    : absent
stderr                 : empty
process return code    : 0
```

This eliminated notebook import caching as a source of false confidence.

---

### Automated Regression Tests

Notebook 08 added:

```text
tests/test_sensors.py
```

containing focused regression tests for:

- seeded measurement reproducibility;
- gyroscope truth extraction;
- stationary accelerometer specific force;
- depth sign convention;
- above-surface depth rejection;
- DVL water-relative velocity;
- zero DVL velocity when co-moving with water;
- heading boundary wrapping;
- heading re-wrapping after bias;
- explicit RNG requirements;
- public API contents.

The dedicated sensor test suite produced:

```text
14 passed
```

The full HydroLab-3D regression suite then produced:

```text
127 passed
```

with:

```text
0 failures
```

compared with the pre-Notebook-08 baseline of:

```text
113 passed
```

Thus all previously validated simulator behavior remained intact while the new sensor regression coverage was added.

---

### Generated Artifacts

Notebook 08 generated compact validation artifacts under:

```text
results/notebooks/08_sensor_models/
```

including:

```text
sensor_snapshot.csv
sensor_summary.json
sensor_validation.png
```

The snapshot records eleven scalar measurement channels:

```text
gyroscope      x y z
accelerometer  x y z
depth          scalar
DVL            x y z
heading        scalar
```

The final seeded imperfect snapshot was:

```text
Gyroscope:
[ 0.05965014 -0.04057997  0.09190372]

Accelerometer:
[0.82540754 1.74790331 9.58212153]

Depth:
12.109348537090632

DVL:
[ 0.93796059 -0.15444446  0.15352021]

Heading:
0.5790080396996693
```

These artifacts were generated through the promoted public production API rather than notebook-local prototypes.

---

### Deliberately Deferred Features

Notebook 08 does not attempt to simulate every possible underwater sensor effect.

The following remain outside the current scope:

```text
sonar geometry and ray interaction
acoustic multipath
DVL bottom-lock / water-track switching
beam-level DVL modeling
pressure-to-depth thermodynamics
IMU bias random walk
Allan-variance noise models
sensor update rates
sensor latency
packet loss
dropout
quantization
saturation
magnetometer calibration
absolute underwater position fixes
GPS surfacing behavior
acoustic positioning systems
state estimation / sensor fusion
```

These are not accidental omissions.

They are deferred until the simulator contains the environmental geometry, timing infrastructure, localization architecture, or experimental requirements needed to model them meaningfully.

---

## Final Result

Notebook 08 successfully transforms HydroLab-3D from a simulator that knows only the vehicle's physical state into one that can also expose **imperfect observations of that state**.

The resulting architecture is:

```text
World + Vehicle Parameters
            │
            ▼
      Six-DOF Dynamics
            │
            ▼
       Ground Truth
            │
     ┌──────┼────────┬────────┬─────────┐
     ▼      ▼        ▼        ▼         ▼
   Gyro   Accel    Depth     DVL      Heading
     │      │        │        │         │
     └──────┴────────┴────────┴─────────┘
                    │
                    ▼
          Imperfect Measurements
                    │
                    ▼
       Future Estimation / Autonomy
```

The sensor layer is:

- frame-consistent;
- unit-consistent;
- deterministic when noise is disabled;
- reproducibly stochastic when noise is enabled;
- isolated from latent physical truth;
- compatible with the production six-DOF model;
- exposed through a clean public API;
- validated from a fresh Python process;
- protected by automated regression tests;
- lightweight enough to preserve HydroLab-3D's algorithm-first philosophy.

With:

```text
127 passed
```

and all Notebook 08 validation artifacts generated successfully, the initial HydroLab-3D sensor-model layer is complete.

---